### Top30 as Tgt

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Flatten, Concatenate, Reshape, RepeatVector
from tensorflow.keras.models import Model
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
import tensorflow as tf
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,confusion_matrix,roc_auc_score
from tensorflow.keras.metrics import AUC, Precision
import os

In [2]:
train_data_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\Fund1\data\lag\train_data_after_multicollinearity_correlation_3yrs_test_size_time_based_split_2025_07_25.xlsx"
test_data_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\Fund1\data\lag\test_data_after_multicollinearity_correlation_3yrs_test_size_time_based_split_2025_07_25.xlsx"
master_data_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\data\Time_Series_Master_wo_Index_Last20Yrs1.xlsx"

df_train = pd.read_excel(train_data_path)
df_test = pd.read_excel(test_data_path)
df_master = pd.read_excel(master_data_path)
qtrly_returns_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\data\lag\Quarterly_Price_Returns_Labels.xlsx"

In [3]:
df_train.reset_index(drop=True,inplace=True)
df_test.reset_index(drop=True,inplace=True)

In [4]:
df_train.columns

Index(['Ticker', 'Quarter', 'TMR', 'Momentum', 'Beta', 'Alpha', 'EP', 'BP',
       'SP', 'Dividend_Flag', 'DE', 'EPS', 'Zscore', 'Target'],
      dtype='object')

##### Scaling

In [5]:
scaler = StandardScaler()

In [6]:
df_train.columns

Index(['Ticker', 'Quarter', 'TMR', 'Momentum', 'Beta', 'Alpha', 'EP', 'BP',
       'SP', 'Dividend_Flag', 'DE', 'EPS', 'Zscore', 'Target'],
      dtype='object')

In [7]:
vif_cols = list(df_train.columns)
for col in ['Ticker','Quarter','Set','Target']:
    if col in vif_cols:
        vif_cols.remove(col)

In [8]:
vif_cols

['TMR',
 'Momentum',
 'Beta',
 'Alpha',
 'EP',
 'BP',
 'SP',
 'Dividend_Flag',
 'DE',
 'EPS',
 'Zscore']

In [9]:
df_test.replace([np.inf, -np.inf], 0, inplace=True)

In [10]:
# df_test.replace([np.inf, -np.inf], 0, inplace=True)

In [10]:
train_data = scaler.fit_transform(df_train[vif_cols])
test_data = scaler.transform(df_test[vif_cols])

In [11]:
df_train1 = pd.DataFrame(train_data,columns=vif_cols)
df_test1 = pd.DataFrame(test_data,columns=vif_cols)

In [12]:
for col in ['Ticker','Target', 'Quarter']:
    df_train1[col] = df_train[col]
    df_test1[col] = df_test[col]

In [13]:
checkpoints_path = r"D:\Python\Final Quant Fund\Quant Fund\Fund1\src\3yrs_as_test_data_ts"

In [14]:
features = vif_cols.copy()
features

['TMR',
 'Momentum',
 'Beta',
 'Alpha',
 'EP',
 'BP',
 'SP',
 'Dividend_Flag',
 'DE',
 'EPS',
 'Zscore']

#### Model

In [15]:
# features.remove("Quarter")
X_train = df_train1[features].values
y_train = df_train1['Target'].values

X_test = df_test1[features].values
y_test = df_test1['Target'].values

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd
import numpy as np

def calculate_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'roc':roc_auc_score(y_true,y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
        'confusion_matrix': confusion_matrix(y_true, y_pred)
    }

def train_logistic_regression(X_train, y_train, X_test, y_test, k=5):
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    fold_metrics = []

    for train_index, val_index in kf.split(X_train):
        X_train_fold, X_val = X_train[train_index], X_train[val_index]
        y_train_fold, y_val = y_train[train_index], y_train[val_index]

        model = LogisticRegression(max_iter=1000)
        model.fit(X_train_fold, y_train_fold)

        y_val_pred = model.predict(X_val)
        metrics = calculate_metrics(y_val, y_val_pred)
        fold_metrics.append(metrics)

    # Average scalar metrics
    avg_metrics = {
        key: np.mean([m[key] for m in fold_metrics]) for key in ['accuracy','roc', 'precision', 'recall', 'f1']
    }

    # Sum confusion matrices
    conf_matrix_sum = np.sum([m['confusion_matrix'] for m in fold_metrics], axis=0)


    # Final model on full training set
    final_model = LogisticRegression(max_iter=1000)
    final_model.fit(X_train, y_train)
    y_test_pred = final_model.predict(X_test)
    test_metrics = calculate_metrics(y_test, y_test_pred)

    # Combine into DataFrame
    metrics_df = pd.DataFrame({
        'set': ['cross_val_avg', 'test'],
        'accuracy': [avg_metrics['accuracy'], test_metrics['accuracy']],
        'roc':[avg_metrics['roc'],test_metrics['roc']],
        'precision': [avg_metrics['precision'], test_metrics['precision']],
        'recall': [avg_metrics['recall'], test_metrics['recall']],
        'f1': [avg_metrics['f1'], test_metrics['f1']],
        'confusion_matrix': [conf_matrix_sum, test_metrics['confusion_matrix']]
    })

    return metrics_df,model

# Reshape if needed
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)

metrics_df,lr_model = train_logistic_regression(X_train, y_train, X_test, y_test)

metrics_df

,set,accuracy,roc,precision,recall,f1,confusion_matrix
0,cross_val_avg,0.70271,0.538604,0.598712,0.110138,0.185953,"[[28746, 979], [11800, 1460]]"
1,test,0.78921,0.543736,0.543388,0.113558,0.187857,"[[8251, 221], [2053, 263]]"


In [16]:
import pickle as pkl

In [ ]:
with open(r"D:\Python\Final Quant Fund\Quant Fund\Fund1\src\3yrs_as_test_data_ts\lr_top30.pkl","wb") as file:
    pkl.dump(lr_model, file)

In [17]:
metrics_df

,set,accuracy,roc,precision,recall,f1,confusion_matrix
0,cross_val_avg,0.70271,0.538604,0.598712,0.110138,0.185953,"[[28746, 979], [11800, 1460]]"
1,test,0.78921,0.543736,0.543388,0.113558,0.187857,"[[8251, 221], [2053, 263]]"


In [18]:
pred_probs_test = lr_model.predict_proba(X_test)[:,1]
pred_probs_train = lr_model.predict_proba(X_train)[:,1]

In [19]:
np.isnan(pred_probs_test).sum()

np.int64(0)

In [20]:
np.isnan(pred_probs_train).sum()

np.int64(0)

In [21]:

def calculate_metrics1(y_true, y_pred_label,y_pred_prob):
    accuracy = accuracy_score(y_true, y_pred_label)
    precision = precision_score(y_true, y_pred_label)
    recall = recall_score(y_true, y_pred_label)
    f1 = f1_score(y_true, y_pred_label)
    confusion = confusion_matrix(y_true,y_pred_label)
    roc_score = roc_auc_score(y_true,y_pred_prob)
    
    return {
        'accuracy': accuracy,
        'roc':roc_score,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix':confusion
    }



In [22]:
train_metrics1 = calculate_metrics1(y_train,(pred_probs_train>=0.5).astype(int),pred_probs_train)
test_metrics1 = calculate_metrics1(y_test,(pred_probs_test>=0.5).astype(int),pred_probs_test)

In [23]:
train_metrics_df1 = pd.DataFrame([train_metrics1])
train_metrics_df1['Set'] = "Train"
test_metrics_df1 = pd.DataFrame([test_metrics1])
test_metrics_df1['Set'] = "Test"
model_metrics1 = pd.concat([train_metrics_df1,test_metrics_df1],axis=0)
model_metrics1.reset_index(drop=True,inplace=True)
model_metrics1.head()

,accuracy,roc,precision,recall,f1,confusion_matrix,Set
0,0.703222,0.659859,0.601045,0.112821,0.189980,"[[28732, 993], [11764, 1496]]",Train
1,0.789396,0.694064,0.544177,0.117012,0.192608,"[[8245, 227], [2045, 271]]",Test


In [24]:
model_name = 'lr_top30'
top_n = 30

In [25]:

def convert_qtr_to_date(qtr):
    y = int(qtr.split("FY")[1])
    q = int(qtr.split("FY")[0][-1])
    quarter_end_dates = {
        "Q1": "30-06-",
        "Q2": "30-09-",
        "Q3": "31-12-",
        "Q4": "31-03-"
    }
    y = y if q==4 else y-1
    fiscal_year = 2000 + int(y)
    end_date = quarter_end_dates.get(f'Q{q}')
    if end_date:
        return end_date + str(fiscal_year)  # Concatenate the end date with the full year
    else:
        raise ValueError("Invalid quarter format")

In [26]:
df_master = pd.read_excel(master_data_path)

In [27]:
df_master['Quarter'] = pd.to_datetime(df_master['Quarter'].apply(convert_qtr_to_date),dayfirst=True)

In [56]:
# df_pred1.columns

In [28]:
def label_prediction(row, model_name,top_n):
    in_top_50 = row[f'Top_{top_n}_Pred_Prob_{model_name}']
    target = row['Target']
    
    if in_top_50 and target == 1:
        return 'TP'
    elif in_top_50 and target == 0:
        return 'FP'
    elif not in_top_50 and target == 0:
        return 'TN'
    elif not in_top_50 and target == 1:
        return 'FN'

def flag_top50_nlargest(group, pred_prob_col,top_n):
    # If all pred_probs are zero, return all False
    if group[pred_prob_col].sum() == 0:
        return pd.Series([False] * len(group), index=group.index)
    # Use nlargest to get exactly 50
    top50_idx = group[pred_prob_col].nlargest(top_n).index
    flag = pd.Series(False, index=group.index)
    flag.loc[top50_idx] = True
    return flag

def label_prediction1(row, model_name):
    has_more_prob_than_threshold = row[f'Pred_Prob_Threshold_{model_name}']
    target = row['Target']
    
    if has_more_prob_than_threshold and target == 1:
        return 'TP'
    elif has_more_prob_than_threshold and target == 0:
        return 'FP'
    elif not has_more_prob_than_threshold and target == 0:
        return 'TN'
    elif not has_more_prob_than_threshold and target == 1:
        return 'FN'
    
def add_suffix_to_cols(df, suffix, exclude_cols):
    return df.rename(
        columns={col: f"{col}{suffix}" for col in df.columns if col not in exclude_cols + ['Quarter', 'Set']}
    )


In [29]:

if len(set(['Rank','Returns'])-set(df_train1.columns))!=0:
    print("There are no rank and returns columns, merging now..!")
    df_train1 = df_train1.merge(df_master[['Ticker','Quarter','Returns','Rank']],on=['Ticker','Quarter'],how='left')
    df_test1 = df_test1.merge(df_master[['Ticker','Quarter','Returns','Rank']],on=['Ticker','Quarter'],how='left')

if np.isnan(pred_probs_train).all():
    print("All predictionsa re NaNs")

df_train_pred = df_train1[['Ticker','Quarter','Target','Returns','Rank']]
df_test_pred = df_test1[['Ticker','Quarter','Target','Returns','Rank']]

df_test_pred[f'Pred_Prob_{model_name}'] = pred_probs_test.reshape(-1)
df_train_pred[f'Pred_Prob_{model_name}'] = pred_probs_train.reshape(-1)

#### Top30 as fund - irrespective of the prob values
df_train_pred.loc[:,'Set'] = 'Train'
df_test_pred.loc[:,'Set'] = 'Test'
df_pred = pd.concat([df_train_pred,df_test_pred],axis=0)



df_pred.reset_index(drop=True,inplace=True)
df_pred.sort_values(by=['Quarter',f'Pred_Prob_{model_name}'],ascending=False,inplace=True)
df_pred[f'Top_{top_n}_Pred_Prob_{model_name}'] = df_pred.groupby('Quarter', group_keys=False).apply(flag_top50_nlargest, f'Pred_Prob_{model_name}',top_n)

df_pred[f'Top_50_Pred_Prob_{model_name}'] = df_pred.groupby('Quarter', group_keys=False).apply(flag_top50_nlargest, f'Pred_Prob_{model_name}',50)
# df_pred[(df_pred['Top_{top_n}_Pred_Prob_lstm_top50']==True)&(df_pred['Quarter']=='2005-12-31')]

# test_qtr_preds = df_pred[(df_pred['Quarter']=='2005-12-31')]['Pred_Prob_lstm_top50']
# test_qtr_preds[test_qtr_preds>0.9].shape
df_pred['Pred_Category_Top30'] = df_pred.apply(lambda row: label_prediction(row, model_name=model_name,top_n=30), axis=1)
df_pred['Pred_Category_Top50'] = df_pred.apply(lambda row: label_prediction(row, model_name=model_name,top_n=50), axis=1)
df_pred.reset_index(drop=True,inplace=True)

qtr_summary_df = df_pred.groupby('Quarter').agg(
                    Set = ('Set','first'),
                    Avg_Returns_Predicted_Top30=('Returns',lambda x: x[df_pred.loc[x.index, f'Top_30_Pred_Prob_{model_name}'] == True].mean()),
                    Avg_Returns_Predicted_Top50=('Returns',lambda x: x[df_pred.loc[x.index, f'Top_50_Pred_Prob_{model_name}'] == True].mean()),
                    Count_TP_Top30=('Pred_Category_Top30', lambda x: (x == 'TP').sum()),
                    Count_TN_Top30=('Pred_Category_Top30', lambda x: (x == 'TN').sum()),
                    Count_FP_Top30=('Pred_Category_Top30', lambda x: (x == 'FP').sum()),
                    Count_FN_Top30=('Pred_Category_Top30', lambda x: (x == 'FN').sum()),
                    Count_TP_Top50=('Pred_Category_Top50', lambda x: (x == 'TP').sum()),
                    Count_TN_Top50=('Pred_Category_Top50', lambda x: (x == 'TN').sum()),
                    Count_FP_Top50=('Pred_Category_Top50', lambda x: (x == 'FP').sum()),
                    Count_FN_Top50=('Pred_Category_Top50', lambda x: (x == 'FN').sum())
                    
                    ).reset_index()

# set_summary_df = qtr_summary_df.groupby('Set').agg(
#     Count_TP = ('Count_TP','sum'),
#     Count_TN = ('Count_TN','sum'),
#     Count_FP = ('Count_FP','sum'),
#     Count_FN = ('Count_FN','sum')
# )

df_pred1 = df_pred[['Ticker','Quarter','Target','Returns','Rank',f'Pred_Prob_{model_name}','Set']].copy()
df_pred1[f'Pred_Prob_Threshold_{model_name}'] = df_pred1[f'Pred_Prob_{model_name}'].between(0.5,1)



df_pred1['Pred_Category'] = df_pred1.apply(lambda row: label_prediction1(row, model_name=model_name), axis=1)
quarterly_returns_f2 = df_pred1.groupby('Quarter').agg(
                    Set = ('Set','first'),
                    Avg_Returns_Predicted_Thresh=('Returns', lambda x: x[df_pred1.loc[x.index, f'Pred_Prob_Threshold_{model_name}'] == True].mean()),
                    Count_TP_Thresh=('Pred_Category', lambda x: (x == 'TP').sum()),
                    Count_TN_Thresh=('Pred_Category', lambda x: (x == 'TN').sum()),
                    Count_FP_Thresh=('Pred_Category', lambda x: (x == 'FP').sum()),
                    Count_FN_Thresh=('Pred_Category', lambda x: (x == 'FN').sum())
                    ).reset_index()
# if quarterly_returns_f2['Avg_Returns_Predicted'].isna().any():
#     print("there are quarters where there were no 1s by the model")
# else:
#     print("there are no quarters where there were no 1s by the model")
qtr_summary_df2 = pd.DataFrame(quarterly_returns_f2)


#### Actual and Index returns mapping
##### Actual returns
act_returns = df_pred[df_pred['Target']==1].groupby(by='Quarter')['Returns'].mean()
df_act_returns = pd.DataFrame(act_returns)
df_act_returns.reset_index(inplace=True)
df_act_returns.rename(columns={'Returns':'Avg_Returns_Actual'},inplace=True)
qtr_summary_df = qtr_summary_df.merge(df_act_returns,on='Quarter',how='left')

##### Index returns - Qtrly
df_index = pd.read_excel(qtrly_returns_path,sheet_name='Qtrly_Stock_Returns')
df_index1 = df_index[df_index['Ticker'] == 'Nsemcap'].drop(columns=['Industry']).set_index(keys=['Ticker'])
df_index1 = df_index1.T
df_index1.reset_index(inplace=True)
df_index1.rename(columns={'index':'Quarter','Nsemcap':'Index_Returns'},inplace=True)

df_index1['Quarter'] = df_index1['Quarter'].apply(convert_qtr_to_date)
df_index1['Quarter'] = pd.to_datetime(df_index1['Quarter'],dayfirst=True)

qtr_summary_df = qtr_summary_df.merge(df_index1,how='left',on='Quarter')
qtr_summary_df['delta_returns_act_vs_pred_Top30'] = qtr_summary_df['Avg_Returns_Actual']-qtr_summary_df['Avg_Returns_Predicted_Top30']
qtr_summary_df['delta_returns_pred_vs_index_Top30'] = qtr_summary_df['Avg_Returns_Predicted_Top30']-qtr_summary_df['Index_Returns']
qtr_summary_df['delta_returns_act_vs_pred_Top50'] = qtr_summary_df['Avg_Returns_Actual']-qtr_summary_df['Avg_Returns_Predicted_Top50']
qtr_summary_df['delta_returns_pred_vs_index_Top50'] = qtr_summary_df['Avg_Returns_Predicted_Top50']-qtr_summary_df['Index_Returns']

qtr_summary_df2 = qtr_summary_df2.merge(df_act_returns,on='Quarter',how='left')
qtr_summary_df2 = qtr_summary_df2.merge(df_index1,on='Quarter',how='left')

qtr_summary_df2['delta_returns_act_vs_pred_Thresh'] = qtr_summary_df2['Avg_Returns_Actual']-qtr_summary_df2['Avg_Returns_Predicted_Thresh']
qtr_summary_df2['delta_returns_pred_vs_index_Thresh'] = qtr_summary_df2['Avg_Returns_Predicted_Thresh']-qtr_summary_df2['Index_Returns']

# set_summary_df2 = qtr_summary_df2.groupby('Set').agg(
#     Count_TP = ('Count_TP','sum'),
#     Count_TN = ('Count_TN','sum'),
#     Count_FP = ('Count_FP','sum'),
#     Count_FN = ('Count_FN','sum')
# )

exclude_cols = ['Avg_Returns_Actual', 'Index_Returns']
# suffix_map = {
#     'qtr_summary_df2': '_thresh',
#     'qtr_summary_df': '_top'
# }


# Reset index for merge
qtr_summary_df2_reset = qtr_summary_df2.reset_index(drop=True)
qtr_summary_df_reset = qtr_summary_df.reset_index(drop=True)

# Add suffixes
# qtr_summary_df2_suff = add_suffix_to_cols(qtr_summary_df2_reset, suffix_map['qtr_summary_df2'], exclude_cols)
# qtr_summary_df_suff = add_suffix_to_cols(qtr_summary_df_reset, suffix_map['qtr_summary_df'], exclude_cols)
# ...existing code...

# Merge as before
merged_qtr_summary = pd.merge(
    qtr_summary_df_reset, qtr_summary_df2_reset,
    on=['Quarter', 'Set', 'Avg_Returns_Actual', 'Index_Returns'],
    how='inner',
    suffixes=('', '')
)

# Define the desired column order
ordered_cols = [
    'Quarter','Set',
    'Avg_Returns_Predicted_Top30','Avg_Returns_Predicted_Top50',
    'Avg_Returns_Predicted_Thresh',
    'Avg_Returns_Actual', 'Index_Returns',
    'delta_returns_act_vs_pred_Top30','delta_returns_pred_vs_index_Top30',
    'delta_returns_act_vs_pred_Top50','delta_returns_pred_vs_index_Top50',
    'delta_returns_act_vs_pred_Thresh','delta_returns_pred_vs_index_Thresh',
    'Count_TP_Top30', 'Count_TP_Top50', 'Count_TP_Thresh',
    'Count_TN_Top30', 'Count_TN_Top50', 'Count_TN_Thresh',
    'Count_FP_Top30', 'Count_FP_Top50', 'Count_FP_Thresh',
    'Count_FN_Top30', 'Count_FN_Top50', 'Count_FN_Thresh'
]

# Add any other columns that might exist but are not in the list
ordered_cols += [col for col in merged_qtr_summary.columns if col not in ordered_cols]

# Reorder columns
merged_qtr_summary = merged_qtr_summary[ordered_cols]
# merged_qtr_summary.reset_index(inplace=True)
merged_qtr_summary.head()

There are no rank and returns columns, merging now..!


C:\Users\43737\AppData\Local\Temp\ipykernel_4976\4113010357.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test_pred[f'Pred_Prob_{model_name}'] = pred_probs_test.reshape(-1)
C:\Users\43737\AppData\Local\Temp\ipykernel_4976\4113010357.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train_pred[f'Pred_Prob_{model_name}'] = pred_probs_train.reshape(-1)
C:\Users\43737\AppData\Local\Temp\ipykernel_4976\4113010357.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice fro

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Avg_Returns_Predicted_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,...,Count_TP_Thresh,Count_TN_Top30,Count_TN_Top50,Count_TN_Thresh,Count_FP_Top30,Count_FP_Top50,Count_FP_Thresh,Count_FN_Top30,Count_FN_Top50,Count_FN_Thresh
0,2005-03-31,Train,0.220976,0.320713,0.297456,0.462237,-0.019376,0.241262,0.240351,0.141524,...,52,285,280,272,6,11,19,170,155,142
1,2005-06-30,Train,0.301431,0.314079,0.307537,0.495418,0.045476,0.193988,0.255955,0.181340,...,58,284,276,264,8,16,28,171,159,135
2,2005-09-30,Train,0.126790,0.107212,0.097784,0.180268,0.244277,0.053477,-0.117486,0.073055,...,30,279,269,268,10,20,21,176,166,166
3,2005-12-31,Train,0.434809,0.380636,0.466451,0.407782,0.058583,-0.027027,0.376226,0.027147,...,16,285,276,289,5,14,1,170,159,179
4,2006-03-31,Train,0.031898,-0.066579,-0.030370,0.018142,0.187746,-0.013756,-0.155848,0.084720,...,32,292,283,287,3,12,8,168,157,163


In [30]:
merged_qtr_summary['Avg_Returns_Predicted_Thresh'].isna().all()

np.False_

In [31]:
merged_qtr_summary.columns

Index(['Quarter', 'Set', 'Avg_Returns_Predicted_Top30',
       'Avg_Returns_Predicted_Top50', 'Avg_Returns_Predicted_Thresh',
       'Avg_Returns_Actual', 'Index_Returns',
       'delta_returns_act_vs_pred_Top30', 'delta_returns_pred_vs_index_Top30',
       'delta_returns_act_vs_pred_Top50', 'delta_returns_pred_vs_index_Top50',
       'delta_returns_act_vs_pred_Thresh',
       'delta_returns_pred_vs_index_Thresh', 'Count_TP_Top30',
       'Count_TP_Top50', 'Count_TP_Thresh', 'Count_TN_Top30', 'Count_TN_Top50',
       'Count_TN_Thresh', 'Count_FP_Top30', 'Count_FP_Top50',
       'Count_FP_Thresh', 'Count_FN_Top30', 'Count_FN_Top50',
       'Count_FN_Thresh'],
      dtype='object')

In [32]:
df_pred[df_pred['Set']=='Test'][f'Pred_Prob_{model_name}'].unique()

array([0.99999999, 0.74631573, 0.70247438, ..., 0.10971012, 0.10843475,
       0.06628969])

In [33]:
set_summary_df3 =  merged_qtr_summary.groupby('Set').agg(
    Count_TP_Thresh = ('Count_TP_Thresh','sum'),
    Count_TN_Thresh = ('Count_TN_Thresh','sum'),
    Count_FP_Thresh = ('Count_FP_Thresh','sum'),
    Count_FN_Thresh = ('Count_FN_Thresh','sum'),
    Count_TP_Top30 = ('Count_TP_Top30','sum'),
    Count_TN_Top30 = ('Count_TN_Top30','sum'),
    Count_FP_Top30 = ('Count_FP_Top30','sum'),
    Count_FN_Top30 = ('Count_FN_Top30','sum'),
    Count_TP_Top50 = ('Count_TP_Top50','sum'),
    Count_TN_Top50 = ('Count_TN_Top50','sum'),
    Count_FP_Top50 = ('Count_FP_Top50','sum'),
    Count_FN_Top50 = ('Count_FN_Top50','sum')
)
set_summary_df3

,Count_TP_Thresh,Count_TN_Thresh,Count_FP_Thresh,Count_FN_Thresh,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50
Set,,,,,,,,,,,,
Test,271,8245,227,2045,200,8312,160,2116,318,8190,282,1998
Train,1496,28732,993,11764,1231,28916,809,12029,1988,28313,1412,11272


In [34]:

def calc_metrics(tp, tn, fp, fn):
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return accuracy, precision, recall, f1


In [35]:
for suffix in ['_Top30','_Top50', '_Thresh']:
    print(f"\nMetrics for {suffix}:")
    for set_name, row in set_summary_df3.iterrows():
        tp = row[f'Count_TP{suffix}']
        tn = row[f'Count_TN{suffix}']
        fp = row[f'Count_FP{suffix}']
        fn = row[f'Count_FN{suffix}']
        accuracy, precision, recall, f1 = calc_metrics(tp, tn, fp, fn)
        print(f"{set_name}: accuracy={accuracy:.4f}, precision={precision:.4f}, recall={recall:.4f}, f1={f1:.4f}")


Metrics for _Top30:
Test: accuracy=0.7890, precision=0.5556, recall=0.0864, f1=0.1495
Train: accuracy=0.7013, precision=0.6034, recall=0.0928, f1=0.1609

Metrics for _Top50:
Test: accuracy=0.7887, precision=0.5300, recall=0.1373, f1=0.2181
Train: accuracy=0.7049, precision=0.5847, recall=0.1499, f1=0.2387

Metrics for _Thresh:
Test: accuracy=0.7894, precision=0.5442, recall=0.1170, f1=0.1926
Train: accuracy=0.7032, precision=0.6010, recall=0.1128, f1=0.1900


In [36]:
if merged_qtr_summary['Avg_Returns_Predicted_Thresh'].mean()==merged_qtr_summary['Avg_Returns_Predicted_Top30'].mean():
    print("Both the stock picking strategies (Top 30 and Pred prob theshold) have same returns")
if merged_qtr_summary['Avg_Returns_Predicted_Thresh'].mean()==merged_qtr_summary['Avg_Returns_Predicted_Top50'].mean():
    print("Both the stock picking strategies (Top 50 and Pred prob theshold) have same returns")
if merged_qtr_summary['Avg_Returns_Predicted_Top30'].mean()==merged_qtr_summary['Avg_Returns_Predicted_Top50'].mean():
    print("Both the stock picking strategies (Top 30 and Top 50) have same returns")
else:
    print("All of the stock picking strategies (Top 30, Top 50 and Pred prob theshold) have different returns")

All of the stock picking strategies (Top 30, Top 50 and Pred prob theshold) have different returns


In [66]:
qtr_summary_df[qtr_summary_df['Quarter']=='2021-09-30']

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,delta_returns_pred_vs_index_Top50
66,2021-09-30,Train,0.530652,0.503063,7,770,23,22,10,753,40,19,1.519144,0.126554,0.988492,0.404098,1.01608,0.37651


In [67]:
qtr_summary_df2[qtr_summary_df2['Quarter']=='2021-09-30']

,Quarter,Set,Avg_Returns_Predicted_Thresh,Count_TP_Thresh,Count_TN_Thresh,Count_FP_Thresh,Count_FN_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Thresh,delta_returns_pred_vs_index_Thresh
66,2021-09-30,Train,-0.340932,0,792,1,29,1.519144,0.126554,1.860076,-0.467486


In [37]:
df_pred.columns

Index(['Ticker', 'Quarter', 'Target', 'Returns', 'Rank', 'Pred_Prob_lr_top30',
       'Set', 'Top_30_Pred_Prob_lr_top30', 'Top_50_Pred_Prob_lr_top30',
       'Pred_Category_Top30', 'Pred_Category_Top50'],
      dtype='object')

In [38]:
df_pred1.columns

Index(['Ticker', 'Quarter', 'Target', 'Returns', 'Rank', 'Pred_Prob_lr_top30',
       'Set', 'Pred_Prob_Threshold_lr_top30', 'Pred_Category'],
      dtype='object')

In [70]:
merged_qtr_summary[merged_qtr_summary['Quarter']=='2021-09-30']

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Avg_Returns_Predicted_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,...,Count_TP_Thresh,Count_TN_Top30,Count_TN_Top50,Count_TN_Thresh,Count_FP_Top30,Count_FP_Top50,Count_FP_Thresh,Count_FN_Top30,Count_FN_Top50,Count_FN_Thresh
66,2021-09-30,Train,0.530652,0.503063,-0.340932,1.519144,0.126554,0.988492,0.404098,1.01608,...,0,770,753,792,23,40,1,22,19,29


In [39]:
merged_qtr_summary[merged_qtr_summary['Avg_Returns_Predicted_Top30'].isna()].shape

(2, 25)

In [40]:
df_pred[df_pred['Set']=='Test'][f'Pred_Prob_{model_name}'].unique()

array([0.99999999, 0.74631573, 0.70247438, ..., 0.10971012, 0.10843475,
       0.06628969])

In [41]:
df_pred[df_pred['Set']=='Test'][f'Pred_Prob_{model_name}'].describe()

count    10788.000000
mean         0.300274
std          0.102323
min          0.031570
25%          0.237537
50%          0.273702
75%          0.331630
max          1.000000
Name: Pred_Prob_lr_top30, dtype: float64

In [42]:
merged_qtr_summary['Quarter'].is_monotonic_increasing

True

In [43]:
set_summary_df3

,Count_TP_Thresh,Count_TN_Thresh,Count_FP_Thresh,Count_FN_Thresh,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50
Set,,,,,,,,,,,,
Test,271,8245,227,2045,200,8312,160,2116,318,8190,282,1998
Train,1496,28732,993,11764,1231,28916,809,12029,1988,28313,1412,11272


In [44]:
merged_qtr_summary['Avg_Returns_Predicted_Top30'] = merged_qtr_summary['Avg_Returns_Predicted_Top30'].ffill()
merged_qtr_summary['Avg_Returns_Predicted_Top50'] = merged_qtr_summary['Avg_Returns_Predicted_Top50'].ffill()
merged_qtr_summary['Avg_Returns_Predicted_Thresh'] = merged_qtr_summary['Avg_Returns_Predicted_Thresh'].ffill()

In [45]:
merged_qtr_summary['Avg_Returns_Predicted_Top30'].fillna(0,inplace=True)
merged_qtr_summary['Avg_Returns_Predicted_Top50'].fillna(0,inplace=True)
merged_qtr_summary['Avg_Returns_Predicted_Thresh'].fillna(0,inplace=True)

C:\Users\43737\AppData\Local\Temp\ipykernel_4976\4263771019.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_qtr_summary['Avg_Returns_Predicted_Top30'].fillna(0,inplace=True)
C:\Users\43737\AppData\Local\Temp\ipykernel_4976\4263771019.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behav

In [46]:
merged_qtr_summary.head()

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Avg_Returns_Predicted_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,...,Count_TP_Thresh,Count_TN_Top30,Count_TN_Top50,Count_TN_Thresh,Count_FP_Top30,Count_FP_Top50,Count_FP_Thresh,Count_FN_Top30,Count_FN_Top50,Count_FN_Thresh
0,2005-03-31,Train,0.220976,0.320713,0.297456,0.462237,-0.019376,0.241262,0.240351,0.141524,...,52,285,280,272,6,11,19,170,155,142
1,2005-06-30,Train,0.301431,0.314079,0.307537,0.495418,0.045476,0.193988,0.255955,0.181340,...,58,284,276,264,8,16,28,171,159,135
2,2005-09-30,Train,0.126790,0.107212,0.097784,0.180268,0.244277,0.053477,-0.117486,0.073055,...,30,279,269,268,10,20,21,176,166,166
3,2005-12-31,Train,0.434809,0.380636,0.466451,0.407782,0.058583,-0.027027,0.376226,0.027147,...,16,285,276,289,5,14,1,170,159,179
4,2006-03-31,Train,0.031898,-0.066579,-0.030370,0.018142,0.187746,-0.013756,-0.155848,0.084720,...,32,292,283,287,3,12,8,168,157,163


In [47]:
df_pred.head()

,Ticker,Quarter,Target,Returns,Rank,Pred_Prob_lr_top30,Set,Top_30_Pred_Prob_lr_top30,Top_50_Pred_Prob_lr_top30,Pred_Category_Top30,Pred_Category_Top50
0,uel,2024-12-31,0,NaN,NaN,1.000000,Test,True,True,FP,FP
1,dhani,2024-12-31,0,-0.394675,763.0,0.746316,Test,True,True,FP,FP
2,siya,2024-12-31,0,-0.240569,545.0,0.702474,Test,True,True,FP,FP
3,wesl,2024-12-31,0,-0.279325,626.0,0.661647,Test,True,True,FP,FP
4,pgel,2024-12-31,1,-0.065266,208.0,0.659881,Test,True,True,TP,TP


In [48]:
qtr_summary_df.head()

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,delta_returns_pred_vs_index_Top50
0,2005-03-31,Train,0.220976,0.320713,24,285,6,170,39,280,11,155,0.462237,-0.019376,0.241262,0.240351,0.141524,0.340089
1,2005-06-30,Train,0.301431,0.314079,22,284,8,171,34,276,16,159,0.495418,0.045476,0.193988,0.255955,0.181340,0.268603
2,2005-09-30,Train,0.126790,0.107212,20,279,10,176,30,269,20,166,0.180268,0.244277,0.053477,-0.117486,0.073055,-0.137065
3,2005-12-31,Train,0.434809,0.380636,25,285,5,170,36,276,14,159,0.407782,0.058583,-0.027027,0.376226,0.027147,0.322052
4,2006-03-31,Train,0.031898,-0.066579,27,292,3,168,38,283,12,157,0.018142,0.187746,-0.013756,-0.155848,0.084720,-0.254325


In [49]:
qtr_summary_df['Set'].value_counts()

Set
Train    68
Test     12
Name: count, dtype: int64

In [50]:
with pd.ExcelWriter(r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\Fund1\results\lag\top30\3yrs_as_test_data_ts\Pred_Vs_Act_LR_time_split_25_07_2025.xlsx", ) as writer:
    merged_qtr_summary.to_excel(writer, sheet_name='Qtr_Summary', index=False)
    qtr_summary_df.to_excel(writer, sheet_name='Qtr_Summary_Top', index=False)
    qtr_summary_df2.to_excel(writer, sheet_name='Qtr_Summary_Thresh', index=False)
    df_pred.to_excel(writer, sheet_name='Pred_Vs_Act_Top', index=False)
    df_pred1.to_excel(writer, sheet_name='Pred_Vs_Act_Thresh', index=False)
    set_summary_df3.to_excel(writer, sheet_name='Set_Summary')

### Top30 Xform

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Flatten, Concatenate, Reshape, RepeatVector
from tensorflow.keras.models import Model
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
import tensorflow as tf
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,confusion_matrix,roc_auc_score
from tensorflow.keras.metrics import AUC, Precision
import os

In [2]:

def convert_qtr_to_date(qtr):
    y = int(qtr.split("FY")[1])
    q = int(qtr.split("FY")[0][-1])
    quarter_end_dates = {
        "Q1": "30-06-",
        "Q2": "30-09-",
        "Q3": "31-12-",
        "Q4": "31-03-"
    }
    y = y if q==4 else y-1
    fiscal_year = 2000 + int(y)
    end_date = quarter_end_dates.get(f'Q{q}')
    if end_date:
        return end_date + str(fiscal_year)  # Concatenate the end date with the full year
    else:
        raise ValueError("Invalid quarter format")

In [3]:
train_data_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\Fund1\data\lag\train_data_after_multicollinearity_correlation_3yrs_test_size_time_based_split_2025_07_25.xlsx"
test_data_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\Fund1\data\lag\test_data_after_multicollinearity_correlation_3yrs_test_size_time_based_split_2025_07_25.xlsx"
master_data_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\data\Time_Series_Master_wo_Index_Last20Yrs1.xlsx"

df_train = pd.read_excel(train_data_path)
df_test = pd.read_excel(test_data_path)
df_master = pd.read_excel(master_data_path)
qtrly_returns_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\data\lag\Quarterly_Price_Returns_Labels.xlsx"

In [4]:
df_train.reset_index(drop=True,inplace=True)
df_test.reset_index(drop=True,inplace=True)

In [5]:
df_train.columns

Index(['Ticker', 'Quarter', 'TMR', 'Momentum', 'Beta', 'Alpha', 'EP', 'BP',
       'SP', 'Dividend_Flag', 'DE', 'EPS', 'Zscore', 'Target'],
      dtype='object')

##### Scaling

In [6]:
scaler = StandardScaler()

In [7]:
df_train.columns

Index(['Ticker', 'Quarter', 'TMR', 'Momentum', 'Beta', 'Alpha', 'EP', 'BP',
       'SP', 'Dividend_Flag', 'DE', 'EPS', 'Zscore', 'Target'],
      dtype='object')

In [8]:
vif_cols = list(df_train.columns)
for col in ['Ticker','Quarter','Set','Target']:
    if col in vif_cols:
        vif_cols.remove(col)

In [9]:
vif_cols

['TMR',
 'Momentum',
 'Beta',
 'Alpha',
 'EP',
 'BP',
 'SP',
 'Dividend_Flag',
 'DE',
 'EPS',
 'Zscore']

In [10]:
df_test.replace([np.inf, -np.inf], 0, inplace=True)

In [92]:
# df_test.replace([np.inf, -np.inf], 0, inplace=True)

In [11]:
train_data = scaler.fit_transform(df_train[vif_cols])
test_data = scaler.transform(df_test[vif_cols])

In [12]:
df_train1 = pd.DataFrame(train_data,columns=vif_cols)
df_test1 = pd.DataFrame(test_data,columns=vif_cols)

In [13]:
for col in ['Ticker','Target', 'Quarter']:
    df_train1[col] = df_train[col]
    df_test1[col] = df_test[col]

In [16]:
from scipy.stats import skew

In [17]:
# ...existing code...
from sklearn.preprocessing import PowerTransformer
from scipy.stats import boxcox
import numpy as np

best_transforms = {}
for col in vif_cols:
    x = df_train[col].dropna()
    results = {}
    if (x <= 0).any():
        # Only Yeo-Johnson possible
        pt = PowerTransformer(method='yeo-johnson')
        x_yj = pt.fit_transform(x.values.reshape(-1, 1)).flatten()
        results['yeo-johnson'] = abs(skew(x_yj))
    else:
        x_log = np.log1p(x)
        results['log1p'] = abs(skew(x_log))
        try:
            x_bc, _ = boxcox(x)
            results['boxcox'] = abs(skew(x_bc))
        except Exception:
            results['boxcox'] = np.nan
    # Always include original for comparison
    results['original'] = abs(skew(x))
    # Pick the transform with minimum absolute skewness
    best = min(results, key=lambda k: results[k] if not np.isnan(results[k]) else np.inf)
    best_transforms[col] = best
    print(f"{col}: Best transform = {best}, skewness = {results[best]:.3f}")
# ...existing code...
#### Power xforms
# ...existing code...

TMR: Best transform = yeo-johnson, skewness = 1.374
Momentum: Best transform = yeo-johnson, skewness = 0.373
Beta: Best transform = yeo-johnson, skewness = 4.617
Alpha: Best transform = yeo-johnson, skewness = 2.939
EP: Best transform = boxcox, skewness = 0.110
BP: Best transform = yeo-johnson, skewness = 0.206
SP: Best transform = boxcox, skewness = 0.044
Dividend_Flag: Best transform = original, skewness = 1.256
DE: Best transform = yeo-johnson, skewness = 0.049
EPS: Best transform = yeo-johnson, skewness = 1.697
Zscore: Best transform = yeo-johnson, skewness = 0.086


In [20]:
from sklearn.preprocessing import PowerTransformer
from scipy.stats import boxcox
import numpy as np

# 1. Decide best transform for each feature based on your previous results
# best_transform_map = {
#     'TMR': 'yeo-johnson',
#     'Momentum': 'yeo-johnson',
#     'Beta': 'yeo-johnson',
#     'Alpha': 'yeo-johnson',
#     'EP': 'boxcox',
#     'BP': 'yeo-johnson',
#     'SP': 'boxcox',
#     'EPS': 'yeo-johnson',
#     'Zscore': 'yeo-johnson',
#     'FCF': 'yeo-johnson',
#     'ICR': 'yeo-johnson',
#     'Year': 'log1p',
#     'Quarter_number': 'boxcox',  # or 'log1p', both are close
#     # 'Target': 'none',  # Target is not a feature
# }
best_transforms = {
    'TMR': 'yeo-johnson',
    'Momentum': 'yeo-johnson',
    'Beta': 'yeo-johnson',
    'Alpha': 'yeo-johnson',
    'EP': 'boxcox',
    'BP': 'yeo-johnson',
    'SP': 'boxcox',
    'EPS': 'yeo-johnson',
    'Zscore': 'yeo-johnson',
    'DE': 'yeo-johnson', # Added based on your result: Best transform = yeo-johnson
    
    # Keep these commented as they were not in your results:
    # 'FCF': 'yeo-johnson',
    # 'ICR': 'yeo-johnson',
    # 'Year': 'log1p',
    # 'Quarter_number': 'boxcox',
}
def transform_feature(col, x, method):
    if method == 'yeo-johnson':
        pt = PowerTransformer(method='yeo-johnson')
        return pt.fit_transform(x.values.reshape(-1, 1)).flatten()
    elif method == 'boxcox':
        # Box-Cox requires strictly positive values
        # Add a small constant if min(x) == 0
        x_pos = x.copy()
        if (x_pos <= 0).any():
            x_pos = x_pos - x_pos.min() + 1e-6
        return boxcox(x_pos)[0]
    elif method == 'log1p':
        x_pos = x.copy()
        if (x_pos < 0).any():
            x_pos = x_pos - x_pos.min() + 1e-6
        return np.log1p(x_pos)
    else:
        return x.values

# 2. Apply transforms to train and test
df_train_trans = df_train.copy()
df_test_trans = df_test.copy()

for col in vif_cols:
    method = best_transforms.get(col, 'none')
    if method != 'none':
        df_train_trans[col] = transform_feature(col, df_train[col], method)
        df_test_trans[col] = transform_feature(col, df_test[col], method)

# 3. Scaling
scaler = StandardScaler()
train_data = scaler.fit_transform(df_train_trans[vif_cols])
test_data = scaler.transform(df_test_trans[vif_cols])

df_train1 = pd.DataFrame(train_data, columns=vif_cols)
df_test1 = pd.DataFrame(test_data, columns=vif_cols)

for col in ['Ticker', 'Target', 'Quarter']:
    df_train1[col] = df_train[col]
    df_test1[col] = df_test[col]

# 4. Prepare X_train, y_train, X_test, y_test
features = vif_cols.copy()
X_train = df_train1[features].values
y_train = df_train1['Target'].values
X_test = df_test1[features].values
y_test = df_test1['Target'].values

# If your model expects 3D input (samples, timesteps, features):
# X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
# X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
# ...existing code...

X_train shape: (42985, 11)
y_train shape: (42985,)
X_test shape: (10788, 11)
y_test shape: (10788,)


In [21]:
features = vif_cols.copy()
features

['TMR',
 'Momentum',
 'Beta',
 'Alpha',
 'EP',
 'BP',
 'SP',
 'Dividend_Flag',
 'DE',
 'EPS',
 'Zscore']

#### Model

In [26]:
# features.remove("Quarter")
X_train = df_train1[features].values
y_train = df_train1['Target'].values

X_test = df_test1[features].values
y_test = df_test1['Target'].values

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd
import numpy as np

def calculate_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'roc':roc_auc_score(y_true,y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
        'confusion_matrix': confusion_matrix(y_true, y_pred)
    }

def train_logistic_regression_top30_xform(X_train, y_train, X_test, y_test, k=5):
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    fold_metrics = []

    for train_index, val_index in kf.split(X_train):
        X_train_fold, X_val = X_train[train_index], X_train[val_index]
        y_train_fold, y_val = y_train[train_index], y_train[val_index]

        model = LogisticRegression(max_iter=1000)
        model.fit(X_train_fold, y_train_fold)

        y_val_pred = model.predict(X_val)
        metrics = calculate_metrics(y_val, y_val_pred)
        fold_metrics.append(metrics)

    # Average scalar metrics
    avg_metrics = {
        key: np.mean([m[key] for m in fold_metrics]) for key in ['accuracy','roc', 'precision', 'recall', 'f1']
    }

    # Sum confusion matrices
    conf_matrix_sum = np.sum([m['confusion_matrix'] for m in fold_metrics], axis=0)


    # Final model on full training set
    final_model = LogisticRegression(max_iter=1000)
    final_model.fit(X_train, y_train)
    y_test_pred = final_model.predict(X_test)
    test_metrics = calculate_metrics(y_test, y_test_pred)

    # Combine into DataFrame
    metrics_df = pd.DataFrame({
        'set': ['cross_val_avg', 'test'],
        'accuracy': [avg_metrics['accuracy'], test_metrics['accuracy']],
        'roc':[avg_metrics['roc'],test_metrics['roc']],
        'precision': [avg_metrics['precision'], test_metrics['precision']],
        'recall': [avg_metrics['recall'], test_metrics['recall']],
        'f1': [avg_metrics['f1'], test_metrics['f1']],
        'confusion_matrix': [conf_matrix_sum, test_metrics['confusion_matrix']]
    })

    return metrics_df,model

# Reshape if needed
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)

metrics_df,lr_model_top30_xform = train_logistic_regression_top30_xform(X_train, y_train, X_test, y_test)

metrics_df

,set,accuracy,roc,precision,recall,f1,confusion_matrix
0,cross_val_avg,0.704525,0.545714,0.596134,0.131084,0.214829,"[[28546, 1179], [11522, 1738]]"
1,test,0.789303,0.557286,0.532824,0.150691,0.234938,"[[8166, 306], [1967, 349]]"


In [103]:
import pickle as pkl

In [ ]:
with open(r"D:\Python\Final Quant Fund\Quant Fund\Fund1\src\3yrs_as_test_data_ts\lr_top30_xform.pkl","wb") as file:
    pkl.dump(lr_model_top30_xform, file)

In [23]:
metrics_df

,set,accuracy,roc,precision,recall,f1,confusion_matrix
0,cross_val_avg,0.704525,0.545714,0.596134,0.131084,0.214829,"[[28546, 1179], [11522, 1738]]"
1,test,0.789303,0.557286,0.532824,0.150691,0.234938,"[[8166, 306], [1967, 349]]"


In [27]:
pred_probs_test = lr_model_top30_xform.predict_proba(X_test)[:,1]
pred_probs_train = lr_model_top30_xform.predict_proba(X_train)[:,1]

In [28]:
np.isnan(pred_probs_test).sum()

np.int64(0)

In [29]:
np.isnan(pred_probs_train).sum()

np.int64(0)

In [30]:

def calculate_metrics1(y_true, y_pred_label,y_pred_prob):
    accuracy = accuracy_score(y_true, y_pred_label)
    precision = precision_score(y_true, y_pred_label)
    recall = recall_score(y_true, y_pred_label)
    f1 = f1_score(y_true, y_pred_label)
    confusion = confusion_matrix(y_true,y_pred_label)
    roc_score = roc_auc_score(y_true,y_pred_prob)
    
    return {
        'accuracy': accuracy,
        'roc':roc_score,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix':confusion
    }



In [31]:
train_metrics1 = calculate_metrics1(y_train,(pred_probs_train>=0.5).astype(int),pred_probs_train)
test_metrics1 = calculate_metrics1(y_test,(pred_probs_test>=0.5).astype(int),pred_probs_test)

In [32]:
train_metrics_df1 = pd.DataFrame([train_metrics1])
train_metrics_df1['Set'] = "Train"
test_metrics_df1 = pd.DataFrame([test_metrics1])
test_metrics_df1['Set'] = "Test"
model_metrics1 = pd.concat([train_metrics_df1,test_metrics_df1],axis=0)
model_metrics1.reset_index(drop=True,inplace=True)
model_metrics1.head()

,accuracy,roc,precision,recall,f1,confusion_matrix,Set
0,0.704571,0.660936,0.595311,0.132127,0.216256,"[[28534, 1191], [11508, 1752]]",Train
1,0.789396,0.694597,0.533133,0.152850,0.237584,"[[8162, 310], [1962, 354]]",Test


In [33]:
model_name = 'lr_top30_xform'
top_n = 30

In [34]:

def convert_qtr_to_date(qtr):
    y = int(qtr.split("FY")[1])
    q = int(qtr.split("FY")[0][-1])
    quarter_end_dates = {
        "Q1": "30-06-",
        "Q2": "30-09-",
        "Q3": "31-12-",
        "Q4": "31-03-"
    }
    y = y if q==4 else y-1
    fiscal_year = 2000 + int(y)
    end_date = quarter_end_dates.get(f'Q{q}')
    if end_date:
        return end_date + str(fiscal_year)  # Concatenate the end date with the full year
    else:
        raise ValueError("Invalid quarter format")

In [35]:
df_master = pd.read_excel(master_data_path)

In [36]:
df_master['Quarter'] = pd.to_datetime(df_master['Quarter'].apply(convert_qtr_to_date),dayfirst=True)

In [117]:
# df_pred1.columns

In [37]:
def label_prediction(row, model_name,top_n):
    in_top_50 = row[f'Top_{top_n}_Pred_Prob_{model_name}']
    target = row['Target']
    
    if in_top_50 and target == 1:
        return 'TP'
    elif in_top_50 and target == 0:
        return 'FP'
    elif not in_top_50 and target == 0:
        return 'TN'
    elif not in_top_50 and target == 1:
        return 'FN'

def flag_top50_nlargest(group, pred_prob_col,top_n):
    # If all pred_probs are zero, return all False
    if group[pred_prob_col].sum() == 0:
        return pd.Series([False] * len(group), index=group.index)
    # Use nlargest to get exactly 50
    top50_idx = group[pred_prob_col].nlargest(top_n).index
    flag = pd.Series(False, index=group.index)
    flag.loc[top50_idx] = True
    return flag

def label_prediction1(row, model_name):
    has_more_prob_than_threshold = row[f'Pred_Prob_Threshold_{model_name}']
    target = row['Target']
    
    if has_more_prob_than_threshold and target == 1:
        return 'TP'
    elif has_more_prob_than_threshold and target == 0:
        return 'FP'
    elif not has_more_prob_than_threshold and target == 0:
        return 'TN'
    elif not has_more_prob_than_threshold and target == 1:
        return 'FN'
    
def add_suffix_to_cols(df, suffix, exclude_cols):
    return df.rename(
        columns={col: f"{col}{suffix}" for col in df.columns if col not in exclude_cols + ['Quarter', 'Set']}
    )


In [38]:

if len(set(['Rank','Returns'])-set(df_train1.columns))!=0:
    print("There are no rank and returns columns, merging now..!")
    df_train1 = df_train1.merge(df_master[['Ticker','Quarter','Returns','Rank']],on=['Ticker','Quarter'],how='left')
    df_test1 = df_test1.merge(df_master[['Ticker','Quarter','Returns','Rank']],on=['Ticker','Quarter'],how='left')

if np.isnan(pred_probs_train).all():
    print("All predictionsa re NaNs")

df_train_pred = df_train1[['Ticker','Quarter','Target','Returns','Rank']]
df_test_pred = df_test1[['Ticker','Quarter','Target','Returns','Rank']]

df_test_pred[f'Pred_Prob_{model_name}'] = pred_probs_test.reshape(-1)
df_train_pred[f'Pred_Prob_{model_name}'] = pred_probs_train.reshape(-1)

#### Top30 as fund - irrespective of the prob values
df_train_pred.loc[:,'Set'] = 'Train'
df_test_pred.loc[:,'Set'] = 'Test'
df_pred = pd.concat([df_train_pred,df_test_pred],axis=0)



df_pred.reset_index(drop=True,inplace=True)
df_pred.sort_values(by=['Quarter',f'Pred_Prob_{model_name}'],ascending=False,inplace=True)
df_pred[f'Top_{top_n}_Pred_Prob_{model_name}'] = df_pred.groupby('Quarter', group_keys=False).apply(flag_top50_nlargest, f'Pred_Prob_{model_name}',top_n)

df_pred[f'Top_50_Pred_Prob_{model_name}'] = df_pred.groupby('Quarter', group_keys=False).apply(flag_top50_nlargest, f'Pred_Prob_{model_name}',50)
# df_pred[(df_pred['Top_{top_n}_Pred_Prob_lstm_top50']==True)&(df_pred['Quarter']=='2005-12-31')]

# test_qtr_preds = df_pred[(df_pred['Quarter']=='2005-12-31')]['Pred_Prob_lstm_top50']
# test_qtr_preds[test_qtr_preds>0.9].shape
df_pred['Pred_Category_Top30'] = df_pred.apply(lambda row: label_prediction(row, model_name=model_name,top_n=30), axis=1)
df_pred['Pred_Category_Top50'] = df_pred.apply(lambda row: label_prediction(row, model_name=model_name,top_n=50), axis=1)
df_pred.reset_index(drop=True,inplace=True)

qtr_summary_df = df_pred.groupby('Quarter').agg(
                    Set = ('Set','first'),
                    Avg_Returns_Predicted_Top30=('Returns',lambda x: x[df_pred.loc[x.index, f'Top_30_Pred_Prob_{model_name}'] == True].mean()),
                    Avg_Returns_Predicted_Top50=('Returns',lambda x: x[df_pred.loc[x.index, f'Top_50_Pred_Prob_{model_name}'] == True].mean()),
                    Count_TP_Top30=('Pred_Category_Top30', lambda x: (x == 'TP').sum()),
                    Count_TN_Top30=('Pred_Category_Top30', lambda x: (x == 'TN').sum()),
                    Count_FP_Top30=('Pred_Category_Top30', lambda x: (x == 'FP').sum()),
                    Count_FN_Top30=('Pred_Category_Top30', lambda x: (x == 'FN').sum()),
                    Count_TP_Top50=('Pred_Category_Top50', lambda x: (x == 'TP').sum()),
                    Count_TN_Top50=('Pred_Category_Top50', lambda x: (x == 'TN').sum()),
                    Count_FP_Top50=('Pred_Category_Top50', lambda x: (x == 'FP').sum()),
                    Count_FN_Top50=('Pred_Category_Top50', lambda x: (x == 'FN').sum())
                    
                    ).reset_index()

# set_summary_df = qtr_summary_df.groupby('Set').agg(
#     Count_TP = ('Count_TP','sum'),
#     Count_TN = ('Count_TN','sum'),
#     Count_FP = ('Count_FP','sum'),
#     Count_FN = ('Count_FN','sum')
# )

df_pred1 = df_pred[['Ticker','Quarter','Target','Returns','Rank',f'Pred_Prob_{model_name}','Set']].copy()
df_pred1[f'Pred_Prob_Threshold_{model_name}'] = df_pred1[f'Pred_Prob_{model_name}'].between(0.5,1)



df_pred1['Pred_Category'] = df_pred1.apply(lambda row: label_prediction1(row, model_name=model_name), axis=1)
quarterly_returns_f2 = df_pred1.groupby('Quarter').agg(
                    Set = ('Set','first'),
                    Avg_Returns_Predicted_Thresh=('Returns', lambda x: x[df_pred1.loc[x.index, f'Pred_Prob_Threshold_{model_name}'] == True].mean()),
                    Count_TP_Thresh=('Pred_Category', lambda x: (x == 'TP').sum()),
                    Count_TN_Thresh=('Pred_Category', lambda x: (x == 'TN').sum()),
                    Count_FP_Thresh=('Pred_Category', lambda x: (x == 'FP').sum()),
                    Count_FN_Thresh=('Pred_Category', lambda x: (x == 'FN').sum())
                    ).reset_index()
# if quarterly_returns_f2['Avg_Returns_Predicted'].isna().any():
#     print("there are quarters where there were no 1s by the model")
# else:
#     print("there are no quarters where there were no 1s by the model")
qtr_summary_df2 = pd.DataFrame(quarterly_returns_f2)


#### Actual and Index returns mapping
##### Actual returns
act_returns = df_pred[df_pred['Target']==1].groupby(by='Quarter')['Returns'].mean()
df_act_returns = pd.DataFrame(act_returns)
df_act_returns.reset_index(inplace=True)
df_act_returns.rename(columns={'Returns':'Avg_Returns_Actual'},inplace=True)
qtr_summary_df = qtr_summary_df.merge(df_act_returns,on='Quarter',how='left')

##### Index returns - Qtrly
df_index = pd.read_excel(qtrly_returns_path,sheet_name='Qtrly_Stock_Returns')
df_index1 = df_index[df_index['Ticker'] == 'Nsemcap'].drop(columns=['Industry']).set_index(keys=['Ticker'])
df_index1 = df_index1.T
df_index1.reset_index(inplace=True)
df_index1.rename(columns={'index':'Quarter','Nsemcap':'Index_Returns'},inplace=True)

df_index1['Quarter'] = df_index1['Quarter'].apply(convert_qtr_to_date)
df_index1['Quarter'] = pd.to_datetime(df_index1['Quarter'],dayfirst=True)

qtr_summary_df = qtr_summary_df.merge(df_index1,how='left',on='Quarter')
qtr_summary_df['delta_returns_act_vs_pred_Top30'] = qtr_summary_df['Avg_Returns_Actual']-qtr_summary_df['Avg_Returns_Predicted_Top30']
qtr_summary_df['delta_returns_pred_vs_index_Top30'] = qtr_summary_df['Avg_Returns_Predicted_Top30']-qtr_summary_df['Index_Returns']
qtr_summary_df['delta_returns_act_vs_pred_Top50'] = qtr_summary_df['Avg_Returns_Actual']-qtr_summary_df['Avg_Returns_Predicted_Top50']
qtr_summary_df['delta_returns_pred_vs_index_Top50'] = qtr_summary_df['Avg_Returns_Predicted_Top50']-qtr_summary_df['Index_Returns']

qtr_summary_df2 = qtr_summary_df2.merge(df_act_returns,on='Quarter',how='left')
qtr_summary_df2 = qtr_summary_df2.merge(df_index1,on='Quarter',how='left')

qtr_summary_df2['delta_returns_act_vs_pred_Thresh'] = qtr_summary_df2['Avg_Returns_Actual']-qtr_summary_df2['Avg_Returns_Predicted_Thresh']
qtr_summary_df2['delta_returns_pred_vs_index_Thresh'] = qtr_summary_df2['Avg_Returns_Predicted_Thresh']-qtr_summary_df2['Index_Returns']

# set_summary_df2 = qtr_summary_df2.groupby('Set').agg(
#     Count_TP = ('Count_TP','sum'),
#     Count_TN = ('Count_TN','sum'),
#     Count_FP = ('Count_FP','sum'),
#     Count_FN = ('Count_FN','sum')
# )

exclude_cols = ['Avg_Returns_Actual', 'Index_Returns']
# suffix_map = {
#     'qtr_summary_df2': '_thresh',
#     'qtr_summary_df': '_top'
# }


# Reset index for merge
qtr_summary_df2_reset = qtr_summary_df2.reset_index(drop=True)
qtr_summary_df_reset = qtr_summary_df.reset_index(drop=True)

# Add suffixes
# qtr_summary_df2_suff = add_suffix_to_cols(qtr_summary_df2_reset, suffix_map['qtr_summary_df2'], exclude_cols)
# qtr_summary_df_suff = add_suffix_to_cols(qtr_summary_df_reset, suffix_map['qtr_summary_df'], exclude_cols)
# ...existing code...

# Merge as before
merged_qtr_summary = pd.merge(
    qtr_summary_df_reset, qtr_summary_df2_reset,
    on=['Quarter', 'Set', 'Avg_Returns_Actual', 'Index_Returns'],
    how='inner',
    suffixes=('', '')
)

# Define the desired column order
ordered_cols = [
    'Quarter','Set',
    'Avg_Returns_Predicted_Top30','Avg_Returns_Predicted_Top50',
    'Avg_Returns_Predicted_Thresh',
    'Avg_Returns_Actual', 'Index_Returns',
    'delta_returns_act_vs_pred_Top30','delta_returns_pred_vs_index_Top30',
    'delta_returns_act_vs_pred_Top50','delta_returns_pred_vs_index_Top50',
    'delta_returns_act_vs_pred_Thresh','delta_returns_pred_vs_index_Thresh',
    'Count_TP_Top30', 'Count_TP_Top50', 'Count_TP_Thresh',
    'Count_TN_Top30', 'Count_TN_Top50', 'Count_TN_Thresh',
    'Count_FP_Top30', 'Count_FP_Top50', 'Count_FP_Thresh',
    'Count_FN_Top30', 'Count_FN_Top50', 'Count_FN_Thresh'
]

# Add any other columns that might exist but are not in the list
ordered_cols += [col for col in merged_qtr_summary.columns if col not in ordered_cols]

# Reorder columns
merged_qtr_summary = merged_qtr_summary[ordered_cols]
# merged_qtr_summary.reset_index(inplace=True)
merged_qtr_summary.head()

There are no rank and returns columns, merging now..!


C:\Users\43737\AppData\Local\Temp\ipykernel_31640\4113010357.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test_pred[f'Pred_Prob_{model_name}'] = pred_probs_test.reshape(-1)
C:\Users\43737\AppData\Local\Temp\ipykernel_31640\4113010357.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train_pred[f'Pred_Prob_{model_name}'] = pred_probs_train.reshape(-1)
C:\Users\43737\AppData\Local\Temp\ipykernel_31640\4113010357.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice 

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Avg_Returns_Predicted_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,...,Count_TP_Thresh,Count_TN_Top30,Count_TN_Top50,Count_TN_Thresh,Count_FP_Top30,Count_FP_Top50,Count_FP_Thresh,Count_FN_Top30,Count_FN_Top50,Count_FN_Thresh
0,2005-03-31,Train,0.202669,0.324026,0.343297,0.462237,-0.019376,0.259568,0.222045,0.138212,...,65,285,281,267,6,10,24,170,154,129
1,2005-06-30,Train,0.304000,0.330705,0.314607,0.495418,0.045476,0.191418,0.258525,0.164713,...,65,283,277,261,9,15,31,172,158,128
2,2005-09-30,Train,0.129293,0.092910,0.077915,0.180268,0.244277,0.050974,-0.114984,0.087358,...,32,278,269,265,11,20,24,177,166,164
3,2005-12-31,Train,0.424620,0.346235,0.445298,0.407782,0.058583,-0.016837,0.366036,0.061547,...,17,286,276,288,4,14,2,169,159,178
4,2006-03-31,Train,-0.001229,-0.057411,-0.067324,0.018142,0.187746,0.019371,-0.188975,0.075552,...,42,292,286,285,3,9,10,168,154,153


In [39]:
merged_qtr_summary['Avg_Returns_Predicted_Thresh'].isna().all()

np.False_

In [40]:
merged_qtr_summary.columns

Index(['Quarter', 'Set', 'Avg_Returns_Predicted_Top30',
       'Avg_Returns_Predicted_Top50', 'Avg_Returns_Predicted_Thresh',
       'Avg_Returns_Actual', 'Index_Returns',
       'delta_returns_act_vs_pred_Top30', 'delta_returns_pred_vs_index_Top30',
       'delta_returns_act_vs_pred_Top50', 'delta_returns_pred_vs_index_Top50',
       'delta_returns_act_vs_pred_Thresh',
       'delta_returns_pred_vs_index_Thresh', 'Count_TP_Top30',
       'Count_TP_Top50', 'Count_TP_Thresh', 'Count_TN_Top30', 'Count_TN_Top50',
       'Count_TN_Thresh', 'Count_FP_Top30', 'Count_FP_Top50',
       'Count_FP_Thresh', 'Count_FN_Top30', 'Count_FN_Top50',
       'Count_FN_Thresh'],
      dtype='object')

In [41]:
df_pred[df_pred['Set']=='Test'][f'Pred_Prob_{model_name}'].unique()

array([0.7347319 , 0.63331719, 0.62042498, ..., 0.01804842, 0.01567157,
       0.01388437])

In [42]:
set_summary_df3 =  merged_qtr_summary.groupby('Set').agg(
    Count_TP_Thresh = ('Count_TP_Thresh','sum'),
    Count_TN_Thresh = ('Count_TN_Thresh','sum'),
    Count_FP_Thresh = ('Count_FP_Thresh','sum'),
    Count_FN_Thresh = ('Count_FN_Thresh','sum'),
    Count_TP_Top30 = ('Count_TP_Top30','sum'),
    Count_TN_Top30 = ('Count_TN_Top30','sum'),
    Count_FP_Top30 = ('Count_FP_Top30','sum'),
    Count_FN_Top30 = ('Count_FN_Top30','sum'),
    Count_TP_Top50 = ('Count_TP_Top50','sum'),
    Count_TN_Top50 = ('Count_TN_Top50','sum'),
    Count_FP_Top50 = ('Count_FP_Top50','sum'),
    Count_FN_Top50 = ('Count_FN_Top50','sum')
)
set_summary_df3

,Count_TP_Thresh,Count_TN_Thresh,Count_FP_Thresh,Count_FN_Thresh,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50
Set,,,,,,,,,,,,
Test,354,8162,310,1962,197,8309,163,2119,324,8196,276,1992
Train,1752,28534,1191,11508,1225,28910,815,12035,1986,28311,1414,11274


In [43]:

def calc_metrics(tp, tn, fp, fn):
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return accuracy, precision, recall, f1


In [44]:
for suffix in ['_Top30','_Top50', '_Thresh']:
    print(f"\nMetrics for {suffix}:")
    for set_name, row in set_summary_df3.iterrows():
        tp = row[f'Count_TP{suffix}']
        tn = row[f'Count_TN{suffix}']
        fp = row[f'Count_FP{suffix}']
        fn = row[f'Count_FN{suffix}']
        accuracy, precision, recall, f1 = calc_metrics(tp, tn, fp, fn)
        print(f"{set_name}: accuracy={accuracy:.4f}, precision={precision:.4f}, recall={recall:.4f}, f1={f1:.4f}")


Metrics for _Top30:
Test: accuracy=0.7885, precision=0.5472, recall=0.0851, f1=0.1472
Train: accuracy=0.7011, precision=0.6005, recall=0.0924, f1=0.1601

Metrics for _Top50:
Test: accuracy=0.7898, precision=0.5400, recall=0.1399, f1=0.2222
Train: accuracy=0.7048, precision=0.5841, recall=0.1498, f1=0.2384

Metrics for _Thresh:
Test: accuracy=0.7894, precision=0.5331, recall=0.1528, f1=0.2376
Train: accuracy=0.7046, precision=0.5953, recall=0.1321, f1=0.2163


In [45]:
if merged_qtr_summary['Avg_Returns_Predicted_Thresh'].mean()==merged_qtr_summary['Avg_Returns_Predicted_Top30'].mean():
    print("Both the stock picking strategies (Top 30 and Pred prob theshold) have same returns")
if merged_qtr_summary['Avg_Returns_Predicted_Thresh'].mean()==merged_qtr_summary['Avg_Returns_Predicted_Top50'].mean():
    print("Both the stock picking strategies (Top 50 and Pred prob theshold) have same returns")
if merged_qtr_summary['Avg_Returns_Predicted_Top30'].mean()==merged_qtr_summary['Avg_Returns_Predicted_Top50'].mean():
    print("Both the stock picking strategies (Top 30 and Top 50) have same returns")
else:
    print("All of the stock picking strategies (Top 30, Top 50 and Pred prob theshold) have different returns")

All of the stock picking strategies (Top 30, Top 50 and Pred prob theshold) have different returns


In [46]:
qtr_summary_df[qtr_summary_df['Quarter']=='2021-09-30']

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,delta_returns_pred_vs_index_Top50
66,2021-09-30,Train,0.186083,0.146025,16,613,14,179,26,603,24,169,0.362195,0.126554,0.176112,0.059529,0.216171,0.019471


In [128]:
qtr_summary_df2[qtr_summary_df2['Quarter']=='2021-09-30']

,Quarter,Set,Avg_Returns_Predicted_Thresh,Count_TP_Thresh,Count_TN_Thresh,Count_FP_Thresh,Count_FN_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Thresh,delta_returns_pred_vs_index_Thresh
66,2021-09-30,Train,-0.340932,0,792,1,29,1.519144,0.126554,1.860076,-0.467486


In [129]:
df_pred.columns

Index(['Ticker', 'Quarter', 'Target', 'Returns', 'Rank',
       'Pred_Prob_lr_top30_xform', 'Set', 'Top_30_Pred_Prob_lr_top30_xform',
       'Top_50_Pred_Prob_lr_top30_xform', 'Pred_Category_Top30',
       'Pred_Category_Top50'],
      dtype='object')

In [130]:
df_pred1.columns

Index(['Ticker', 'Quarter', 'Target', 'Returns', 'Rank',
       'Pred_Prob_lr_top30_xform', 'Set', 'Pred_Prob_Threshold_lr_top30_xform',
       'Pred_Category'],
      dtype='object')

In [131]:
merged_qtr_summary[merged_qtr_summary['Quarter']=='2021-09-30']

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Avg_Returns_Predicted_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,...,Count_TP_Thresh,Count_TN_Top30,Count_TN_Top50,Count_TN_Thresh,Count_FP_Top30,Count_FP_Top50,Count_FP_Thresh,Count_FN_Top30,Count_FN_Top50,Count_FN_Thresh
66,2021-09-30,Train,0.408642,0.400875,-0.340932,1.519144,0.126554,1.110502,0.282088,1.118268,...,0,769,752,792,24,41,1,23,20,29


In [47]:
merged_qtr_summary[merged_qtr_summary['Avg_Returns_Predicted_Top30'].isna()].shape

(2, 25)

In [48]:
df_pred[df_pred['Set']=='Test'][f'Pred_Prob_{model_name}'].unique()

array([0.7347319 , 0.63331719, 0.62042498, ..., 0.01804842, 0.01567157,
       0.01388437])

In [49]:
df_pred[df_pred['Set']=='Test'][f'Pred_Prob_{model_name}'].describe()

count    10788.000000
mean         0.289112
std          0.118714
min          0.000281
25%          0.204663
50%          0.268304
75%          0.353909
max          0.917692
Name: Pred_Prob_lr_top30_xform, dtype: float64

In [50]:
merged_qtr_summary['Quarter'].is_monotonic_increasing

True

In [136]:
set_summary_df3

,Count_TP_Thresh,Count_TN_Thresh,Count_FP_Thresh,Count_FN_Thresh,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50
Set,,,,,,,,,,,,
Test,0,10438,4,346,42,10124,318,304,62,9904,538,284
Train,0,41050,6,1929,255,39271,1785,1674,373,38029,3027,1556


In [51]:
merged_qtr_summary['Avg_Returns_Predicted_Top30'] = merged_qtr_summary['Avg_Returns_Predicted_Top30'].ffill()
merged_qtr_summary['Avg_Returns_Predicted_Top50'] = merged_qtr_summary['Avg_Returns_Predicted_Top50'].ffill()
merged_qtr_summary['Avg_Returns_Predicted_Thresh'] = merged_qtr_summary['Avg_Returns_Predicted_Thresh'].ffill()

In [52]:
merged_qtr_summary['Avg_Returns_Predicted_Top30'].fillna(0,inplace=True)
merged_qtr_summary['Avg_Returns_Predicted_Top50'].fillna(0,inplace=True)
merged_qtr_summary['Avg_Returns_Predicted_Thresh'].fillna(0,inplace=True)

C:\Users\43737\AppData\Local\Temp\ipykernel_31640\4263771019.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_qtr_summary['Avg_Returns_Predicted_Top30'].fillna(0,inplace=True)
C:\Users\43737\AppData\Local\Temp\ipykernel_31640\4263771019.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always beh

In [139]:
merged_qtr_summary.head()

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Avg_Returns_Predicted_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,...,Count_TP_Thresh,Count_TN_Top30,Count_TN_Top50,Count_TN_Thresh,Count_FP_Top30,Count_FP_Top50,Count_FP_Thresh,Count_FN_Top30,Count_FN_Top50,Count_FN_Thresh
0,2005-03-31,Train,0.487041,0.436550,0.0,1.455923,-0.019376,0.968882,0.506417,1.019373,...,0,431,415,457,26,42,0,24,20,28
1,2005-06-30,Train,0.517589,0.444255,0.0,1.790164,0.045476,1.272575,0.472114,1.345909,...,0,434,416,458,24,42,0,21,19,27
2,2005-09-30,Train,0.088815,0.052912,0.0,0.496120,0.244277,0.407304,-0.155461,0.443207,...,0,431,412,456,25,44,0,24,23,29
3,2005-12-31,Train,0.299423,0.301841,0.0,1.147137,0.058583,0.847714,0.240840,0.845296,...,0,431,414,457,26,43,0,24,21,28
4,2006-03-31,Train,-0.033331,0.193029,0.0,0.772814,0.187746,0.806145,-0.221077,0.579785,...,0,436,422,461,25,39,0,24,18,29


In [140]:
df_pred.head()

,Ticker,Quarter,Target,Returns,Rank,Pred_Prob_lr_top30_xform,Set,Top_30_Pred_Prob_lr_top30_xform,Top_50_Pred_Prob_lr_top30_xform,Pred_Category_Top30,Pred_Category_Top50
0,uel,2024-12-31,0,NaN,NaN,0.727036,Test,True,True,FP,FP
1,ut,2024-12-31,0,-0.358306,745.0,0.133512,Test,True,True,FP,FP
2,rtpow,2024-12-31,0,-0.283309,635.0,0.133457,Test,True,True,FP,FP
3,sdb,2024-12-31,0,0.122062,36.0,0.119482,Test,True,True,FP,FP
4,dcxindia,2024-12-31,0,-0.341313,730.0,0.115898,Test,True,True,FP,FP


In [141]:
qtr_summary_df.head()

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,delta_returns_pred_vs_index_Top50
0,2005-03-31,Train,0.487041,0.436550,4,431,26,24,8,415,42,20,1.455923,-0.019376,0.968882,0.506417,1.019373,0.455926
1,2005-06-30,Train,0.517589,0.444255,6,434,24,21,8,416,42,19,1.790164,0.045476,1.272575,0.472114,1.345909,0.398779
2,2005-09-30,Train,0.088815,0.052912,5,431,25,24,6,412,44,23,0.496120,0.244277,0.407304,-0.155461,0.443207,-0.191364
3,2005-12-31,Train,0.299423,0.301841,4,431,26,24,7,414,43,21,1.147137,0.058583,0.847714,0.240840,0.845296,0.243258
4,2006-03-31,Train,-0.033331,0.193029,5,436,25,24,11,422,39,18,0.772814,0.187746,0.806145,-0.221077,0.579785,0.005283


In [53]:
qtr_summary_df['Set'].value_counts()

Set
Train    68
Test     12
Name: count, dtype: int64

In [54]:
with pd.ExcelWriter(r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\Fund1\results\lag\top30\3yrs_as_test_data_ts\Pred_Vs_Act_LR_xform_time_split_25_07_2025.xlsx", ) as writer:
    merged_qtr_summary.to_excel(writer, sheet_name='Qtr_Summary', index=False)
    qtr_summary_df.to_excel(writer, sheet_name='Qtr_Summary_Top', index=False)
    qtr_summary_df2.to_excel(writer, sheet_name='Qtr_Summary_Thresh', index=False)
    df_pred.to_excel(writer, sheet_name='Pred_Vs_Act_Top', index=False)
    df_pred1.to_excel(writer, sheet_name='Pred_Vs_Act_Thresh', index=False)
    set_summary_df3.to_excel(writer, sheet_name='Set_Summary')

### Top50 as tgt

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Flatten, Concatenate, Reshape, RepeatVector
from tensorflow.keras.models import Model
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
import tensorflow as tf
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,confusion_matrix,roc_auc_score
from tensorflow.keras.metrics import AUC, Precision
import os

In [2]:
train_data_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\Fund1\data\lag\train_data_after_multicollinearity_correlation_3yrs_test_size_time_based_split_2025_07_25.xlsx"
test_data_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\Fund1\data\lag\test_data_after_multicollinearity_correlation_3yrs_test_size_time_based_split_2025_07_25.xlsx"
master_data_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\data\Time_Series_Master_wo_Index_Last20Yrs1.xlsx"

df_train = pd.read_excel(train_data_path)
df_test = pd.read_excel(test_data_path)
df_master = pd.read_excel(master_data_path)
qtrly_returns_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\data\lag\Quarterly_Price_Returns_Labels.xlsx"

In [3]:
df_train.reset_index(drop=True,inplace=True)
df_test.reset_index(drop=True,inplace=True)

In [4]:
df_train.columns

Index(['Ticker', 'Quarter', 'TMR', 'Momentum', 'Beta', 'Alpha', 'EP', 'BP',
       'SP', 'Dividend_Flag', 'DE', 'EPS', 'Zscore', 'Target'],
      dtype='object')

##### Scaling

In [5]:
scaler = StandardScaler()

In [6]:
df_train.columns

Index(['Ticker', 'Quarter', 'TMR', 'Momentum', 'Beta', 'Alpha', 'EP', 'BP',
       'SP', 'Dividend_Flag', 'DE', 'EPS', 'Zscore', 'Target'],
      dtype='object')

In [7]:
vif_cols = list(df_train.columns)
for col in ['Ticker','Quarter','Set','Target']:
    if col in vif_cols:
        vif_cols.remove(col)

In [8]:
vif_cols

['TMR',
 'Momentum',
 'Beta',
 'Alpha',
 'EP',
 'BP',
 'SP',
 'Dividend_Flag',
 'DE',
 'EPS',
 'Zscore']

In [9]:
df_test.replace([np.inf, -np.inf], 0, inplace=True)

In [153]:
# df_test.replace([np.inf, -np.inf], 0, inplace=True)

In [10]:
train_data = scaler.fit_transform(df_train[vif_cols])
test_data = scaler.transform(df_test[vif_cols])

In [11]:
df_train1 = pd.DataFrame(train_data,columns=vif_cols)
df_test1 = pd.DataFrame(test_data,columns=vif_cols)

In [12]:
for col in ['Ticker','Target', 'Quarter']:
    df_train1[col] = df_train[col]
    df_test1[col] = df_test[col]

In [ ]:
df_train1.merge(quarterly_returns_f2)

In [ ]:
df_master['Quarter'] = pd.to_datetime(df_master['Quarter'].apply(convert_qtr_to_date),dayfirst=True)

In [ ]:
df_train1 = df_train1.merge(df_master[['Ticker','Quarter','Returns','Rank']],on=['Ticker','Quarter'])
df_test1 = df_test1.merge(df_master[['Ticker','Quarter','Returns','Rank']],on=['Ticker','Quarter'])

In [ ]:
df_train1['Target'] = np.where(df_train1['Rank']<=50,1,0)
df_test1['Target'] = np.where(df_test1['Rank']<=50,1,0)

In [13]:
features = vif_cols.copy()
features

['TMR',
 'Momentum',
 'Beta',
 'Alpha',
 'EP',
 'BP',
 'SP',
 'Dividend_Flag',
 'DE',
 'EPS',
 'Zscore']

#### Model

In [14]:
# features.remove("Quarter")
X_train = df_train1[features].values
y_train = df_train1['Target'].values

X_test = df_test1[features].values
y_test = df_test1['Target'].values

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd
import numpy as np

def calculate_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'roc':roc_auc_score(y_true,y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
        'confusion_matrix': confusion_matrix(y_true, y_pred)
    }

def train_logistic_regression_top50(X_train, y_train, X_test, y_test, k=5):
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    fold_metrics = []

    for train_index, val_index in kf.split(X_train):
        X_train_fold, X_val = X_train[train_index], X_train[val_index]
        y_train_fold, y_val = y_train[train_index], y_train[val_index]

        model = LogisticRegression(max_iter=1000)
        model.fit(X_train_fold, y_train_fold)

        y_val_pred = model.predict(X_val)
        metrics = calculate_metrics(y_val, y_val_pred)
        fold_metrics.append(metrics)

    # Average scalar metrics
    avg_metrics = {
        key: np.mean([m[key] for m in fold_metrics]) for key in ['accuracy','roc', 'precision', 'recall', 'f1']
    }

    # Sum confusion matrices
    conf_matrix_sum = np.sum([m['confusion_matrix'] for m in fold_metrics], axis=0)


    # Final model on full training set
    final_model = LogisticRegression(max_iter=1000)
    final_model.fit(X_train, y_train)
    y_test_pred = final_model.predict(X_test)
    test_metrics = calculate_metrics(y_test, y_test_pred)

    # Combine into DataFrame
    metrics_df = pd.DataFrame({
        'set': ['cross_val_avg', 'test'],
        'accuracy': [avg_metrics['accuracy'], test_metrics['accuracy']],
        'roc':[avg_metrics['roc'],test_metrics['roc']],
        'precision': [avg_metrics['precision'], test_metrics['precision']],
        'recall': [avg_metrics['recall'], test_metrics['recall']],
        'f1': [avg_metrics['f1'], test_metrics['f1']],
        'confusion_matrix': [conf_matrix_sum, test_metrics['confusion_matrix']]
    })

    return metrics_df,model

# Reshape if needed
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)

In [15]:
metrics_df,lr_model_top50 = train_logistic_regression_top50(X_train, y_train, X_test, y_test)

In [16]:
metrics_df

,set,accuracy,roc,precision,recall,f1,confusion_matrix
0,cross_val_avg,0.70271,0.538604,0.598712,0.110138,0.185953,"[[28746, 979], [11800, 1460]]"
1,test,0.78921,0.543736,0.543388,0.113558,0.187857,"[[8251, 221], [2053, 263]]"


In [ ]:
with open(r"D:\Python\Final Quant Fund\Quant Fund\Fund1\src\3yrs_as_test_data_ts\lr_top50.pkl","wb") as file:
    pkl.dump(lr_model_top50, file)

In [17]:
pred_probs_test = lr_model_top50.predict_proba(X_test)[:,1]
pred_probs_train = lr_model_top50.predict_proba(X_train)[:,1]

In [18]:
np.isnan(pred_probs_test).sum()

np.int64(0)

In [19]:
np.isnan(pred_probs_train).sum()

np.int64(0)

In [20]:

def calculate_metrics1(y_true, y_pred_label,y_pred_prob):
    accuracy = accuracy_score(y_true, y_pred_label)
    precision = precision_score(y_true, y_pred_label)
    recall = recall_score(y_true, y_pred_label)
    f1 = f1_score(y_true, y_pred_label)
    confusion = confusion_matrix(y_true,y_pred_label)
    roc_score = roc_auc_score(y_true,y_pred_prob)
    
    return {
        'accuracy': accuracy,
        'roc':roc_score,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix':confusion
    }



In [21]:
train_metrics1 = calculate_metrics1(y_train,(pred_probs_train>=0.5).astype(int),pred_probs_train)
test_metrics1 = calculate_metrics1(y_test,(pred_probs_test>=0.5).astype(int),pred_probs_test)

In [22]:
train_metrics_df1 = pd.DataFrame([train_metrics1])
train_metrics_df1['Set'] = "Train"
test_metrics_df1 = pd.DataFrame([test_metrics1])
test_metrics_df1['Set'] = "Test"
model_metrics1 = pd.concat([train_metrics_df1,test_metrics_df1],axis=0)
model_metrics1.reset_index(drop=True,inplace=True)
model_metrics1.head()

,accuracy,roc,precision,recall,f1,confusion_matrix,Set
0,0.703222,0.659859,0.601045,0.112821,0.189980,"[[28732, 993], [11764, 1496]]",Train
1,0.789396,0.694064,0.544177,0.117012,0.192608,"[[8245, 227], [2045, 271]]",Test


In [23]:
model_name = 'lr_top50'
top_n = 30

In [24]:

def convert_qtr_to_date(qtr):
    y = int(qtr.split("FY")[1])
    q = int(qtr.split("FY")[0][-1])
    quarter_end_dates = {
        "Q1": "30-06-",
        "Q2": "30-09-",
        "Q3": "31-12-",
        "Q4": "31-03-"
    }
    y = y if q==4 else y-1
    fiscal_year = 2000 + int(y)
    end_date = quarter_end_dates.get(f'Q{q}')
    if end_date:
        return end_date + str(fiscal_year)  # Concatenate the end date with the full year
    else:
        raise ValueError("Invalid quarter format")

In [25]:
df_master = pd.read_excel(master_data_path)

In [26]:
df_master['Quarter'] = pd.to_datetime(df_master['Quarter'].apply(convert_qtr_to_date),dayfirst=True)

In [194]:
# df_pred1.columns

In [27]:
def label_prediction(row, model_name,top_n):
    in_top_50 = row[f'Top_{top_n}_Pred_Prob_{model_name}']
    target = row['Target']
    
    if in_top_50 and target == 1:
        return 'TP'
    elif in_top_50 and target == 0:
        return 'FP'
    elif not in_top_50 and target == 0:
        return 'TN'
    elif not in_top_50 and target == 1:
        return 'FN'

def flag_top50_nlargest(group, pred_prob_col,top_n):
    # If all pred_probs are zero, return all False
    if group[pred_prob_col].sum() == 0:
        return pd.Series([False] * len(group), index=group.index)
    # Use nlargest to get exactly 50
    top50_idx = group[pred_prob_col].nlargest(top_n).index
    flag = pd.Series(False, index=group.index)
    flag.loc[top50_idx] = True
    return flag

def label_prediction1(row, model_name):
    has_more_prob_than_threshold = row[f'Pred_Prob_Threshold_{model_name}']
    target = row['Target']
    
    if has_more_prob_than_threshold and target == 1:
        return 'TP'
    elif has_more_prob_than_threshold and target == 0:
        return 'FP'
    elif not has_more_prob_than_threshold and target == 0:
        return 'TN'
    elif not has_more_prob_than_threshold and target == 1:
        return 'FN'
    
def add_suffix_to_cols(df, suffix, exclude_cols):
    return df.rename(
        columns={col: f"{col}{suffix}" for col in df.columns if col not in exclude_cols + ['Quarter', 'Set']}
    )


In [28]:
def label_prediction(row, model_name,top_n):
    in_top_50 = row[f'Top_{top_n}_Pred_Prob_{model_name}']
    target = row['Target']
    
    if in_top_50 and target == 1:
        return 'TP'
    elif in_top_50 and target == 0:
        return 'FP'
    elif not in_top_50 and target == 0:
        return 'TN'
    elif not in_top_50 and target == 1:
        return 'FN'

def flag_top50_nlargest(group, pred_prob_col,top_n):
    # If all pred_probs are zero, return all False
    if group[pred_prob_col].sum() == 0:
        return pd.Series([False] * len(group), index=group.index)
    # Use nlargest to get exactly 50
    top50_idx = group[pred_prob_col].nlargest(top_n).index
    flag = pd.Series(False, index=group.index)
    flag.loc[top50_idx] = True
    return flag

def label_prediction1(row, model_name):
    has_more_prob_than_threshold = row[f'Pred_Prob_Threshold_{model_name}']
    target = row['Target']
    
    if has_more_prob_than_threshold and target == 1:
        return 'TP'
    elif has_more_prob_than_threshold and target == 0:
        return 'FP'
    elif not has_more_prob_than_threshold and target == 0:
        return 'TN'
    elif not has_more_prob_than_threshold and target == 1:
        return 'FN'
    
def add_suffix_to_cols(df, suffix, exclude_cols):
    return df.rename(
        columns={col: f"{col}{suffix}" for col in df.columns if col not in exclude_cols + ['Quarter', 'Set']}
    )


if len(set(['Rank','Returns'])-set(df_train1.columns))!=0:
    print("There are no rank and returns columns, merging now..!")
    df_train1 = df_train1.merge(df_master[['Ticker','Quarter','Returns','Rank']],on=['Ticker','Quarter'],how='left')
    df_test1 = df_test1.merge(df_master[['Ticker','Quarter','Returns','Rank']],on=['Ticker','Quarter'],how='left')

if np.isnan(pred_probs_train).all():
    print("All predictionsa re NaNs")

df_train_pred = df_train1[['Ticker','Quarter','Target','Returns','Rank']]
df_test_pred = df_test1[['Ticker','Quarter','Target','Returns','Rank']]

df_test_pred[f'Pred_Prob_{model_name}'] = pred_probs_test.reshape(-1)
df_train_pred[f'Pred_Prob_{model_name}'] = pred_probs_train.reshape(-1)

#### Top30 as fund - irrespective of the prob values
df_train_pred.loc[:,'Set'] = 'Train'
df_test_pred.loc[:,'Set'] = 'Test'
df_pred = pd.concat([df_train_pred,df_test_pred],axis=0)



df_pred.reset_index(drop=True,inplace=True)
df_pred.sort_values(by=['Quarter',f'Pred_Prob_{model_name}'],ascending=False,inplace=True)
df_pred[f'Top_{top_n}_Pred_Prob_{model_name}'] = df_pred.groupby('Quarter', group_keys=False).apply(flag_top50_nlargest, f'Pred_Prob_{model_name}',top_n)

df_pred[f'Top_50_Pred_Prob_{model_name}'] = df_pred.groupby('Quarter', group_keys=False).apply(flag_top50_nlargest, f'Pred_Prob_{model_name}',50)
# df_pred[(df_pred['Top_{top_n}_Pred_Prob_lstm_top50']==True)&(df_pred['Quarter']=='2005-12-31')]

# test_qtr_preds = df_pred[(df_pred['Quarter']=='2005-12-31')]['Pred_Prob_lstm_top50']
# test_qtr_preds[test_qtr_preds>0.9].shape
df_pred['Pred_Category_Top30'] = df_pred.apply(lambda row: label_prediction(row, model_name=model_name,top_n=30), axis=1)
df_pred['Pred_Category_Top50'] = df_pred.apply(lambda row: label_prediction(row, model_name=model_name,top_n=50), axis=1)
df_pred.reset_index(drop=True,inplace=True)

qtr_summary_df = df_pred.groupby('Quarter').agg(
                    Set = ('Set','first'),
                    Avg_Returns_Predicted_Top30=('Returns',lambda x: x[df_pred.loc[x.index, f'Top_30_Pred_Prob_{model_name}'] == True].mean()),
                    Avg_Returns_Predicted_Top50=('Returns',lambda x: x[df_pred.loc[x.index, f'Top_50_Pred_Prob_{model_name}'] == True].mean()),
                    Count_TP_Top30=('Pred_Category_Top30', lambda x: (x == 'TP').sum()),
                    Count_TN_Top30=('Pred_Category_Top30', lambda x: (x == 'TN').sum()),
                    Count_FP_Top30=('Pred_Category_Top30', lambda x: (x == 'FP').sum()),
                    Count_FN_Top30=('Pred_Category_Top30', lambda x: (x == 'FN').sum()),
                    Count_TP_Top50=('Pred_Category_Top50', lambda x: (x == 'TP').sum()),
                    Count_TN_Top50=('Pred_Category_Top50', lambda x: (x == 'TN').sum()),
                    Count_FP_Top50=('Pred_Category_Top50', lambda x: (x == 'FP').sum()),
                    Count_FN_Top50=('Pred_Category_Top50', lambda x: (x == 'FN').sum())
                    
                    ).reset_index()

# set_summary_df = qtr_summary_df.groupby('Set').agg(
#     Count_TP = ('Count_TP','sum'),
#     Count_TN = ('Count_TN','sum'),
#     Count_FP = ('Count_FP','sum'),
#     Count_FN = ('Count_FN','sum')
# )

df_pred1 = df_pred[['Ticker','Quarter','Target','Returns','Rank',f'Pred_Prob_{model_name}','Set']].copy()
df_pred1[f'Pred_Prob_Threshold_{model_name}'] = df_pred1[f'Pred_Prob_{model_name}'].between(0.5,1)



df_pred1['Pred_Category'] = df_pred1.apply(lambda row: label_prediction1(row, model_name=model_name), axis=1)
quarterly_returns_f2 = df_pred1.groupby('Quarter').agg(
                    Set = ('Set','first'),
                    Avg_Returns_Predicted_Thresh=('Returns', lambda x: x[df_pred1.loc[x.index, f'Pred_Prob_Threshold_{model_name}'] == True].mean()),
                    Count_TP_Thresh=('Pred_Category', lambda x: (x == 'TP').sum()),
                    Count_TN_Thresh=('Pred_Category', lambda x: (x == 'TN').sum()),
                    Count_FP_Thresh=('Pred_Category', lambda x: (x == 'FP').sum()),
                    Count_FN_Thresh=('Pred_Category', lambda x: (x == 'FN').sum())
                    ).reset_index()

qtr_summary_df2 = pd.DataFrame(quarterly_returns_f2)


#### Actual and Index returns mapping
##### Actual returns
act_returns = df_pred[df_pred['Target']==1].groupby(by='Quarter')['Returns'].mean()
df_act_returns = pd.DataFrame(act_returns)
df_act_returns.reset_index(inplace=True)
df_act_returns.rename(columns={'Returns':'Avg_Returns_Actual'},inplace=True)
qtr_summary_df = qtr_summary_df.merge(df_act_returns,on='Quarter',how='left')

##### Index returns - Qtrly
df_index = pd.read_excel(qtrly_returns_path,sheet_name='Qtrly_Stock_Returns')
df_index1 = df_index[df_index['Ticker'] == 'Nsemcap'].drop(columns=['Industry']).set_index(keys=['Ticker'])
df_index1 = df_index1.T
df_index1.reset_index(inplace=True)
df_index1.rename(columns={'index':'Quarter','Nsemcap':'Index_Returns'},inplace=True)

df_index1['Quarter'] = df_index1['Quarter'].apply(convert_qtr_to_date)
df_index1['Quarter'] = pd.to_datetime(df_index1['Quarter'],dayfirst=True)

qtr_summary_df = qtr_summary_df.merge(df_index1,how='left',on='Quarter')
qtr_summary_df['delta_returns_act_vs_pred_Top30'] = qtr_summary_df['Avg_Returns_Actual']-qtr_summary_df['Avg_Returns_Predicted_Top30']
qtr_summary_df['delta_returns_pred_vs_index_Top30'] = qtr_summary_df['Avg_Returns_Predicted_Top30']-qtr_summary_df['Index_Returns']
qtr_summary_df['delta_returns_act_vs_pred_Top50'] = qtr_summary_df['Avg_Returns_Actual']-qtr_summary_df['Avg_Returns_Predicted_Top50']
qtr_summary_df['delta_returns_pred_vs_index_Top50'] = qtr_summary_df['Avg_Returns_Predicted_Top50']-qtr_summary_df['Index_Returns']

qtr_summary_df2 = qtr_summary_df2.merge(df_act_returns,on='Quarter',how='left')
qtr_summary_df2 = qtr_summary_df2.merge(df_index1,on='Quarter',how='left')

qtr_summary_df2['delta_returns_act_vs_pred_Thresh'] = qtr_summary_df2['Avg_Returns_Actual']-qtr_summary_df2['Avg_Returns_Predicted_Thresh']
qtr_summary_df2['delta_returns_pred_vs_index_Thresh'] = qtr_summary_df2['Avg_Returns_Predicted_Thresh']-qtr_summary_df2['Index_Returns']


exclude_cols = ['Avg_Returns_Actual', 'Index_Returns']

# Reset index for merge
qtr_summary_df2_reset = qtr_summary_df2.reset_index(drop=True)
qtr_summary_df_reset = qtr_summary_df.reset_index(drop=True)

# Merge as before
merged_qtr_summary = pd.merge(
    qtr_summary_df_reset, qtr_summary_df2_reset,
    on=['Quarter', 'Set', 'Avg_Returns_Actual', 'Index_Returns'],
    how='inner',
    suffixes=('', '')
)

# Define the desired column order
ordered_cols = [
    'Quarter','Set',
    'Avg_Returns_Predicted_Top30','Avg_Returns_Predicted_Top50',
    'Avg_Returns_Predicted_Thresh',
    'Avg_Returns_Actual', 'Index_Returns',
    'delta_returns_act_vs_pred_Top30','delta_returns_pred_vs_index_Top30',
    'delta_returns_act_vs_pred_Top50','delta_returns_pred_vs_index_Top50',
    'delta_returns_act_vs_pred_Thresh','delta_returns_pred_vs_index_Thresh',
    'Count_TP_Top30', 'Count_TP_Top50', 'Count_TP_Thresh',
    'Count_TN_Top30', 'Count_TN_Top50', 'Count_TN_Thresh',
    'Count_FP_Top30', 'Count_FP_Top50', 'Count_FP_Thresh',
    'Count_FN_Top30', 'Count_FN_Top50', 'Count_FN_Thresh'
]

# Add any other columns that might exist but are not in the list
ordered_cols += [col for col in merged_qtr_summary.columns if col not in ordered_cols]

# Reorder columns
merged_qtr_summary = merged_qtr_summary[ordered_cols]
# merged_qtr_summary.reset_index(inplace=True)
merged_qtr_summary.head()

There are no rank and returns columns, merging now..!


C:\Users\43737\AppData\Local\Temp\ipykernel_10612\2793545741.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test_pred[f'Pred_Prob_{model_name}'] = pred_probs_test.reshape(-1)
C:\Users\43737\AppData\Local\Temp\ipykernel_10612\2793545741.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train_pred[f'Pred_Prob_{model_name}'] = pred_probs_train.reshape(-1)
C:\Users\43737\AppData\Local\Temp\ipykernel_10612\2793545741.py:58: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice 

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Avg_Returns_Predicted_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,...,Count_TP_Thresh,Count_TN_Top30,Count_TN_Top50,Count_TN_Thresh,Count_FP_Top30,Count_FP_Top50,Count_FP_Thresh,Count_FN_Top30,Count_FN_Top50,Count_FN_Thresh
0,2005-03-31,Train,0.220976,0.320713,0.297456,0.462237,-0.019376,0.241262,0.240351,0.141524,...,52,285,280,272,6,11,19,170,155,142
1,2005-06-30,Train,0.301431,0.314079,0.307537,0.495418,0.045476,0.193988,0.255955,0.181340,...,58,284,276,264,8,16,28,171,159,135
2,2005-09-30,Train,0.126790,0.107212,0.097784,0.180268,0.244277,0.053477,-0.117486,0.073055,...,30,279,269,268,10,20,21,176,166,166
3,2005-12-31,Train,0.434809,0.380636,0.466451,0.407782,0.058583,-0.027027,0.376226,0.027147,...,16,285,276,289,5,14,1,170,159,179
4,2006-03-31,Train,0.031898,-0.066579,-0.030370,0.018142,0.187746,-0.013756,-0.155848,0.084720,...,32,292,283,287,3,12,8,168,157,163


In [29]:
merged_qtr_summary['Avg_Returns_Predicted_Thresh'].isna().all()

np.False_

In [30]:
merged_qtr_summary.columns

Index(['Quarter', 'Set', 'Avg_Returns_Predicted_Top30',
       'Avg_Returns_Predicted_Top50', 'Avg_Returns_Predicted_Thresh',
       'Avg_Returns_Actual', 'Index_Returns',
       'delta_returns_act_vs_pred_Top30', 'delta_returns_pred_vs_index_Top30',
       'delta_returns_act_vs_pred_Top50', 'delta_returns_pred_vs_index_Top50',
       'delta_returns_act_vs_pred_Thresh',
       'delta_returns_pred_vs_index_Thresh', 'Count_TP_Top30',
       'Count_TP_Top50', 'Count_TP_Thresh', 'Count_TN_Top30', 'Count_TN_Top50',
       'Count_TN_Thresh', 'Count_FP_Top30', 'Count_FP_Top50',
       'Count_FP_Thresh', 'Count_FN_Top30', 'Count_FN_Top50',
       'Count_FN_Thresh'],
      dtype='object')

In [31]:
df_pred[df_pred['Set']=='Test'][f'Pred_Prob_{model_name}'].unique()

array([0.99999999, 0.74631573, 0.70247438, ..., 0.10971012, 0.10843475,
       0.06628969])

In [32]:
set_summary_df3 =  merged_qtr_summary.groupby('Set').agg(
    Count_TP_Thresh = ('Count_TP_Thresh','sum'),
    Count_TN_Thresh = ('Count_TN_Thresh','sum'),
    Count_FP_Thresh = ('Count_FP_Thresh','sum'),
    Count_FN_Thresh = ('Count_FN_Thresh','sum'),
    Count_TP_Top30 = ('Count_TP_Top30','sum'),
    Count_TN_Top30 = ('Count_TN_Top30','sum'),
    Count_FP_Top30 = ('Count_FP_Top30','sum'),
    Count_FN_Top30 = ('Count_FN_Top30','sum'),
    Count_TP_Top50 = ('Count_TP_Top50','sum'),
    Count_TN_Top50 = ('Count_TN_Top50','sum'),
    Count_FP_Top50 = ('Count_FP_Top50','sum'),
    Count_FN_Top50 = ('Count_FN_Top50','sum')
)
set_summary_df3

,Count_TP_Thresh,Count_TN_Thresh,Count_FP_Thresh,Count_FN_Thresh,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50
Set,,,,,,,,,,,,
Test,271,8245,227,2045,200,8312,160,2116,318,8190,282,1998
Train,1496,28732,993,11764,1231,28916,809,12029,1988,28313,1412,11272


In [33]:

def calc_metrics(tp, tn, fp, fn):
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return accuracy, precision, recall, f1


In [34]:
for suffix in ['_Top30','_Top50', '_Thresh']:
    print(f"\nMetrics for {suffix}:")
    for set_name, row in set_summary_df3.iterrows():
        tp = row[f'Count_TP{suffix}']
        tn = row[f'Count_TN{suffix}']
        fp = row[f'Count_FP{suffix}']
        fn = row[f'Count_FN{suffix}']
        accuracy, precision, recall, f1 = calc_metrics(tp, tn, fp, fn)
        print(f"{set_name}: accuracy={accuracy:.4f}, precision={precision:.4f}, recall={recall:.4f}, f1={f1:.4f}")


Metrics for _Top30:
Test: accuracy=0.7890, precision=0.5556, recall=0.0864, f1=0.1495
Train: accuracy=0.7013, precision=0.6034, recall=0.0928, f1=0.1609

Metrics for _Top50:
Test: accuracy=0.7887, precision=0.5300, recall=0.1373, f1=0.2181
Train: accuracy=0.7049, precision=0.5847, recall=0.1499, f1=0.2387

Metrics for _Thresh:
Test: accuracy=0.7894, precision=0.5442, recall=0.1170, f1=0.1926
Train: accuracy=0.7032, precision=0.6010, recall=0.1128, f1=0.1900


In [35]:
if merged_qtr_summary['Avg_Returns_Predicted_Thresh'].mean()==merged_qtr_summary['Avg_Returns_Predicted_Top30'].mean():
    print("Both the stock picking strategies (Top 30 and Pred prob theshold) have same returns")
if merged_qtr_summary['Avg_Returns_Predicted_Thresh'].mean()==merged_qtr_summary['Avg_Returns_Predicted_Top50'].mean():
    print("Both the stock picking strategies (Top 50 and Pred prob theshold) have same returns")
if merged_qtr_summary['Avg_Returns_Predicted_Top30'].mean()==merged_qtr_summary['Avg_Returns_Predicted_Top50'].mean():
    print("Both the stock picking strategies (Top 30 and Top 50) have same returns")
else:
    print("All of the stock picking strategies (Top 30, Top 50 and Pred prob theshold) have different returns")

All of the stock picking strategies (Top 30, Top 50 and Pred prob theshold) have different returns


In [204]:
qtr_summary_df[qtr_summary_df['Quarter']=='2021-09-30']

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,delta_returns_pred_vs_index_Top50
66,2021-09-30,Train,0.534084,0.525449,9,752,21,40,13,736,37,36,1.145164,0.126554,0.61108,0.40753,0.619715,0.398895


In [205]:
qtr_summary_df2[qtr_summary_df2['Quarter']=='2021-09-30']

,Quarter,Set,Avg_Returns_Predicted_Thresh,Count_TP_Thresh,Count_TN_Thresh,Count_FP_Thresh,Count_FN_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Thresh,delta_returns_pred_vs_index_Thresh
66,2021-09-30,Train,0.177851,1,772,1,48,1.145164,0.126554,0.967313,0.051297


In [206]:
df_pred.columns

Index(['Ticker', 'Quarter', 'Target', 'Returns', 'Rank', 'Pred_Prob_lr_top50',
       'Set', 'Top_30_Pred_Prob_lr_top50', 'Top_50_Pred_Prob_lr_top50',
       'Pred_Category_Top30', 'Pred_Category_Top50'],
      dtype='object')

In [207]:
df_pred1.columns

Index(['Ticker', 'Quarter', 'Target', 'Returns', 'Rank', 'Pred_Prob_lr_top50',
       'Set', 'Pred_Prob_Threshold_lr_top50', 'Pred_Category'],
      dtype='object')

In [208]:
merged_qtr_summary[merged_qtr_summary['Quarter']=='2021-09-30']

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Avg_Returns_Predicted_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,...,Count_TP_Thresh,Count_TN_Top30,Count_TN_Top50,Count_TN_Thresh,Count_FP_Top30,Count_FP_Top50,Count_FP_Thresh,Count_FN_Top30,Count_FN_Top50,Count_FN_Thresh
66,2021-09-30,Train,0.534084,0.525449,0.177851,1.145164,0.126554,0.61108,0.40753,0.619715,...,1,752,736,772,21,37,1,40,36,48


In [36]:
merged_qtr_summary[merged_qtr_summary['Avg_Returns_Predicted_Top30'].isna()].shape

(2, 25)

In [37]:
df_pred[df_pred['Set']=='Test'][f'Pred_Prob_{model_name}'].unique()

array([0.99999999, 0.74631573, 0.70247438, ..., 0.10971012, 0.10843475,
       0.06628969])

In [38]:
df_pred[df_pred['Set']=='Test'][f'Pred_Prob_{model_name}'].describe()

count    10788.000000
mean         0.300274
std          0.102323
min          0.031570
25%          0.237537
50%          0.273702
75%          0.331630
max          1.000000
Name: Pred_Prob_lr_top50, dtype: float64

In [212]:
merged_qtr_summary['Quarter'].is_monotonic_increasing

True

In [213]:
set_summary_df3

,Count_TP_Thresh,Count_TN_Thresh,Count_FP_Thresh,Count_FN_Thresh,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50
Set,,,,,,,,,,,,
Test,1,10203,12,572,66,9921,294,507,88,9703,512,485
Train,9,39748,23,3205,364,38095,1676,2850,544,36915,2856,2670


In [39]:
merged_qtr_summary['Avg_Returns_Predicted_Top30'] = merged_qtr_summary['Avg_Returns_Predicted_Top30'].ffill()
merged_qtr_summary['Avg_Returns_Predicted_Top50'] = merged_qtr_summary['Avg_Returns_Predicted_Top50'].ffill()
merged_qtr_summary['Avg_Returns_Predicted_Thresh'] = merged_qtr_summary['Avg_Returns_Predicted_Thresh'].ffill()

In [40]:
merged_qtr_summary['Avg_Returns_Predicted_Top30'].fillna(0,inplace=True)
merged_qtr_summary['Avg_Returns_Predicted_Top50'].fillna(0,inplace=True)
merged_qtr_summary['Avg_Returns_Predicted_Thresh'].fillna(0,inplace=True)

C:\Users\43737\AppData\Local\Temp\ipykernel_10612\4263771019.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_qtr_summary['Avg_Returns_Predicted_Top30'].fillna(0,inplace=True)
C:\Users\43737\AppData\Local\Temp\ipykernel_10612\4263771019.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always beh

In [216]:
merged_qtr_summary.head()

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Avg_Returns_Predicted_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,...,Count_TP_Thresh,Count_TN_Top30,Count_TN_Top50,Count_TN_Thresh,Count_FP_Top30,Count_FP_Top50,Count_FP_Thresh,Count_FN_Top30,Count_FN_Top50,Count_FN_Thresh
0,2005-03-31,Train,0.561149,0.460551,0.000000,1.165694,-0.019376,0.604545,0.580525,0.705143,...,0,417,401,438,21,37,0,38,34,47
1,2005-06-30,Train,0.440661,0.540775,0.791960,1.375535,0.045476,0.934874,0.395185,0.834760,...,1,415,401,438,24,38,1,40,34,45
2,2005-09-30,Train,0.103280,0.092842,0.656507,0.426376,0.244277,0.323096,-0.140996,0.333534,...,1,413,396,436,23,40,0,42,39,48
3,2005-12-31,Train,0.178330,0.249479,0.902536,0.932621,0.058583,0.754290,0.119747,0.683142,...,1,412,395,437,25,42,0,43,40,47
4,2006-03-31,Train,0.280704,0.139615,-0.181964,0.503488,0.187746,0.222785,0.092958,0.363874,...,0,418,401,439,23,40,2,42,39,49


In [217]:
df_pred.head()

,Ticker,Quarter,Target,Returns,Rank,Pred_Prob_lr_top50,Set,Top_30_Pred_Prob_lr_top50,Top_50_Pred_Prob_lr_top50,Pred_Category_Top30,Pred_Category_Top50
0,uel,2024-12-31,0,NaN,NaN,1.000000,Test,True,True,FP,FP
1,diacabs,2024-12-31,0,-0.418070,777.0,0.198628,Test,True,True,FP,FP
2,rtpow,2024-12-31,0,-0.283309,635.0,0.147182,Test,True,True,FP,FP
3,wesl,2024-12-31,0,-0.279325,626.0,0.140879,Test,True,True,FP,FP
4,sbglp,2024-12-31,0,-0.732508,803.0,0.137333,Test,True,True,FP,FP


In [218]:
qtr_summary_df.head()

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,delta_returns_pred_vs_index_Top50
0,2005-03-31,Train,0.561149,0.460551,9,417,21,38,13,401,37,34,1.165694,-0.019376,0.604545,0.580525,0.705143,0.479926
1,2005-06-30,Train,0.440661,0.540775,6,415,24,40,12,401,38,34,1.375535,0.045476,0.934874,0.395185,0.834760,0.495299
2,2005-09-30,Train,0.103280,0.092842,7,413,23,42,10,396,40,39,0.426376,0.244277,0.323096,-0.140996,0.333534,-0.151434
3,2005-12-31,Train,0.178330,0.249479,5,412,25,43,8,395,42,40,0.932621,0.058583,0.754290,0.119747,0.683142,0.190896
4,2006-03-31,Train,0.280704,0.139615,7,418,23,42,10,401,40,39,0.503488,0.187746,0.222785,0.092958,0.363874,-0.048131


In [219]:
qtr_summary_df['Set'].value_counts()

Set
Train    68
Test     12
Name: count, dtype: int64

In [41]:
with pd.ExcelWriter(r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\Fund1\results\lag\top50\3yrs_as_test_data_ts\Pred_Vs_Act_LR_time_split_25_07_2025.xlsx", ) as writer:
    merged_qtr_summary.to_excel(writer, sheet_name='Qtr_Summary', index=False)
    qtr_summary_df.to_excel(writer, sheet_name='Qtr_Summary_Top', index=False)
    qtr_summary_df2.to_excel(writer, sheet_name='Qtr_Summary_Thresh', index=False)
    df_pred.to_excel(writer, sheet_name='Pred_Vs_Act_Top', index=False)
    df_pred1.to_excel(writer, sheet_name='Pred_Vs_Act_Thresh', index=False)
    set_summary_df3.to_excel(writer, sheet_name='Set_Summary')

### Top50 Xform

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Flatten, Concatenate, Reshape, RepeatVector
from tensorflow.keras.models import Model
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
import tensorflow as tf
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,confusion_matrix,roc_auc_score
from tensorflow.keras.metrics import AUC, Precision
import os
from scipy.stats import skew

In [2]:

def convert_qtr_to_date(qtr):
    y = int(qtr.split("FY")[1])
    q = int(qtr.split("FY")[0][-1])
    quarter_end_dates = {
        "Q1": "30-06-",
        "Q2": "30-09-",
        "Q3": "31-12-",
        "Q4": "31-03-"
    }
    y = y if q==4 else y-1
    fiscal_year = 2000 + int(y)
    end_date = quarter_end_dates.get(f'Q{q}')
    if end_date:
        return end_date + str(fiscal_year)  # Concatenate the end date with the full year
    else:
        raise ValueError("Invalid quarter format")

In [3]:
train_data_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\Fund1\data\lag\train_data_after_multicollinearity_correlation_3yrs_test_size_time_based_split_2025_07_25.xlsx"
test_data_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\Fund1\data\lag\test_data_after_multicollinearity_correlation_3yrs_test_size_time_based_split_2025_07_25.xlsx"
master_data_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\data\Time_Series_Master_wo_Index_Last20Yrs1.xlsx"

df_train = pd.read_excel(train_data_path)
df_test = pd.read_excel(test_data_path)
df_master = pd.read_excel(master_data_path)
qtrly_returns_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\data\lag\Quarterly_Price_Returns_Labels.xlsx"

In [4]:
df_train.reset_index(drop=True,inplace=True)
df_test.reset_index(drop=True,inplace=True)

In [448]:
df_train.columns

Index(['Ticker', 'Quarter', 'TMR', 'Momentum', 'Beta', 'Alpha', 'EP', 'BP',
       'SP', 'Dividend_Flag', 'DE', 'EPS', 'Zscore', 'Target'],
      dtype='object')

##### Scaling

In [5]:
scaler = StandardScaler()

In [6]:
df_train.columns

Index(['Ticker', 'Quarter', 'TMR', 'Momentum', 'Beta', 'Alpha', 'EP', 'BP',
       'SP', 'Dividend_Flag', 'DE', 'EPS', 'Zscore', 'Target'],
      dtype='object')

In [7]:
vif_cols = list(df_train.columns)
for col in ['Ticker','Quarter','Set','Target']:
    if col in vif_cols:
        vif_cols.remove(col)

In [452]:
vif_cols

['TMR',
 'Momentum',
 'Beta',
 'Alpha',
 'EP',
 'BP',
 'SP',
 'Dividend_Flag',
 'DE',
 'EPS',
 'Zscore']

In [8]:
df_test.replace([np.inf, -np.inf], 0, inplace=True)

In [454]:
# df_test.replace([np.inf, -np.inf], 0, inplace=True)

In [9]:
train_data = scaler.fit_transform(df_train[vif_cols])
test_data = scaler.transform(df_test[vif_cols])

In [10]:
df_train1 = pd.DataFrame(train_data,columns=vif_cols)
df_test1 = pd.DataFrame(test_data,columns=vif_cols)

In [11]:
for col in ['Ticker','Target', 'Quarter']:
    df_train1[col] = df_train[col]
    df_test1[col] = df_test[col]

#### tgt definition

In [12]:
df_master['Quarter'] = pd.to_datetime(df_master['Quarter'].apply(convert_qtr_to_date),dayfirst=True)
df_train1 = df_train1.merge(df_master[['Ticker','Quarter','Returns','Rank']],on=['Ticker','Quarter'])
df_test1 = df_test1.merge(df_master[['Ticker','Quarter','Returns','Rank']],on=['Ticker','Quarter'])
df_train1['Target'] = np.where(df_train1['Rank']<=50,1,0)
df_test1['Target'] = np.where(df_test1['Rank']<=50,1,0)

In [13]:
# ...existing code...
from sklearn.preprocessing import PowerTransformer
from scipy.stats import boxcox
import numpy as np

best_transforms = {}
for col in vif_cols:
    x = df_train[col].dropna()
    results = {}
    if (x <= 0).any():
        # Only Yeo-Johnson possible
        pt = PowerTransformer(method='yeo-johnson')
        x_yj = pt.fit_transform(x.values.reshape(-1, 1)).flatten()
        results['yeo-johnson'] = abs(skew(x_yj))
    else:
        x_log = np.log1p(x)
        results['log1p'] = abs(skew(x_log))
        try:
            x_bc, _ = boxcox(x)
            results['boxcox'] = abs(skew(x_bc))
        except Exception:
            results['boxcox'] = np.nan
    # Always include original for comparison
    results['original'] = abs(skew(x))
    # Pick the transform with minimum absolute skewness
    best = min(results, key=lambda k: results[k] if not np.isnan(results[k]) else np.inf)
    best_transforms[col] = best
    print(f"{col}: Best transform = {best}, skewness = {results[best]:.3f}")
# ...existing code...
#### Power xforms
# ...existing code...
from sklearn.preprocessing import PowerTransformer
from scipy.stats import boxcox
import numpy as np

# 1. Decide best transform for each feature based on your previous results
# best_transform_map = {
#     'TMR': 'yeo-johnson',
#     'Momentum': 'yeo-johnson',
#     'Beta': 'yeo-johnson',
#     'Alpha': 'yeo-johnson',
#     'EP': 'boxcox',
#     'BP': 'yeo-johnson',
#     'SP': 'boxcox',
#     'EPS': 'yeo-johnson',
#     'Zscore': 'yeo-johnson',
#     'FCF': 'yeo-johnson',
#     'ICR': 'yeo-johnson',
#     'Year': 'log1p',
#     'Quarter_number': 'boxcox',  # or 'log1p', both are close
#     # 'Target': 'none',  # Target is not a feature
# }

def transform_feature(col, x, method):
    if method == 'yeo-johnson':
        pt = PowerTransformer(method='yeo-johnson')
        return pt.fit_transform(x.values.reshape(-1, 1)).flatten()
    elif method == 'boxcox':
        # Box-Cox requires strictly positive values
        # Add a small constant if min(x) == 0
        x_pos = x.copy()
        if (x_pos <= 0).any():
            x_pos = x_pos - x_pos.min() + 1e-6
        return boxcox(x_pos)[0]
    elif method == 'log1p':
        x_pos = x.copy()
        if (x_pos < 0).any():
            x_pos = x_pos - x_pos.min() + 1e-6
        return np.log1p(x_pos)
    else:
        return x.values

# 2. Apply transforms to train and test
df_train_trans = df_train.copy()
df_test_trans = df_test.copy()

for col in vif_cols:
    method = best_transforms.get(col, 'none')
    if method != 'none':
        df_train_trans[col] = transform_feature(col, df_train[col], method)
        df_test_trans[col] = transform_feature(col, df_test[col], method)

# 3. Scaling
scaler = StandardScaler()
train_data = scaler.fit_transform(df_train_trans[vif_cols])
test_data = scaler.transform(df_test_trans[vif_cols])

df_train1 = pd.DataFrame(train_data, columns=vif_cols)
df_test1 = pd.DataFrame(test_data, columns=vif_cols)

for col in ['Ticker', 'Target', 'Quarter']:
    df_train1[col] = df_train[col]
    df_test1[col] = df_test[col]

# 4. Prepare X_train, y_train, X_test, y_test
features = vif_cols.copy()
X_train = df_train1[features].values
y_train = df_train1['Target'].values
X_test = df_test1[features].values
y_test = df_test1['Target'].values

# If your model expects 3D input (samples, timesteps, features):
# X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
# X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
# ...existing code...

TMR: Best transform = yeo-johnson, skewness = 1.374
Momentum: Best transform = yeo-johnson, skewness = 0.373
Beta: Best transform = yeo-johnson, skewness = 4.617
Alpha: Best transform = yeo-johnson, skewness = 2.939
EP: Best transform = boxcox, skewness = 0.110
BP: Best transform = yeo-johnson, skewness = 0.206
SP: Best transform = boxcox, skewness = 0.044
Dividend_Flag: Best transform = original, skewness = 1.256
DE: Best transform = yeo-johnson, skewness = 0.049
EPS: Best transform = yeo-johnson, skewness = 1.697
Zscore: Best transform = yeo-johnson, skewness = 0.086
X_train shape: (42985, 11)
y_train shape: (42985,)
X_test shape: (10788, 11)
y_test shape: (10788,)


In [460]:
features = vif_cols.copy()
features

['TMR',
 'Momentum',
 'Beta',
 'Alpha',
 'EP',
 'BP',
 'SP',
 'Dividend_Flag',
 'DE',
 'EPS',
 'Zscore']

#### Model

In [14]:
# features.remove("Quarter")
X_train = df_train1[features].values
y_train = df_train1['Target'].values

X_test = df_test1[features].values
y_test = df_test1['Target'].values

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd
import numpy as np

def calculate_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'roc':roc_auc_score(y_true,y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
        'confusion_matrix': confusion_matrix(y_true, y_pred)
    }

def train_logistic_regression_top50_xform(X_train, y_train, X_test, y_test, k=5):
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    fold_metrics = []

    for train_index, val_index in kf.split(X_train):
        X_train_fold, X_val = X_train[train_index], X_train[val_index]
        y_train_fold, y_val = y_train[train_index], y_train[val_index]

        model = LogisticRegression(max_iter=1000)
        model.fit(X_train_fold, y_train_fold)

        y_val_pred = model.predict(X_val)
        metrics = calculate_metrics(y_val, y_val_pred)
        fold_metrics.append(metrics)

    # Average scalar metrics
    avg_metrics = {
        key: np.mean([m[key] for m in fold_metrics]) for key in ['accuracy','roc', 'precision', 'recall', 'f1']
    }

    # Sum confusion matrices
    conf_matrix_sum = np.sum([m['confusion_matrix'] for m in fold_metrics], axis=0)


    # Final model on full training set
    final_model = LogisticRegression(max_iter=1000)
    final_model.fit(X_train, y_train)
    y_test_pred = final_model.predict(X_test)
    test_metrics = calculate_metrics(y_test, y_test_pred)

    # Combine into DataFrame
    metrics_df = pd.DataFrame({
        'set': ['cross_val_avg', 'test'],
        'accuracy': [avg_metrics['accuracy'], test_metrics['accuracy']],
        'roc':[avg_metrics['roc'],test_metrics['roc']],
        'precision': [avg_metrics['precision'], test_metrics['precision']],
        'recall': [avg_metrics['recall'], test_metrics['recall']],
        'f1': [avg_metrics['f1'], test_metrics['f1']],
        'confusion_matrix': [conf_matrix_sum, test_metrics['confusion_matrix']]
    })

    return metrics_df,model

# Reshape if needed
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)

metrics_df,lr_model_top50_xform = train_logistic_regression_top50_xform(X_train, y_train, X_test, y_test)

metrics_df

,set,accuracy,roc,precision,recall,f1,confusion_matrix
0,cross_val_avg,0.704525,0.545714,0.596134,0.131084,0.214829,"[[28546, 1179], [11522, 1738]]"
1,test,0.789303,0.557286,0.532824,0.150691,0.234938,"[[8166, 306], [1967, 349]]"


In [462]:
import pickle as pkl

In [ ]:
with open(r"D:\Python\Final Quant Fund\Quant Fund\Fund1\src\3yrs_as_test_data_ts\lr_top50_xform.pkl","wb") as file:
    pkl.dump(lr_model_top50_xform, file)

In [464]:
metrics_df

,set,accuracy,roc,precision,recall,f1,confusion_matrix
0,cross_val_avg,0.954984,0.499927,0.0,0.0,0.0,"[[41050, 6], [1929, 0]]"
1,test,0.967557,0.499808,0.0,0.0,0.0,"[[10438, 4], [346, 0]]"


In [15]:
pred_probs_test = lr_model_top50_xform.predict_proba(X_test)[:,1]
pred_probs_train = lr_model_top50_xform.predict_proba(X_train)[:,1]

In [16]:
np.isnan(pred_probs_test).sum()

np.int64(0)

In [17]:
np.isnan(pred_probs_train).sum()

np.int64(0)

In [18]:

def calculate_metrics1(y_true, y_pred_label,y_pred_prob):
    accuracy = accuracy_score(y_true, y_pred_label)
    precision = precision_score(y_true, y_pred_label)
    recall = recall_score(y_true, y_pred_label)
    f1 = f1_score(y_true, y_pred_label)
    confusion = confusion_matrix(y_true,y_pred_label)
    roc_score = roc_auc_score(y_true,y_pred_prob)
    
    return {
        'accuracy': accuracy,
        'roc':roc_score,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix':confusion
    }



In [19]:
train_metrics1 = calculate_metrics1(y_train,(pred_probs_train>=0.5).astype(int),pred_probs_train)
test_metrics1 = calculate_metrics1(y_test,(pred_probs_test>=0.5).astype(int),pred_probs_test)

In [20]:
train_metrics_df1 = pd.DataFrame([train_metrics1])
train_metrics_df1['Set'] = "Train"
test_metrics_df1 = pd.DataFrame([test_metrics1])
test_metrics_df1['Set'] = "Test"
model_metrics1 = pd.concat([train_metrics_df1,test_metrics_df1],axis=0)
model_metrics1.reset_index(drop=True,inplace=True)
model_metrics1.head()

,accuracy,roc,precision,recall,f1,confusion_matrix,Set
0,0.704571,0.660936,0.595311,0.132127,0.216256,"[[28534, 1191], [11508, 1752]]",Train
1,0.789396,0.694597,0.533133,0.152850,0.237584,"[[8162, 310], [1962, 354]]",Test


In [21]:
model_name = 'lr_top50_xform'
top_n = 30

In [22]:

def convert_qtr_to_date(qtr):
    y = int(qtr.split("FY")[1])
    q = int(qtr.split("FY")[0][-1])
    quarter_end_dates = {
        "Q1": "30-06-",
        "Q2": "30-09-",
        "Q3": "31-12-",
        "Q4": "31-03-"
    }
    y = y if q==4 else y-1
    fiscal_year = 2000 + int(y)
    end_date = quarter_end_dates.get(f'Q{q}')
    if end_date:
        return end_date + str(fiscal_year)  # Concatenate the end date with the full year
    else:
        raise ValueError("Invalid quarter format")

In [23]:
df_master = pd.read_excel(master_data_path)

In [24]:
df_master['Quarter'] = pd.to_datetime(df_master['Quarter'].apply(convert_qtr_to_date),dayfirst=True)

In [25]:
def label_prediction(row, model_name,top_n):
    in_top_50 = row[f'Top_{top_n}_Pred_Prob_{model_name}']
    target = row['Target']
    
    if in_top_50 and target == 1:
        return 'TP'
    elif in_top_50 and target == 0:
        return 'FP'
    elif not in_top_50 and target == 0:
        return 'TN'
    elif not in_top_50 and target == 1:
        return 'FN'

def flag_top50_nlargest(group, pred_prob_col,top_n):
    # If all pred_probs are zero, return all False
    if group[pred_prob_col].sum() == 0:
        return pd.Series([False] * len(group), index=group.index)
    # Use nlargest to get exactly 50
    top50_idx = group[pred_prob_col].nlargest(top_n).index
    flag = pd.Series(False, index=group.index)
    flag.loc[top50_idx] = True
    return flag

def label_prediction1(row, model_name):
    has_more_prob_than_threshold = row[f'Pred_Prob_Threshold_{model_name}']
    target = row['Target']
    
    if has_more_prob_than_threshold and target == 1:
        return 'TP'
    elif has_more_prob_than_threshold and target == 0:
        return 'FP'
    elif not has_more_prob_than_threshold and target == 0:
        return 'TN'
    elif not has_more_prob_than_threshold and target == 1:
        return 'FN'
    
def add_suffix_to_cols(df, suffix, exclude_cols):
    return df.rename(
        columns={col: f"{col}{suffix}" for col in df.columns if col not in exclude_cols + ['Quarter', 'Set']}
    )


In [26]:

if len(set(['Rank','Returns'])-set(df_train1.columns))!=0:
    print("There are no rank and returns columns, merging now..!")
    df_train1 = df_train1.merge(df_master[['Ticker','Quarter','Returns','Rank']],on=['Ticker','Quarter'],how='left')
    df_test1 = df_test1.merge(df_master[['Ticker','Quarter','Returns','Rank']],on=['Ticker','Quarter'],how='left')

if np.isnan(pred_probs_train).all():
    print("All predictionsa re NaNs")

df_train_pred = df_train1[['Ticker','Quarter','Target','Returns','Rank']]
df_test_pred = df_test1[['Ticker','Quarter','Target','Returns','Rank']]

df_test_pred[f'Pred_Prob_{model_name}'] = pred_probs_test.reshape(-1)
df_train_pred[f'Pred_Prob_{model_name}'] = pred_probs_train.reshape(-1)

#### Top30 as fund - irrespective of the prob values
df_train_pred.loc[:,'Set'] = 'Train'
df_test_pred.loc[:,'Set'] = 'Test'
df_pred = pd.concat([df_train_pred,df_test_pred],axis=0)



df_pred.reset_index(drop=True,inplace=True)
df_pred.sort_values(by=['Quarter',f'Pred_Prob_{model_name}'],ascending=False,inplace=True)
df_pred[f'Top_{top_n}_Pred_Prob_{model_name}'] = df_pred.groupby('Quarter', group_keys=False).apply(flag_top50_nlargest, f'Pred_Prob_{model_name}',top_n)

df_pred[f'Top_50_Pred_Prob_{model_name}'] = df_pred.groupby('Quarter', group_keys=False).apply(flag_top50_nlargest, f'Pred_Prob_{model_name}',50)
# df_pred[(df_pred['Top_{top_n}_Pred_Prob_lstm_top50']==True)&(df_pred['Quarter']=='2005-12-31')]

# test_qtr_preds = df_pred[(df_pred['Quarter']=='2005-12-31')]['Pred_Prob_lstm_top50']
# test_qtr_preds[test_qtr_preds>0.9].shape
df_pred['Pred_Category_Top30'] = df_pred.apply(lambda row: label_prediction(row, model_name=model_name,top_n=30), axis=1)
df_pred['Pred_Category_Top50'] = df_pred.apply(lambda row: label_prediction(row, model_name=model_name,top_n=50), axis=1)
df_pred.reset_index(drop=True,inplace=True)

qtr_summary_df = df_pred.groupby('Quarter').agg(
                    Set = ('Set','first'),
                    Avg_Returns_Predicted_Top30=('Returns',lambda x: x[df_pred.loc[x.index, f'Top_30_Pred_Prob_{model_name}'] == True].mean()),
                    Avg_Returns_Predicted_Top50=('Returns',lambda x: x[df_pred.loc[x.index, f'Top_50_Pred_Prob_{model_name}'] == True].mean()),
                    Count_TP_Top30=('Pred_Category_Top30', lambda x: (x == 'TP').sum()),
                    Count_TN_Top30=('Pred_Category_Top30', lambda x: (x == 'TN').sum()),
                    Count_FP_Top30=('Pred_Category_Top30', lambda x: (x == 'FP').sum()),
                    Count_FN_Top30=('Pred_Category_Top30', lambda x: (x == 'FN').sum()),
                    Count_TP_Top50=('Pred_Category_Top50', lambda x: (x == 'TP').sum()),
                    Count_TN_Top50=('Pred_Category_Top50', lambda x: (x == 'TN').sum()),
                    Count_FP_Top50=('Pred_Category_Top50', lambda x: (x == 'FP').sum()),
                    Count_FN_Top50=('Pred_Category_Top50', lambda x: (x == 'FN').sum())
                    
                    ).reset_index()

# set_summary_df = qtr_summary_df.groupby('Set').agg(
#     Count_TP = ('Count_TP','sum'),
#     Count_TN = ('Count_TN','sum'),
#     Count_FP = ('Count_FP','sum'),
#     Count_FN = ('Count_FN','sum')
# )

df_pred1 = df_pred[['Ticker','Quarter','Target','Returns','Rank',f'Pred_Prob_{model_name}','Set']].copy()
df_pred1[f'Pred_Prob_Threshold_{model_name}'] = df_pred1[f'Pred_Prob_{model_name}'].between(0.5,1)



df_pred1['Pred_Category'] = df_pred1.apply(lambda row: label_prediction1(row, model_name=model_name), axis=1)
quarterly_returns_f2 = df_pred1.groupby('Quarter').agg(
                    Set = ('Set','first'),
                    Avg_Returns_Predicted_Thresh=('Returns', lambda x: x[df_pred1.loc[x.index, f'Pred_Prob_Threshold_{model_name}'] == True].mean()),
                    Count_TP_Thresh=('Pred_Category', lambda x: (x == 'TP').sum()),
                    Count_TN_Thresh=('Pred_Category', lambda x: (x == 'TN').sum()),
                    Count_FP_Thresh=('Pred_Category', lambda x: (x == 'FP').sum()),
                    Count_FN_Thresh=('Pred_Category', lambda x: (x == 'FN').sum())
                    ).reset_index()
# if quarterly_returns_f2['Avg_Returns_Predicted'].isna().any():
#     print("there are quarters where there were no 1s by the model")
# else:
#     print("there are no quarters where there were no 1s by the model")
qtr_summary_df2 = pd.DataFrame(quarterly_returns_f2)


#### Actual and Index returns mapping
##### Actual returns
act_returns = df_pred[df_pred['Target']==1].groupby(by='Quarter')['Returns'].mean()
df_act_returns = pd.DataFrame(act_returns)
df_act_returns.reset_index(inplace=True)
df_act_returns.rename(columns={'Returns':'Avg_Returns_Actual'},inplace=True)
qtr_summary_df = qtr_summary_df.merge(df_act_returns,on='Quarter',how='left')

##### Index returns - Qtrly
df_index = pd.read_excel(qtrly_returns_path,sheet_name='Qtrly_Stock_Returns')
df_index1 = df_index[df_index['Ticker'] == 'Nsemcap'].drop(columns=['Industry']).set_index(keys=['Ticker'])
df_index1 = df_index1.T
df_index1.reset_index(inplace=True)
df_index1.rename(columns={'index':'Quarter','Nsemcap':'Index_Returns'},inplace=True)

df_index1['Quarter'] = df_index1['Quarter'].apply(convert_qtr_to_date)
df_index1['Quarter'] = pd.to_datetime(df_index1['Quarter'],dayfirst=True)

qtr_summary_df = qtr_summary_df.merge(df_index1,how='left',on='Quarter')
qtr_summary_df['delta_returns_act_vs_pred_Top30'] = qtr_summary_df['Avg_Returns_Actual']-qtr_summary_df['Avg_Returns_Predicted_Top30']
qtr_summary_df['delta_returns_pred_vs_index_Top30'] = qtr_summary_df['Avg_Returns_Predicted_Top30']-qtr_summary_df['Index_Returns']
qtr_summary_df['delta_returns_act_vs_pred_Top50'] = qtr_summary_df['Avg_Returns_Actual']-qtr_summary_df['Avg_Returns_Predicted_Top50']
qtr_summary_df['delta_returns_pred_vs_index_Top50'] = qtr_summary_df['Avg_Returns_Predicted_Top50']-qtr_summary_df['Index_Returns']

qtr_summary_df2 = qtr_summary_df2.merge(df_act_returns,on='Quarter',how='left')
qtr_summary_df2 = qtr_summary_df2.merge(df_index1,on='Quarter',how='left')

qtr_summary_df2['delta_returns_act_vs_pred_Thresh'] = qtr_summary_df2['Avg_Returns_Actual']-qtr_summary_df2['Avg_Returns_Predicted_Thresh']
qtr_summary_df2['delta_returns_pred_vs_index_Thresh'] = qtr_summary_df2['Avg_Returns_Predicted_Thresh']-qtr_summary_df2['Index_Returns']

# set_summary_df2 = qtr_summary_df2.groupby('Set').agg(
#     Count_TP = ('Count_TP','sum'),
#     Count_TN = ('Count_TN','sum'),
#     Count_FP = ('Count_FP','sum'),
#     Count_FN = ('Count_FN','sum')
# )

exclude_cols = ['Avg_Returns_Actual', 'Index_Returns']
# suffix_map = {
#     'qtr_summary_df2': '_thresh',
#     'qtr_summary_df': '_top'
# }


# Reset index for merge
qtr_summary_df2_reset = qtr_summary_df2.reset_index(drop=True)
qtr_summary_df_reset = qtr_summary_df.reset_index(drop=True)

# Add suffixes
# qtr_summary_df2_suff = add_suffix_to_cols(qtr_summary_df2_reset, suffix_map['qtr_summary_df2'], exclude_cols)
# qtr_summary_df_suff = add_suffix_to_cols(qtr_summary_df_reset, suffix_map['qtr_summary_df'], exclude_cols)
# ...existing code...

# Merge as before
merged_qtr_summary = pd.merge(
    qtr_summary_df_reset, qtr_summary_df2_reset,
    on=['Quarter', 'Set', 'Avg_Returns_Actual', 'Index_Returns'],
    how='inner',
    suffixes=('', '')
)

# Define the desired column order
ordered_cols = [
    'Quarter','Set',
    'Avg_Returns_Predicted_Top30','Avg_Returns_Predicted_Top50',
    'Avg_Returns_Predicted_Thresh',
    'Avg_Returns_Actual', 'Index_Returns',
    'delta_returns_act_vs_pred_Top30','delta_returns_pred_vs_index_Top30',
    'delta_returns_act_vs_pred_Top50','delta_returns_pred_vs_index_Top50',
    'delta_returns_act_vs_pred_Thresh','delta_returns_pred_vs_index_Thresh',
    'Count_TP_Top30', 'Count_TP_Top50', 'Count_TP_Thresh',
    'Count_TN_Top30', 'Count_TN_Top50', 'Count_TN_Thresh',
    'Count_FP_Top30', 'Count_FP_Top50', 'Count_FP_Thresh',
    'Count_FN_Top30', 'Count_FN_Top50', 'Count_FN_Thresh'
]

# Add any other columns that might exist but are not in the list
ordered_cols += [col for col in merged_qtr_summary.columns if col not in ordered_cols]

# Reorder columns
merged_qtr_summary = merged_qtr_summary[ordered_cols]
# merged_qtr_summary.reset_index(inplace=True)
merged_qtr_summary.head()

There are no rank and returns columns, merging now..!


C:\Users\43737\AppData\Local\Temp\ipykernel_39152\4113010357.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test_pred[f'Pred_Prob_{model_name}'] = pred_probs_test.reshape(-1)
C:\Users\43737\AppData\Local\Temp\ipykernel_39152\4113010357.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train_pred[f'Pred_Prob_{model_name}'] = pred_probs_train.reshape(-1)
C:\Users\43737\AppData\Local\Temp\ipykernel_39152\4113010357.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice 

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Avg_Returns_Predicted_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,...,Count_TP_Thresh,Count_TN_Top30,Count_TN_Top50,Count_TN_Thresh,Count_FP_Top30,Count_FP_Top50,Count_FP_Thresh,Count_FN_Top30,Count_FN_Top50,Count_FN_Thresh
0,2005-03-31,Train,0.202669,0.324026,0.343297,0.462237,-0.019376,0.259568,0.222045,0.138212,...,65,285,281,267,6,10,24,170,154,129
1,2005-06-30,Train,0.304000,0.330705,0.314607,0.495418,0.045476,0.191418,0.258525,0.164713,...,65,283,277,261,9,15,31,172,158,128
2,2005-09-30,Train,0.129293,0.092910,0.077915,0.180268,0.244277,0.050974,-0.114984,0.087358,...,32,278,269,265,11,20,24,177,166,164
3,2005-12-31,Train,0.424620,0.346235,0.445298,0.407782,0.058583,-0.016837,0.366036,0.061547,...,17,286,276,288,4,14,2,169,159,178
4,2006-03-31,Train,-0.001229,-0.057411,-0.067324,0.018142,0.187746,0.019371,-0.188975,0.075552,...,42,292,286,285,3,9,10,168,154,153


In [27]:
merged_qtr_summary['Avg_Returns_Predicted_Thresh'].isna().all()

np.False_

In [28]:
merged_qtr_summary.columns

Index(['Quarter', 'Set', 'Avg_Returns_Predicted_Top30',
       'Avg_Returns_Predicted_Top50', 'Avg_Returns_Predicted_Thresh',
       'Avg_Returns_Actual', 'Index_Returns',
       'delta_returns_act_vs_pred_Top30', 'delta_returns_pred_vs_index_Top30',
       'delta_returns_act_vs_pred_Top50', 'delta_returns_pred_vs_index_Top50',
       'delta_returns_act_vs_pred_Thresh',
       'delta_returns_pred_vs_index_Thresh', 'Count_TP_Top30',
       'Count_TP_Top50', 'Count_TP_Thresh', 'Count_TN_Top30', 'Count_TN_Top50',
       'Count_TN_Thresh', 'Count_FP_Top30', 'Count_FP_Top50',
       'Count_FP_Thresh', 'Count_FN_Top30', 'Count_FN_Top50',
       'Count_FN_Thresh'],
      dtype='object')

In [29]:
df_pred[df_pred['Set']=='Test'][f'Pred_Prob_{model_name}'].unique()

array([0.7347319 , 0.63331719, 0.62042498, ..., 0.01804842, 0.01567157,
       0.01388437])

In [30]:
set_summary_df3 =  merged_qtr_summary.groupby('Set').agg(
    Count_TP_Thresh = ('Count_TP_Thresh','sum'),
    Count_TN_Thresh = ('Count_TN_Thresh','sum'),
    Count_FP_Thresh = ('Count_FP_Thresh','sum'),
    Count_FN_Thresh = ('Count_FN_Thresh','sum'),
    Count_TP_Top30 = ('Count_TP_Top30','sum'),
    Count_TN_Top30 = ('Count_TN_Top30','sum'),
    Count_FP_Top30 = ('Count_FP_Top30','sum'),
    Count_FN_Top30 = ('Count_FN_Top30','sum'),
    Count_TP_Top50 = ('Count_TP_Top50','sum'),
    Count_TN_Top50 = ('Count_TN_Top50','sum'),
    Count_FP_Top50 = ('Count_FP_Top50','sum'),
    Count_FN_Top50 = ('Count_FN_Top50','sum')
)
set_summary_df3

,Count_TP_Thresh,Count_TN_Thresh,Count_FP_Thresh,Count_FN_Thresh,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50
Set,,,,,,,,,,,,
Test,354,8162,310,1962,197,8309,163,2119,324,8196,276,1992
Train,1752,28534,1191,11508,1225,28910,815,12035,1986,28311,1414,11274


In [31]:

def calc_metrics(tp, tn, fp, fn):
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return accuracy, precision, recall, f1


In [32]:
for suffix in ['_Top30','_Top50', '_Thresh']:
    print(f"\nMetrics for {suffix}:")
    for set_name, row in set_summary_df3.iterrows():
        tp = row[f'Count_TP{suffix}']
        tn = row[f'Count_TN{suffix}']
        fp = row[f'Count_FP{suffix}']
        fn = row[f'Count_FN{suffix}']
        accuracy, precision, recall, f1 = calc_metrics(tp, tn, fp, fn)
        print(f"{set_name}: accuracy={accuracy:.4f}, precision={precision:.4f}, recall={recall:.4f}, f1={f1:.4f}")


Metrics for _Top30:
Test: accuracy=0.7885, precision=0.5472, recall=0.0851, f1=0.1472
Train: accuracy=0.7011, precision=0.6005, recall=0.0924, f1=0.1601

Metrics for _Top50:
Test: accuracy=0.7898, precision=0.5400, recall=0.1399, f1=0.2222
Train: accuracy=0.7048, precision=0.5841, recall=0.1498, f1=0.2384

Metrics for _Thresh:
Test: accuracy=0.7894, precision=0.5331, recall=0.1528, f1=0.2376
Train: accuracy=0.7046, precision=0.5953, recall=0.1321, f1=0.2163


In [33]:
if merged_qtr_summary['Avg_Returns_Predicted_Thresh'].mean()==merged_qtr_summary['Avg_Returns_Predicted_Top30'].mean():
    print("Both the stock picking strategies (Top 30 and Pred prob theshold) have same returns")
if merged_qtr_summary['Avg_Returns_Predicted_Thresh'].mean()==merged_qtr_summary['Avg_Returns_Predicted_Top50'].mean():
    print("Both the stock picking strategies (Top 50 and Pred prob theshold) have same returns")
if merged_qtr_summary['Avg_Returns_Predicted_Top30'].mean()==merged_qtr_summary['Avg_Returns_Predicted_Top50'].mean():
    print("Both the stock picking strategies (Top 30 and Top 50) have same returns")
else:
    print("All of the stock picking strategies (Top 30, Top 50 and Pred prob theshold) have different returns")

All of the stock picking strategies (Top 30, Top 50 and Pred prob theshold) have different returns


In [34]:
qtr_summary_df[qtr_summary_df['Quarter']=='2021-09-30']

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,delta_returns_pred_vs_index_Top50
66,2021-09-30,Train,0.186083,0.146025,16,613,14,179,26,603,24,169,0.362195,0.126554,0.176112,0.059529,0.216171,0.019471


In [35]:
qtr_summary_df2[qtr_summary_df2['Quarter']=='2021-09-30']

,Quarter,Set,Avg_Returns_Predicted_Thresh,Count_TP_Thresh,Count_TN_Thresh,Count_FP_Thresh,Count_FN_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Thresh,delta_returns_pred_vs_index_Thresh
66,2021-09-30,Train,0.147607,31,600,27,164,0.362195,0.126554,0.214588,0.021053


In [486]:
df_pred.columns

Index(['Ticker', 'Quarter', 'Target', 'Returns', 'Rank',
       'Pred_Prob_lr_top50_xform', 'Set', 'Top_30_Pred_Prob_lr_top50_xform',
       'Top_50_Pred_Prob_lr_top50_xform', 'Pred_Category_Top30',
       'Pred_Category_Top50'],
      dtype='object')

In [487]:
df_pred1.columns

Index(['Ticker', 'Quarter', 'Target', 'Returns', 'Rank',
       'Pred_Prob_lr_top50_xform', 'Set', 'Pred_Prob_Threshold_lr_top50_xform',
       'Pred_Category'],
      dtype='object')

In [488]:
merged_qtr_summary[merged_qtr_summary['Quarter']=='2021-09-30']

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Avg_Returns_Predicted_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,...,Count_TP_Thresh,Count_TN_Top30,Count_TN_Top50,Count_TN_Thresh,Count_FP_Top30,Count_FP_Top50,Count_FP_Thresh,Count_FN_Top30,Count_FN_Top50,Count_FN_Thresh
66,2021-09-30,Train,0.408642,0.400875,-0.340932,1.519144,0.126554,1.110502,0.282088,1.118268,...,0,769,752,792,24,41,1,23,20,29


In [36]:
merged_qtr_summary[merged_qtr_summary['Avg_Returns_Predicted_Top30'].isna()].shape

(2, 25)

In [37]:
df_pred[df_pred['Set']=='Test'][f'Pred_Prob_{model_name}'].unique()

array([0.7347319 , 0.63331719, 0.62042498, ..., 0.01804842, 0.01567157,
       0.01388437])

In [38]:
df_pred[df_pred['Set']=='Test'][f'Pred_Prob_{model_name}'].describe()

count    10788.000000
mean         0.289112
std          0.118714
min          0.000281
25%          0.204663
50%          0.268304
75%          0.353909
max          0.917692
Name: Pred_Prob_lr_top50_xform, dtype: float64

In [39]:
merged_qtr_summary['Quarter'].is_monotonic_increasing

True

In [493]:
set_summary_df3

,Count_TP_Thresh,Count_TN_Thresh,Count_FP_Thresh,Count_FN_Thresh,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50
Set,,,,,,,,,,,,
Test,0,10438,4,346,42,10124,318,304,62,9904,538,284
Train,0,41050,6,1929,255,39271,1785,1674,373,38029,3027,1556


In [40]:
merged_qtr_summary['Avg_Returns_Predicted_Top30'] = merged_qtr_summary['Avg_Returns_Predicted_Top30'].ffill()
merged_qtr_summary['Avg_Returns_Predicted_Top50'] = merged_qtr_summary['Avg_Returns_Predicted_Top50'].ffill()
merged_qtr_summary['Avg_Returns_Predicted_Thresh'] = merged_qtr_summary['Avg_Returns_Predicted_Thresh'].ffill()

In [41]:
merged_qtr_summary['Avg_Returns_Predicted_Top30'].fillna(0,inplace=True)
merged_qtr_summary['Avg_Returns_Predicted_Top50'].fillna(0,inplace=True)
merged_qtr_summary['Avg_Returns_Predicted_Thresh'].fillna(0,inplace=True)

C:\Users\43737\AppData\Local\Temp\ipykernel_39152\4263771019.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_qtr_summary['Avg_Returns_Predicted_Top30'].fillna(0,inplace=True)
C:\Users\43737\AppData\Local\Temp\ipykernel_39152\4263771019.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always beh

In [496]:
merged_qtr_summary.head()

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Avg_Returns_Predicted_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,...,Count_TP_Thresh,Count_TN_Top30,Count_TN_Top50,Count_TN_Thresh,Count_FP_Top30,Count_FP_Top50,Count_FP_Thresh,Count_FN_Top30,Count_FN_Top50,Count_FN_Thresh
0,2005-03-31,Train,0.487041,0.436550,0.0,1.455923,-0.019376,0.968882,0.506417,1.019373,...,0,431,415,457,26,42,0,24,20,28
1,2005-06-30,Train,0.517589,0.444255,0.0,1.790164,0.045476,1.272575,0.472114,1.345909,...,0,434,416,458,24,42,0,21,19,27
2,2005-09-30,Train,0.088815,0.052912,0.0,0.496120,0.244277,0.407304,-0.155461,0.443207,...,0,431,412,456,25,44,0,24,23,29
3,2005-12-31,Train,0.299423,0.301841,0.0,1.147137,0.058583,0.847714,0.240840,0.845296,...,0,431,414,457,26,43,0,24,21,28
4,2006-03-31,Train,-0.033331,0.193029,0.0,0.772814,0.187746,0.806145,-0.221077,0.579785,...,0,436,422,461,25,39,0,24,18,29


In [497]:
df_pred.head()

,Ticker,Quarter,Target,Returns,Rank,Pred_Prob_lr_top50_xform,Set,Top_30_Pred_Prob_lr_top50_xform,Top_50_Pred_Prob_lr_top50_xform,Pred_Category_Top30,Pred_Category_Top50
0,uel,2024-12-31,0,NaN,NaN,0.727036,Test,True,True,FP,FP
1,ut,2024-12-31,0,-0.358306,745.0,0.133512,Test,True,True,FP,FP
2,rtpow,2024-12-31,0,-0.283309,635.0,0.133457,Test,True,True,FP,FP
3,sdb,2024-12-31,0,0.122062,36.0,0.119482,Test,True,True,FP,FP
4,dcxindia,2024-12-31,0,-0.341313,730.0,0.115898,Test,True,True,FP,FP


In [498]:
qtr_summary_df.head()

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,delta_returns_pred_vs_index_Top50
0,2005-03-31,Train,0.487041,0.436550,4,431,26,24,8,415,42,20,1.455923,-0.019376,0.968882,0.506417,1.019373,0.455926
1,2005-06-30,Train,0.517589,0.444255,6,434,24,21,8,416,42,19,1.790164,0.045476,1.272575,0.472114,1.345909,0.398779
2,2005-09-30,Train,0.088815,0.052912,5,431,25,24,6,412,44,23,0.496120,0.244277,0.407304,-0.155461,0.443207,-0.191364
3,2005-12-31,Train,0.299423,0.301841,4,431,26,24,7,414,43,21,1.147137,0.058583,0.847714,0.240840,0.845296,0.243258
4,2006-03-31,Train,-0.033331,0.193029,5,436,25,24,11,422,39,18,0.772814,0.187746,0.806145,-0.221077,0.579785,0.005283


In [499]:
qtr_summary_df['Set'].value_counts()

Set
Train    68
Test     12
Name: count, dtype: int64

In [42]:
with pd.ExcelWriter(r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\Fund1\results\lag\top50\3yrs_as_test_data_ts\Pred_Vs_Act_LR_xform_time_split_25_07_2025.xlsx", ) as writer:
    merged_qtr_summary.to_excel(writer, sheet_name='Qtr_Summary', index=False)
    qtr_summary_df.to_excel(writer, sheet_name='Qtr_Summary_Top', index=False)
    qtr_summary_df2.to_excel(writer, sheet_name='Qtr_Summary_Thresh', index=False)
    df_pred.to_excel(writer, sheet_name='Pred_Vs_Act_Top', index=False)
    df_pred1.to_excel(writer, sheet_name='Pred_Vs_Act_Thresh', index=False)
    set_summary_df3.to_excel(writer, sheet_name='Set_Summary')

### Top100 as tgt

In [43]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Flatten, Concatenate, Reshape, RepeatVector
from tensorflow.keras.models import Model
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
import tensorflow as tf
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,confusion_matrix,roc_auc_score
from tensorflow.keras.metrics import AUC, Precision
import os

In [44]:
train_data_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\Fund1\data\lag\train_data_after_multicollinearity_correlation_3yrs_test_size_time_based_split_2025_07_25.xlsx"
test_data_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\Fund1\data\lag\test_data_after_multicollinearity_correlation_3yrs_test_size_time_based_split_2025_07_25.xlsx"
master_data_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\data\Time_Series_Master_wo_Index_Last20Yrs1.xlsx"

df_train = pd.read_excel(train_data_path)
df_test = pd.read_excel(test_data_path)
df_master = pd.read_excel(master_data_path)
qtrly_returns_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\data\lag\Quarterly_Price_Returns_Labels.xlsx"

In [45]:
df_train.reset_index(drop=True,inplace=True)
df_test.reset_index(drop=True,inplace=True)

In [46]:
df_train.columns

Index(['Ticker', 'Quarter', 'TMR', 'Momentum', 'Beta', 'Alpha', 'EP', 'BP',
       'SP', 'Dividend_Flag', 'DE', 'EPS', 'Zscore', 'Target'],
      dtype='object')

##### Scaling

In [47]:
scaler = StandardScaler()

In [48]:
df_train.columns

Index(['Ticker', 'Quarter', 'TMR', 'Momentum', 'Beta', 'Alpha', 'EP', 'BP',
       'SP', 'Dividend_Flag', 'DE', 'EPS', 'Zscore', 'Target'],
      dtype='object')

In [49]:
vif_cols = list(df_train.columns)
for col in ['Ticker','Quarter','Set','Target']:
    if col in vif_cols:
        vif_cols.remove(col)

In [361]:
vif_cols

['TMR',
 'Momentum',
 'Beta',
 'Alpha',
 'EP',
 'BP',
 'SP',
 'Dividend_Flag',
 'DE',
 'EPS',
 'Zscore']

In [50]:
df_test.replace([np.inf, -np.inf], 0, inplace=True)

In [363]:
# df_test.replace([np.inf, -np.inf], 0, inplace=True)

In [51]:
train_data = scaler.fit_transform(df_train[vif_cols])
test_data = scaler.transform(df_test[vif_cols])

In [52]:
df_train1 = pd.DataFrame(train_data,columns=vif_cols)
df_test1 = pd.DataFrame(test_data,columns=vif_cols)

In [53]:
for col in ['Ticker','Target', 'Quarter']:
    df_train1[col] = df_train[col]
    df_test1[col] = df_test[col]

In [54]:
df_master['Quarter'] = pd.to_datetime(df_master['Quarter'].apply(convert_qtr_to_date),dayfirst=True)

In [55]:
df_train1 = df_train1.merge(df_master[['Ticker','Quarter','Returns','Rank']],on=['Ticker','Quarter'])
df_test1 = df_test1.merge(df_master[['Ticker','Quarter','Returns','Rank']],on=['Ticker','Quarter'])

In [56]:
df_train1['Target'] = np.where(df_train1['Rank']<=100,1,0)
df_test1['Target'] = np.where(df_test1['Rank']<=100,1,0)

In [57]:
features = vif_cols.copy()
features

['TMR',
 'Momentum',
 'Beta',
 'Alpha',
 'EP',
 'BP',
 'SP',
 'Dividend_Flag',
 'DE',
 'EPS',
 'Zscore']

#### Model

In [58]:
# features.remove("Quarter")
X_train = df_train1[features].values
y_train = df_train1['Target'].values

X_test = df_test1[features].values
y_test = df_test1['Target'].values

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd
import numpy as np

def calculate_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'roc':roc_auc_score(y_true,y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
        'confusion_matrix': confusion_matrix(y_true, y_pred)
    }

def train_logistic_regression_top100(X_train, y_train, X_test, y_test, k=5):
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    fold_metrics = []

    for train_index, val_index in kf.split(X_train):
        X_train_fold, X_val = X_train[train_index], X_train[val_index]
        y_train_fold, y_val = y_train[train_index], y_train[val_index]

        model = LogisticRegression(max_iter=1000)
        model.fit(X_train_fold, y_train_fold)

        y_val_pred = model.predict(X_val)
        metrics = calculate_metrics(y_val, y_val_pred)
        fold_metrics.append(metrics)

    # Average scalar metrics
    avg_metrics = {
        key: np.mean([m[key] for m in fold_metrics]) for key in ['accuracy','roc', 'precision', 'recall', 'f1']
    }

    # Sum confusion matrices
    conf_matrix_sum = np.sum([m['confusion_matrix'] for m in fold_metrics], axis=0)


    # Final model on full training set
    final_model = LogisticRegression(max_iter=1000)
    final_model.fit(X_train, y_train)
    y_test_pred = final_model.predict(X_test)
    test_metrics = calculate_metrics(y_test, y_test_pred)

    # Combine into DataFrame
    metrics_df = pd.DataFrame({
        'set': ['cross_val_avg', 'test'],
        'accuracy': [avg_metrics['accuracy'], test_metrics['accuracy']],
        'roc':[avg_metrics['roc'],test_metrics['roc']],
        'precision': [avg_metrics['precision'], test_metrics['precision']],
        'recall': [avg_metrics['recall'], test_metrics['recall']],
        'f1': [avg_metrics['f1'], test_metrics['f1']],
        'confusion_matrix': [conf_matrix_sum, test_metrics['confusion_matrix']]
    })

    return metrics_df,model

# Reshape if needed
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)

In [59]:
metrics_df,lr_model_top100 = train_logistic_regression_top100(X_train, y_train, X_test, y_test)

In [60]:
metrics_df

,set,accuracy,roc,precision,recall,f1,confusion_matrix
0,cross_val_avg,0.849855,0.500927,0.353680,0.002947,0.005839,"[[36512, 40], [6414, 19]]"
1,test,0.890805,0.500028,0.111111,0.001718,0.003384,"[[9608, 16], [1162, 2]]"


In [ ]:
with open(r"D:\Python\Final Quant Fund\Quant Fund\Fund1\src\3yrs_as_test_data_ts\lr_top100.pkl","wb") as file:
    pkl.dump(lr_model_top100, file)

In [61]:
pred_probs_test = lr_model_top100.predict_proba(X_test)[:,1]
pred_probs_train = lr_model_top100.predict_proba(X_train)[:,1]

In [62]:
np.isnan(pred_probs_test).sum()

np.int64(0)

In [63]:
np.isnan(pred_probs_train).sum()

np.int64(0)

In [64]:

def calculate_metrics1(y_true, y_pred_label,y_pred_prob):
    accuracy = accuracy_score(y_true, y_pred_label)
    precision = precision_score(y_true, y_pred_label)
    recall = recall_score(y_true, y_pred_label)
    f1 = f1_score(y_true, y_pred_label)
    confusion = confusion_matrix(y_true,y_pred_label)
    roc_score = roc_auc_score(y_true,y_pred_prob)
    
    return {
        'accuracy': accuracy,
        'roc':roc_score,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix':confusion
    }



In [65]:
train_metrics1 = calculate_metrics1(y_train,(pred_probs_train>=0.5).astype(int),pred_probs_train)
test_metrics1 = calculate_metrics1(y_test,(pred_probs_test>=0.5).astype(int),pred_probs_test)

In [66]:
train_metrics_df1 = pd.DataFrame([train_metrics1])
train_metrics_df1['Set'] = "Train"
test_metrics_df1 = pd.DataFrame([test_metrics1])
test_metrics_df1['Set'] = "Test"
model_metrics1 = pd.concat([train_metrics_df1,test_metrics_df1],axis=0)
model_metrics1.reset_index(drop=True,inplace=True)
model_metrics1.head()

,accuracy,roc,precision,recall,f1,confusion_matrix,Set
0,0.849855,0.582998,0.301887,0.002487,0.004934,"[[36515, 37], [6417, 16]]",Train
1,0.890805,0.604045,0.111111,0.001718,0.003384,"[[9608, 16], [1162, 2]]",Test


In [67]:
model_name = 'lr_top100'
top_n = 30

In [68]:

def convert_qtr_to_date(qtr):
    y = int(qtr.split("FY")[1])
    q = int(qtr.split("FY")[0][-1])
    quarter_end_dates = {
        "Q1": "30-06-",
        "Q2": "30-09-",
        "Q3": "31-12-",
        "Q4": "31-03-"
    }
    y = y if q==4 else y-1
    fiscal_year = 2000 + int(y)
    end_date = quarter_end_dates.get(f'Q{q}')
    if end_date:
        return end_date + str(fiscal_year)  # Concatenate the end date with the full year
    else:
        raise ValueError("Invalid quarter format")

In [69]:
df_master = pd.read_excel(master_data_path)

In [70]:
df_master['Quarter'] = pd.to_datetime(df_master['Quarter'].apply(convert_qtr_to_date),dayfirst=True)

In [385]:
# df_pred1.columns

In [71]:
def label_prediction(row, model_name,top_n):
    in_top_50 = row[f'Top_{top_n}_Pred_Prob_{model_name}']
    target = row['Target']
    
    if in_top_50 and target == 1:
        return 'TP'
    elif in_top_50 and target == 0:
        return 'FP'
    elif not in_top_50 and target == 0:
        return 'TN'
    elif not in_top_50 and target == 1:
        return 'FN'

def flag_top50_nlargest(group, pred_prob_col,top_n):
    # If all pred_probs are zero, return all False
    if group[pred_prob_col].sum() == 0:
        return pd.Series([False] * len(group), index=group.index)
    # Use nlargest to get exactly 50
    top50_idx = group[pred_prob_col].nlargest(top_n).index
    flag = pd.Series(False, index=group.index)
    flag.loc[top50_idx] = True
    return flag

def label_prediction1(row, model_name):
    has_more_prob_than_threshold = row[f'Pred_Prob_Threshold_{model_name}']
    target = row['Target']
    
    if has_more_prob_than_threshold and target == 1:
        return 'TP'
    elif has_more_prob_than_threshold and target == 0:
        return 'FP'
    elif not has_more_prob_than_threshold and target == 0:
        return 'TN'
    elif not has_more_prob_than_threshold and target == 1:
        return 'FN'
    
def add_suffix_to_cols(df, suffix, exclude_cols):
    return df.rename(
        columns={col: f"{col}{suffix}" for col in df.columns if col not in exclude_cols + ['Quarter', 'Set']}
    )


In [72]:
def label_prediction(row, model_name,top_n):
    in_top_50 = row[f'Top_{top_n}_Pred_Prob_{model_name}']
    target = row['Target']
    
    if in_top_50 and target == 1:
        return 'TP'
    elif in_top_50 and target == 0:
        return 'FP'
    elif not in_top_50 and target == 0:
        return 'TN'
    elif not in_top_50 and target == 1:
        return 'FN'

def flag_top50_nlargest(group, pred_prob_col,top_n):
    # If all pred_probs are zero, return all False
    if group[pred_prob_col].sum() == 0:
        return pd.Series([False] * len(group), index=group.index)
    # Use nlargest to get exactly 50
    top50_idx = group[pred_prob_col].nlargest(top_n).index
    flag = pd.Series(False, index=group.index)
    flag.loc[top50_idx] = True
    return flag

def label_prediction1(row, model_name):
    has_more_prob_than_threshold = row[f'Pred_Prob_Threshold_{model_name}']
    target = row['Target']
    
    if has_more_prob_than_threshold and target == 1:
        return 'TP'
    elif has_more_prob_than_threshold and target == 0:
        return 'FP'
    elif not has_more_prob_than_threshold and target == 0:
        return 'TN'
    elif not has_more_prob_than_threshold and target == 1:
        return 'FN'
    
def add_suffix_to_cols(df, suffix, exclude_cols):
    return df.rename(
        columns={col: f"{col}{suffix}" for col in df.columns if col not in exclude_cols + ['Quarter', 'Set']}
    )


if len(set(['Rank','Returns'])-set(df_train1.columns))!=0:
    print("There are no rank and returns columns, merging now..!")
    df_train1 = df_train1.merge(df_master[['Ticker','Quarter','Returns','Rank']],on=['Ticker','Quarter'],how='left')
    df_test1 = df_test1.merge(df_master[['Ticker','Quarter','Returns','Rank']],on=['Ticker','Quarter'],how='left')

if np.isnan(pred_probs_train).all():
    print("All predictionsa re NaNs")

df_train_pred = df_train1[['Ticker','Quarter','Target','Returns','Rank']]
df_test_pred = df_test1[['Ticker','Quarter','Target','Returns','Rank']]

df_test_pred[f'Pred_Prob_{model_name}'] = pred_probs_test.reshape(-1)
df_train_pred[f'Pred_Prob_{model_name}'] = pred_probs_train.reshape(-1)

#### Top30 as fund - irrespective of the prob values
df_train_pred.loc[:,'Set'] = 'Train'
df_test_pred.loc[:,'Set'] = 'Test'
df_pred = pd.concat([df_train_pred,df_test_pred],axis=0)



df_pred.reset_index(drop=True,inplace=True)
df_pred.sort_values(by=['Quarter',f'Pred_Prob_{model_name}'],ascending=False,inplace=True)
df_pred[f'Top_{top_n}_Pred_Prob_{model_name}'] = df_pred.groupby('Quarter', group_keys=False).apply(flag_top50_nlargest, f'Pred_Prob_{model_name}',top_n)

df_pred[f'Top_50_Pred_Prob_{model_name}'] = df_pred.groupby('Quarter', group_keys=False).apply(flag_top50_nlargest, f'Pred_Prob_{model_name}',50)
# df_pred[(df_pred['Top_{top_n}_Pred_Prob_lstm_top50']==True)&(df_pred['Quarter']=='2005-12-31')]

# test_qtr_preds = df_pred[(df_pred['Quarter']=='2005-12-31')]['Pred_Prob_lstm_top50']
# test_qtr_preds[test_qtr_preds>0.9].shape
df_pred['Pred_Category_Top30'] = df_pred.apply(lambda row: label_prediction(row, model_name=model_name,top_n=30), axis=1)
df_pred['Pred_Category_Top50'] = df_pred.apply(lambda row: label_prediction(row, model_name=model_name,top_n=50), axis=1)
df_pred.reset_index(drop=True,inplace=True)

qtr_summary_df = df_pred.groupby('Quarter').agg(
                    Set = ('Set','first'),
                    Avg_Returns_Predicted_Top30=('Returns',lambda x: x[df_pred.loc[x.index, f'Top_30_Pred_Prob_{model_name}'] == True].mean()),
                    Avg_Returns_Predicted_Top50=('Returns',lambda x: x[df_pred.loc[x.index, f'Top_50_Pred_Prob_{model_name}'] == True].mean()),
                    Count_TP_Top30=('Pred_Category_Top30', lambda x: (x == 'TP').sum()),
                    Count_TN_Top30=('Pred_Category_Top30', lambda x: (x == 'TN').sum()),
                    Count_FP_Top30=('Pred_Category_Top30', lambda x: (x == 'FP').sum()),
                    Count_FN_Top30=('Pred_Category_Top30', lambda x: (x == 'FN').sum()),
                    Count_TP_Top50=('Pred_Category_Top50', lambda x: (x == 'TP').sum()),
                    Count_TN_Top50=('Pred_Category_Top50', lambda x: (x == 'TN').sum()),
                    Count_FP_Top50=('Pred_Category_Top50', lambda x: (x == 'FP').sum()),
                    Count_FN_Top50=('Pred_Category_Top50', lambda x: (x == 'FN').sum())
                    
                    ).reset_index()

# set_summary_df = qtr_summary_df.groupby('Set').agg(
#     Count_TP = ('Count_TP','sum'),
#     Count_TN = ('Count_TN','sum'),
#     Count_FP = ('Count_FP','sum'),
#     Count_FN = ('Count_FN','sum')
# )

df_pred1 = df_pred[['Ticker','Quarter','Target','Returns','Rank',f'Pred_Prob_{model_name}','Set']].copy()
df_pred1[f'Pred_Prob_Threshold_{model_name}'] = df_pred1[f'Pred_Prob_{model_name}'].between(0.5,1)



df_pred1['Pred_Category'] = df_pred1.apply(lambda row: label_prediction1(row, model_name=model_name), axis=1)
quarterly_returns_f2 = df_pred1.groupby('Quarter').agg(
                    Set = ('Set','first'),
                    Avg_Returns_Predicted_Thresh=('Returns', lambda x: x[df_pred1.loc[x.index, f'Pred_Prob_Threshold_{model_name}'] == True].mean()),
                    Count_TP_Thresh=('Pred_Category', lambda x: (x == 'TP').sum()),
                    Count_TN_Thresh=('Pred_Category', lambda x: (x == 'TN').sum()),
                    Count_FP_Thresh=('Pred_Category', lambda x: (x == 'FP').sum()),
                    Count_FN_Thresh=('Pred_Category', lambda x: (x == 'FN').sum())
                    ).reset_index()

qtr_summary_df2 = pd.DataFrame(quarterly_returns_f2)


#### Actual and Index returns mapping
##### Actual returns
act_returns = df_pred[df_pred['Target']==1].groupby(by='Quarter')['Returns'].mean()
df_act_returns = pd.DataFrame(act_returns)
df_act_returns.reset_index(inplace=True)
df_act_returns.rename(columns={'Returns':'Avg_Returns_Actual'},inplace=True)
qtr_summary_df = qtr_summary_df.merge(df_act_returns,on='Quarter',how='left')

##### Index returns - Qtrly
df_index = pd.read_excel(qtrly_returns_path,sheet_name='Qtrly_Stock_Returns')
df_index1 = df_index[df_index['Ticker'] == 'Nsemcap'].drop(columns=['Industry']).set_index(keys=['Ticker'])
df_index1 = df_index1.T
df_index1.reset_index(inplace=True)
df_index1.rename(columns={'index':'Quarter','Nsemcap':'Index_Returns'},inplace=True)

df_index1['Quarter'] = df_index1['Quarter'].apply(convert_qtr_to_date)
df_index1['Quarter'] = pd.to_datetime(df_index1['Quarter'],dayfirst=True)

qtr_summary_df = qtr_summary_df.merge(df_index1,how='left',on='Quarter')
qtr_summary_df['delta_returns_act_vs_pred_Top30'] = qtr_summary_df['Avg_Returns_Actual']-qtr_summary_df['Avg_Returns_Predicted_Top30']
qtr_summary_df['delta_returns_pred_vs_index_Top30'] = qtr_summary_df['Avg_Returns_Predicted_Top30']-qtr_summary_df['Index_Returns']
qtr_summary_df['delta_returns_act_vs_pred_Top50'] = qtr_summary_df['Avg_Returns_Actual']-qtr_summary_df['Avg_Returns_Predicted_Top50']
qtr_summary_df['delta_returns_pred_vs_index_Top50'] = qtr_summary_df['Avg_Returns_Predicted_Top50']-qtr_summary_df['Index_Returns']

qtr_summary_df2 = qtr_summary_df2.merge(df_act_returns,on='Quarter',how='left')
qtr_summary_df2 = qtr_summary_df2.merge(df_index1,on='Quarter',how='left')

qtr_summary_df2['delta_returns_act_vs_pred_Thresh'] = qtr_summary_df2['Avg_Returns_Actual']-qtr_summary_df2['Avg_Returns_Predicted_Thresh']
qtr_summary_df2['delta_returns_pred_vs_index_Thresh'] = qtr_summary_df2['Avg_Returns_Predicted_Thresh']-qtr_summary_df2['Index_Returns']


exclude_cols = ['Avg_Returns_Actual', 'Index_Returns']

# Reset index for merge
qtr_summary_df2_reset = qtr_summary_df2.reset_index(drop=True)
qtr_summary_df_reset = qtr_summary_df.reset_index(drop=True)

# Merge as before
merged_qtr_summary = pd.merge(
    qtr_summary_df_reset, qtr_summary_df2_reset,
    on=['Quarter', 'Set', 'Avg_Returns_Actual', 'Index_Returns'],
    how='inner',
    suffixes=('', '')
)

# Define the desired column order
ordered_cols = [
    'Quarter','Set',
    'Avg_Returns_Predicted_Top30','Avg_Returns_Predicted_Top50',
    'Avg_Returns_Predicted_Thresh',
    'Avg_Returns_Actual', 'Index_Returns',
    'delta_returns_act_vs_pred_Top30','delta_returns_pred_vs_index_Top30',
    'delta_returns_act_vs_pred_Top50','delta_returns_pred_vs_index_Top50',
    'delta_returns_act_vs_pred_Thresh','delta_returns_pred_vs_index_Thresh',
    'Count_TP_Top30', 'Count_TP_Top50', 'Count_TP_Thresh',
    'Count_TN_Top30', 'Count_TN_Top50', 'Count_TN_Thresh',
    'Count_FP_Top30', 'Count_FP_Top50', 'Count_FP_Thresh',
    'Count_FN_Top30', 'Count_FN_Top50', 'Count_FN_Thresh'
]

# Add any other columns that might exist but are not in the list
ordered_cols += [col for col in merged_qtr_summary.columns if col not in ordered_cols]

# Reorder columns
merged_qtr_summary = merged_qtr_summary[ordered_cols]
# merged_qtr_summary.reset_index(inplace=True)
merged_qtr_summary.head()

C:\Users\43737\AppData\Local\Temp\ipykernel_39152\2793545741.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test_pred[f'Pred_Prob_{model_name}'] = pred_probs_test.reshape(-1)
C:\Users\43737\AppData\Local\Temp\ipykernel_39152\2793545741.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train_pred[f'Pred_Prob_{model_name}'] = pred_probs_train.reshape(-1)
C:\Users\43737\AppData\Local\Temp\ipykernel_39152\2793545741.py:58: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice 

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Avg_Returns_Predicted_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,...,Count_TP_Thresh,Count_TN_Top30,Count_TN_Top50,Count_TN_Thresh,Count_FP_Top30,Count_FP_Top50,Count_FP_Thresh,Count_FN_Top30,Count_FN_Top50,Count_FN_Thresh
0,2005-03-31,Train,0.503553,0.445994,NaN,0.832432,-0.019376,0.328879,0.522929,0.386438,...,0,374,360,391,17,31,0,81,75,94
1,2005-06-30,Train,0.311541,0.507521,0.791960,0.952000,0.045476,0.640459,0.266066,0.444480,...,1,372,359,389,18,31,1,83,76,94
2,2005-09-30,Train,0.131784,0.101603,0.289899,0.331440,0.244277,0.199655,-0.112493,0.229837,...,2,367,351,385,20,36,2,88,84,96
3,2005-12-31,Train,0.194460,0.256720,0.902536,0.682739,0.058583,0.488279,0.135877,0.426019,...,1,366,355,388,22,33,0,89,80,96
4,2006-03-31,Train,0.287264,0.113601,-0.181964,0.254938,0.187746,-0.032326,0.099518,0.141337,...,0,373,357,389,18,34,2,87,83,99


In [73]:
merged_qtr_summary['Avg_Returns_Predicted_Thresh'].isna().all()

np.False_

In [74]:
merged_qtr_summary.columns

Index(['Quarter', 'Set', 'Avg_Returns_Predicted_Top30',
       'Avg_Returns_Predicted_Top50', 'Avg_Returns_Predicted_Thresh',
       'Avg_Returns_Actual', 'Index_Returns',
       'delta_returns_act_vs_pred_Top30', 'delta_returns_pred_vs_index_Top30',
       'delta_returns_act_vs_pred_Top50', 'delta_returns_pred_vs_index_Top50',
       'delta_returns_act_vs_pred_Thresh',
       'delta_returns_pred_vs_index_Thresh', 'Count_TP_Top30',
       'Count_TP_Top50', 'Count_TP_Thresh', 'Count_TN_Top30', 'Count_TN_Top50',
       'Count_TN_Thresh', 'Count_FP_Top30', 'Count_FP_Top50',
       'Count_FP_Thresh', 'Count_FN_Top30', 'Count_FN_Top50',
       'Count_FN_Thresh'],
      dtype='object')

In [75]:
df_pred[df_pred['Set']=='Test'][f'Pred_Prob_{model_name}'].unique()

array([1.        , 0.3117276 , 0.25157467, ..., 0.05320885, 0.04742453,
       0.04706237])

In [76]:
set_summary_df3 =  merged_qtr_summary.groupby('Set').agg(
    Count_TP_Thresh = ('Count_TP_Thresh','sum'),
    Count_TN_Thresh = ('Count_TN_Thresh','sum'),
    Count_FP_Thresh = ('Count_FP_Thresh','sum'),
    Count_FN_Thresh = ('Count_FN_Thresh','sum'),
    Count_TP_Top30 = ('Count_TP_Top30','sum'),
    Count_TN_Top30 = ('Count_TN_Top30','sum'),
    Count_FP_Top30 = ('Count_FP_Top30','sum'),
    Count_FN_Top30 = ('Count_FN_Top30','sum'),
    Count_TP_Top50 = ('Count_TP_Top50','sum'),
    Count_TN_Top50 = ('Count_TN_Top50','sum'),
    Count_FP_Top50 = ('Count_FP_Top50','sum'),
    Count_FN_Top50 = ('Count_FN_Top50','sum')
)
set_summary_df3

,Count_TP_Thresh,Count_TN_Thresh,Count_FP_Thresh,Count_FN_Thresh,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50
Set,,,,,,,,,,,,
Test,2,9608,16,1162,95,9359,265,1069,144,9168,456,1020
Train,16,36515,37,6417,548,35060,1492,5885,832,33984,2568,5601


In [77]:

def calc_metrics(tp, tn, fp, fn):
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return accuracy, precision, recall, f1


In [78]:
for suffix in ['_Top30','_Top50', '_Thresh']:
    print(f"\nMetrics for {suffix}:")
    for set_name, row in set_summary_df3.iterrows():
        tp = row[f'Count_TP{suffix}']
        tn = row[f'Count_TN{suffix}']
        fp = row[f'Count_FP{suffix}']
        fn = row[f'Count_FN{suffix}']
        accuracy, precision, recall, f1 = calc_metrics(tp, tn, fp, fn)
        print(f"{set_name}: accuracy={accuracy:.4f}, precision={precision:.4f}, recall={recall:.4f}, f1={f1:.4f}")


Metrics for _Top30:
Test: accuracy=0.8763, precision=0.2639, recall=0.0816, f1=0.1247
Train: accuracy=0.8284, precision=0.2686, recall=0.0852, f1=0.1294

Metrics for _Top50:
Test: accuracy=0.8632, precision=0.2400, recall=0.1237, f1=0.1633
Train: accuracy=0.8100, precision=0.2447, recall=0.1293, f1=0.1692

Metrics for _Thresh:
Test: accuracy=0.8908, precision=0.1111, recall=0.0017, f1=0.0034
Train: accuracy=0.8499, precision=0.3019, recall=0.0025, f1=0.0049


In [79]:
if merged_qtr_summary['Avg_Returns_Predicted_Thresh'].mean()==merged_qtr_summary['Avg_Returns_Predicted_Top30'].mean():
    print("Both the stock picking strategies (Top 30 and Pred prob theshold) have same returns")
if merged_qtr_summary['Avg_Returns_Predicted_Thresh'].mean()==merged_qtr_summary['Avg_Returns_Predicted_Top50'].mean():
    print("Both the stock picking strategies (Top 50 and Pred prob theshold) have same returns")
if merged_qtr_summary['Avg_Returns_Predicted_Top30'].mean()==merged_qtr_summary['Avg_Returns_Predicted_Top50'].mean():
    print("Both the stock picking strategies (Top 30 and Top 50) have same returns")
else:
    print("All of the stock picking strategies (Top 30, Top 50 and Pred prob theshold) have different returns")

All of the stock picking strategies (Top 30, Top 50 and Pred prob theshold) have different returns


In [80]:
qtr_summary_df[qtr_summary_df['Quarter']=='2021-09-30']

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,delta_returns_pred_vs_index_Top50
66,2021-09-30,Train,0.506205,0.521248,13,709,17,83,19,695,31,77,0.776731,0.126554,0.270526,0.379651,0.255483,0.394694


In [396]:
qtr_summary_df2[qtr_summary_df2['Quarter']=='2021-09-30']

,Quarter,Set,Avg_Returns_Predicted_Thresh,Count_TP_Thresh,Count_TN_Thresh,Count_FP_Thresh,Count_FN_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Thresh,delta_returns_pred_vs_index_Thresh
66,2021-09-30,Train,0.177851,1,725,1,95,0.776731,0.126554,0.59888,0.051297


In [397]:
df_pred.columns

Index(['Ticker', 'Quarter', 'Target', 'Returns', 'Rank', 'Pred_Prob_lr_top100',
       'Set', 'Top_30_Pred_Prob_lr_top100', 'Top_50_Pred_Prob_lr_top100',
       'Pred_Category_Top30', 'Pred_Category_Top50'],
      dtype='object')

In [398]:
df_pred1.columns

Index(['Ticker', 'Quarter', 'Target', 'Returns', 'Rank', 'Pred_Prob_lr_top100',
       'Set', 'Pred_Prob_Threshold_lr_top100', 'Pred_Category'],
      dtype='object')

In [399]:
merged_qtr_summary[merged_qtr_summary['Quarter']=='2021-09-30']

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Avg_Returns_Predicted_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,...,Count_TP_Thresh,Count_TN_Top30,Count_TN_Top50,Count_TN_Thresh,Count_FP_Top30,Count_FP_Top50,Count_FP_Thresh,Count_FN_Top30,Count_FN_Top50,Count_FN_Thresh
66,2021-09-30,Train,0.506205,0.521248,0.177851,0.776731,0.126554,0.270526,0.379651,0.255483,...,1,709,695,725,17,31,1,83,77,95


In [81]:
merged_qtr_summary[merged_qtr_summary['Avg_Returns_Predicted_Top30'].isna()].shape

(2, 25)

In [82]:
df_pred[df_pred['Set']=='Test'][f'Pred_Prob_{model_name}'].unique()

array([1.        , 0.3117276 , 0.25157467, ..., 0.05320885, 0.04742453,
       0.04706237])

In [83]:
df_pred[df_pred['Set']=='Test'][f'Pred_Prob_{model_name}'].describe()

count    10788.000000
mean         0.147451
std          0.040782
min          0.015902
25%          0.133862
50%          0.144086
75%          0.160695
max          1.000000
Name: Pred_Prob_lr_top100, dtype: float64

In [403]:
merged_qtr_summary['Quarter'].is_monotonic_increasing

True

In [404]:
set_summary_df3

,Count_TP_Thresh,Count_TN_Thresh,Count_FP_Thresh,Count_FN_Thresh,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50
Set,,,,,,,,,,,,
Test,2,9608,16,1162,95,9359,265,1069,144,9168,456,1020
Train,16,36515,37,6417,548,35060,1492,5885,832,33984,2568,5601


In [84]:
merged_qtr_summary['Avg_Returns_Predicted_Top30'] = merged_qtr_summary['Avg_Returns_Predicted_Top30'].ffill()
merged_qtr_summary['Avg_Returns_Predicted_Top50'] = merged_qtr_summary['Avg_Returns_Predicted_Top50'].ffill()
merged_qtr_summary['Avg_Returns_Predicted_Thresh'] = merged_qtr_summary['Avg_Returns_Predicted_Thresh'].ffill()

In [85]:
merged_qtr_summary['Avg_Returns_Predicted_Top30'].fillna(0,inplace=True)
merged_qtr_summary['Avg_Returns_Predicted_Top50'].fillna(0,inplace=True)
merged_qtr_summary['Avg_Returns_Predicted_Thresh'].fillna(0,inplace=True)

C:\Users\43737\AppData\Local\Temp\ipykernel_39152\4263771019.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_qtr_summary['Avg_Returns_Predicted_Top30'].fillna(0,inplace=True)
C:\Users\43737\AppData\Local\Temp\ipykernel_39152\4263771019.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always beh

In [86]:
merged_qtr_summary.head()

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Avg_Returns_Predicted_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,...,Count_TP_Thresh,Count_TN_Top30,Count_TN_Top50,Count_TN_Thresh,Count_FP_Top30,Count_FP_Top50,Count_FP_Thresh,Count_FN_Top30,Count_FN_Top50,Count_FN_Thresh
0,2005-03-31,Train,0.503553,0.445994,0.000000,0.832432,-0.019376,0.328879,0.522929,0.386438,...,0,374,360,391,17,31,0,81,75,94
1,2005-06-30,Train,0.311541,0.507521,0.791960,0.952000,0.045476,0.640459,0.266066,0.444480,...,1,372,359,389,18,31,1,83,76,94
2,2005-09-30,Train,0.131784,0.101603,0.289899,0.331440,0.244277,0.199655,-0.112493,0.229837,...,2,367,351,385,20,36,2,88,84,96
3,2005-12-31,Train,0.194460,0.256720,0.902536,0.682739,0.058583,0.488279,0.135877,0.426019,...,1,366,355,388,22,33,0,89,80,96
4,2006-03-31,Train,0.287264,0.113601,-0.181964,0.254938,0.187746,-0.032326,0.099518,0.141337,...,0,373,357,389,18,34,2,87,83,99


In [408]:
df_pred.head()

,Ticker,Quarter,Target,Returns,Rank,Pred_Prob_lr_top100,Set,Top_30_Pred_Prob_lr_top100,Top_50_Pred_Prob_lr_top100,Pred_Category_Top30,Pred_Category_Top50
0,uel,2024-12-31,0,NaN,NaN,1.000000,Test,True,True,FP,FP
1,diacabs,2024-12-31,0,-0.418070,777.0,0.311728,Test,True,True,FP,FP
2,wesl,2024-12-31,0,-0.279325,626.0,0.251575,Test,True,True,FP,FP
3,dolphino,2024-12-31,0,-0.496892,798.0,0.225990,Test,True,True,FP,FP
4,rtpow,2024-12-31,0,-0.283309,635.0,0.218004,Test,True,True,FP,FP


In [409]:
qtr_summary_df.head()

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,delta_returns_pred_vs_index_Top50
0,2005-03-31,Train,0.503553,0.445994,13,374,17,81,19,360,31,75,0.832432,-0.019376,0.328879,0.522929,0.386438,0.465370
1,2005-06-30,Train,0.311541,0.507521,12,372,18,83,19,359,31,76,0.952000,0.045476,0.640459,0.266066,0.444480,0.462045
2,2005-09-30,Train,0.131784,0.101603,10,367,20,88,14,351,36,84,0.331440,0.244277,0.199655,-0.112493,0.229837,-0.142674
3,2005-12-31,Train,0.194460,0.256720,8,366,22,89,17,355,33,80,0.682739,0.058583,0.488279,0.135877,0.426019,0.198137
4,2006-03-31,Train,0.287264,0.113601,12,373,18,87,16,357,34,83,0.254938,0.187746,-0.032326,0.099518,0.141337,-0.074145


In [410]:
qtr_summary_df['Set'].value_counts()

Set
Train    68
Test     12
Name: count, dtype: int64

In [87]:
with pd.ExcelWriter(r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\Fund1\results\lag\top100\3yrs_as_test_data_ts\Pred_Vs_Act_LR_time_split_25_07_2025.xlsx", ) as writer:
    merged_qtr_summary.to_excel(writer, sheet_name='Qtr_Summary', index=False)
    qtr_summary_df.to_excel(writer, sheet_name='Qtr_Summary_Top', index=False)
    qtr_summary_df2.to_excel(writer, sheet_name='Qtr_Summary_Thresh', index=False)
    df_pred.to_excel(writer, sheet_name='Pred_Vs_Act_Top', index=False)
    df_pred1.to_excel(writer, sheet_name='Pred_Vs_Act_Thresh', index=False)
    set_summary_df3.to_excel(writer, sheet_name='Set_Summary')

### Top100 Xform

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Flatten, Concatenate, Reshape, RepeatVector
from tensorflow.keras.models import Model
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
import tensorflow as tf
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,confusion_matrix,roc_auc_score
from tensorflow.keras.metrics import AUC, Precision
import os
from scipy.stats import skew

In [2]:

def convert_qtr_to_date(qtr):
    y = int(qtr.split("FY")[1])
    q = int(qtr.split("FY")[0][-1])
    quarter_end_dates = {
        "Q1": "30-06-",
        "Q2": "30-09-",
        "Q3": "31-12-",
        "Q4": "31-03-"
    }
    y = y if q==4 else y-1
    fiscal_year = 2000 + int(y)
    end_date = quarter_end_dates.get(f'Q{q}')
    if end_date:
        return end_date + str(fiscal_year)  # Concatenate the end date with the full year
    else:
        raise ValueError("Invalid quarter format")

In [4]:
train_data_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\Fund1\data\lag\train_data_after_multicollinearity_correlation_3yrs_test_size_time_based_split_2025_07_25.xlsx"
test_data_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\Fund1\data\lag\test_data_after_multicollinearity_correlation_3yrs_test_size_time_based_split_2025_07_25.xlsx"
master_data_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\data\Time_Series_Master_wo_Index_Last20Yrs1.xlsx"

df_train = pd.read_excel(train_data_path)
df_test = pd.read_excel(test_data_path)
df_master = pd.read_excel(master_data_path)
qtrly_returns_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\data\lag\Quarterly_Price_Returns_Labels.xlsx"

In [5]:
df_train.reset_index(drop=True,inplace=True)
df_test.reset_index(drop=True,inplace=True)

In [504]:
df_train.columns

Index(['Ticker', 'Quarter', 'TMR', 'Momentum', 'Beta', 'Alpha', 'EP', 'BP',
       'SP', 'Dividend_Flag', 'DE', 'EPS', 'Zscore', 'Target'],
      dtype='object')

##### Scaling

In [6]:
scaler = StandardScaler()

In [506]:
df_train.columns

Index(['Ticker', 'Quarter', 'TMR', 'Momentum', 'Beta', 'Alpha', 'EP', 'BP',
       'SP', 'Dividend_Flag', 'DE', 'EPS', 'Zscore', 'Target'],
      dtype='object')

In [7]:
vif_cols = list(df_train.columns)
for col in ['Ticker','Quarter','Set','Target']:
    if col in vif_cols:
        vif_cols.remove(col)

In [508]:
vif_cols

['TMR',
 'Momentum',
 'Beta',
 'Alpha',
 'EP',
 'BP',
 'SP',
 'Dividend_Flag',
 'DE',
 'EPS',
 'Zscore']

In [8]:
df_test.replace([np.inf, -np.inf], 0, inplace=True)

In [510]:
# df_test.replace([np.inf, -np.inf], 0, inplace=True)

In [9]:
train_data = scaler.fit_transform(df_train[vif_cols])
test_data = scaler.transform(df_test[vif_cols])

In [10]:
df_train1 = pd.DataFrame(train_data,columns=vif_cols)
df_test1 = pd.DataFrame(test_data,columns=vif_cols)

In [11]:
for col in ['Ticker','Target', 'Quarter']:
    df_train1[col] = df_train[col]
    df_test1[col] = df_test[col]

#### tgt definition

In [12]:
df_master['Quarter'] = pd.to_datetime(df_master['Quarter'].apply(convert_qtr_to_date),dayfirst=True)
df_train1 = df_train1.merge(df_master[['Ticker','Quarter','Returns','Rank']],on=['Ticker','Quarter'])
df_test1 = df_test1.merge(df_master[['Ticker','Quarter','Returns','Rank']],on=['Ticker','Quarter'])
df_train1['Target'] = np.where(df_train1['Rank']<=100,1,0)
df_test1['Target'] = np.where(df_test1['Rank']<=100,1,0)

In [13]:
# ...existing code...
from sklearn.preprocessing import PowerTransformer
from scipy.stats import boxcox
import numpy as np

best_transforms = {}
for col in vif_cols:
    x = df_train[col].dropna()
    results = {}
    if (x <= 0).any():
        # Only Yeo-Johnson possible
        pt = PowerTransformer(method='yeo-johnson')
        x_yj = pt.fit_transform(x.values.reshape(-1, 1)).flatten()
        results['yeo-johnson'] = abs(skew(x_yj))
    else:
        x_log = np.log1p(x)
        results['log1p'] = abs(skew(x_log))
        try:
            x_bc, _ = boxcox(x)
            results['boxcox'] = abs(skew(x_bc))
        except Exception:
            results['boxcox'] = np.nan
    # Always include original for comparison
    results['original'] = abs(skew(x))
    # Pick the transform with minimum absolute skewness
    best = min(results, key=lambda k: results[k] if not np.isnan(results[k]) else np.inf)
    best_transforms[col] = best
    print(f"{col}: Best transform = {best}, skewness = {results[best]:.3f}")
# ...existing code...
#### Power xforms
# ...existing code...
from sklearn.preprocessing import PowerTransformer
from scipy.stats import boxcox
import numpy as np

# 1. Decide best transform for each feature based on your previous results
# best_transform_map = {
#     'TMR': 'yeo-johnson',
#     'Momentum': 'yeo-johnson',
#     'Beta': 'yeo-johnson',
#     'Alpha': 'yeo-johnson',
#     'EP': 'boxcox',
#     'BP': 'yeo-johnson',
#     'SP': 'boxcox',
#     'EPS': 'yeo-johnson',
#     'Zscore': 'yeo-johnson',
#     'FCF': 'yeo-johnson',
#     'ICR': 'yeo-johnson',
#     'Year': 'log1p',
#     'Quarter_number': 'boxcox',  # or 'log1p', both are close
#     # 'Target': 'none',  # Target is not a feature
# }

def transform_feature(col, x, method):
    if method == 'yeo-johnson':
        pt = PowerTransformer(method='yeo-johnson')
        return pt.fit_transform(x.values.reshape(-1, 1)).flatten()
    elif method == 'boxcox':
        # Box-Cox requires strictly positive values
        # Add a small constant if min(x) == 0
        x_pos = x.copy()
        if (x_pos <= 0).any():
            x_pos = x_pos - x_pos.min() + 1e-6
        return boxcox(x_pos)[0]
    elif method == 'log1p':
        x_pos = x.copy()
        if (x_pos < 0).any():
            x_pos = x_pos - x_pos.min() + 1e-6
        return np.log1p(x_pos)
    else:
        return x.values

# 2. Apply transforms to train and test
df_train_trans = df_train.copy()
df_test_trans = df_test.copy()

for col in vif_cols:
    method = best_transforms.get(col, 'none')
    if method != 'none':
        df_train_trans[col] = transform_feature(col, df_train[col], method)
        df_test_trans[col] = transform_feature(col, df_test[col], method)

# 3. Scaling
scaler = StandardScaler()
train_data = scaler.fit_transform(df_train_trans[vif_cols])
test_data = scaler.transform(df_test_trans[vif_cols])

df_train1 = pd.DataFrame(train_data, columns=vif_cols)
df_test1 = pd.DataFrame(test_data, columns=vif_cols)

for col in ['Ticker', 'Target', 'Quarter']:
    df_train1[col] = df_train[col]
    df_test1[col] = df_test[col]

# 4. Prepare X_train, y_train, X_test, y_test
features = vif_cols.copy()
X_train = df_train1[features].values
y_train = df_train1['Target'].values
X_test = df_test1[features].values
y_test = df_test1['Target'].values

# If your model expects 3D input (samples, timesteps, features):
# X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
# X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
# ...existing code...

TMR: Best transform = yeo-johnson, skewness = 1.374
Momentum: Best transform = yeo-johnson, skewness = 0.373
Beta: Best transform = yeo-johnson, skewness = 4.617
Alpha: Best transform = yeo-johnson, skewness = 2.939
EP: Best transform = boxcox, skewness = 0.110
BP: Best transform = yeo-johnson, skewness = 0.206
SP: Best transform = boxcox, skewness = 0.044
Dividend_Flag: Best transform = original, skewness = 1.256
DE: Best transform = yeo-johnson, skewness = 0.049
EPS: Best transform = yeo-johnson, skewness = 1.697
Zscore: Best transform = yeo-johnson, skewness = 0.086
X_train shape: (42985, 11)
y_train shape: (42985,)
X_test shape: (10788, 11)
y_test shape: (10788,)


In [14]:
features = vif_cols.copy()
features

['TMR',
 'Momentum',
 'Beta',
 'Alpha',
 'EP',
 'BP',
 'SP',
 'Dividend_Flag',
 'DE',
 'EPS',
 'Zscore']

In [15]:
# features.remove("Quarter")
X_train = df_train1[features].values
y_train = df_train1['Target'].values

X_test = df_test1[features].values
y_test = df_test1['Target'].values

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd
import numpy as np

def calculate_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'roc':roc_auc_score(y_true,y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
        'confusion_matrix': confusion_matrix(y_true, y_pred)
    }

def train_logistic_regression_top100_xform(X_train, y_train, X_test, y_test, k=5):
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    fold_metrics = []

    for train_index, val_index in kf.split(X_train):
        X_train_fold, X_val = X_train[train_index], X_train[val_index]
        y_train_fold, y_val = y_train[train_index], y_train[val_index]

        model = LogisticRegression(max_iter=1000)
        model.fit(X_train_fold, y_train_fold)

        y_val_pred = model.predict(X_val)
        metrics = calculate_metrics(y_val, y_val_pred)
        fold_metrics.append(metrics)

    # Average scalar metrics
    avg_metrics = {
        key: np.mean([m[key] for m in fold_metrics]) for key in ['accuracy','roc', 'precision', 'recall', 'f1']
    }

    # Sum confusion matrices
    conf_matrix_sum = np.sum([m['confusion_matrix'] for m in fold_metrics], axis=0)


    # Final model on full training set
    final_model = LogisticRegression(max_iter=1000)
    final_model.fit(X_train, y_train)
    y_test_pred = final_model.predict(X_test)
    test_metrics = calculate_metrics(y_test, y_test_pred)

    # Combine into DataFrame
    metrics_df = pd.DataFrame({
        'set': ['cross_val_avg', 'test'],
        'accuracy': [avg_metrics['accuracy'], test_metrics['accuracy']],
        'roc':[avg_metrics['roc'],test_metrics['roc']],
        'precision': [avg_metrics['precision'], test_metrics['precision']],
        'recall': [avg_metrics['recall'], test_metrics['recall']],
        'f1': [avg_metrics['f1'], test_metrics['f1']],
        'confusion_matrix': [conf_matrix_sum, test_metrics['confusion_matrix']]
    })

    return metrics_df,model

# Reshape if needed
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)

metrics_df,lr_model_top100_xform = train_logistic_regression_top100_xform(X_train, y_train, X_test, y_test)

metrics_df

,set,accuracy,roc,precision,recall,f1,confusion_matrix
0,cross_val_avg,0.704525,0.545714,0.596134,0.131084,0.214829,"[[28546, 1179], [11522, 1738]]"
1,test,0.789303,0.557286,0.532824,0.150691,0.234938,"[[8166, 306], [1967, 349]]"


In [430]:
import pickle as pkl

In [ ]:
with open(r"D:\Python\Final Quant Fund\Quant Fund\Fund1\src\3yrs_as_test_data_ts\lr_top100_xform.pkl","wb") as file:
    pkl.dump(lr_model_top100_xform, file)

In [16]:
metrics_df

,set,accuracy,roc,precision,recall,f1,confusion_matrix
0,cross_val_avg,0.704525,0.545714,0.596134,0.131084,0.214829,"[[28546, 1179], [11522, 1738]]"
1,test,0.789303,0.557286,0.532824,0.150691,0.234938,"[[8166, 306], [1967, 349]]"


In [17]:
pred_probs_test = lr_model_top100_xform.predict_proba(X_test)[:,1]
pred_probs_train = lr_model_top100_xform.predict_proba(X_train)[:,1]

In [18]:
np.isnan(pred_probs_test).sum()

np.int64(0)

In [19]:
np.isnan(pred_probs_train).sum()

np.int64(0)

In [20]:

def calculate_metrics1(y_true, y_pred_label,y_pred_prob):
    accuracy = accuracy_score(y_true, y_pred_label)
    precision = precision_score(y_true, y_pred_label)
    recall = recall_score(y_true, y_pred_label)
    f1 = f1_score(y_true, y_pred_label)
    confusion = confusion_matrix(y_true,y_pred_label)
    roc_score = roc_auc_score(y_true,y_pred_prob)
    
    return {
        'accuracy': accuracy,
        'roc':roc_score,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix':confusion
    }



In [21]:
train_metrics1 = calculate_metrics1(y_train,(pred_probs_train>=0.5).astype(int),pred_probs_train)
test_metrics1 = calculate_metrics1(y_test,(pred_probs_test>=0.5).astype(int),pred_probs_test)

In [22]:
train_metrics_df1 = pd.DataFrame([train_metrics1])
train_metrics_df1['Set'] = "Train"
test_metrics_df1 = pd.DataFrame([test_metrics1])
test_metrics_df1['Set'] = "Test"
model_metrics1 = pd.concat([train_metrics_df1,test_metrics_df1],axis=0)
model_metrics1.reset_index(drop=True,inplace=True)
model_metrics1.head()

,accuracy,roc,precision,recall,f1,confusion_matrix,Set
0,0.704571,0.660936,0.595311,0.132127,0.216256,"[[28534, 1191], [11508, 1752]]",Train
1,0.789396,0.694597,0.533133,0.152850,0.237584,"[[8162, 310], [1962, 354]]",Test


In [23]:
model_name = 'lr_top100_xform'
top_n = 30

In [24]:

def convert_qtr_to_date(qtr):
    y = int(qtr.split("FY")[1])
    q = int(qtr.split("FY")[0][-1])
    quarter_end_dates = {
        "Q1": "30-06-",
        "Q2": "30-09-",
        "Q3": "31-12-",
        "Q4": "31-03-"
    }
    y = y if q==4 else y-1
    fiscal_year = 2000 + int(y)
    end_date = quarter_end_dates.get(f'Q{q}')
    if end_date:
        return end_date + str(fiscal_year)  # Concatenate the end date with the full year
    else:
        raise ValueError("Invalid quarter format")

In [25]:
df_master = pd.read_excel(master_data_path)

In [26]:
df_master['Quarter'] = pd.to_datetime(df_master['Quarter'].apply(convert_qtr_to_date),dayfirst=True)

In [27]:
def label_prediction(row, model_name,top_n):
    in_top_50 = row[f'Top_{top_n}_Pred_Prob_{model_name}']
    target = row['Target']
    
    if in_top_50 and target == 1:
        return 'TP'
    elif in_top_50 and target == 0:
        return 'FP'
    elif not in_top_50 and target == 0:
        return 'TN'
    elif not in_top_50 and target == 1:
        return 'FN'

def flag_top50_nlargest(group, pred_prob_col,top_n):
    # If all pred_probs are zero, return all False
    if group[pred_prob_col].sum() == 0:
        return pd.Series([False] * len(group), index=group.index)
    # Use nlargest to get exactly 50
    top50_idx = group[pred_prob_col].nlargest(top_n).index
    flag = pd.Series(False, index=group.index)
    flag.loc[top50_idx] = True
    return flag

def label_prediction1(row, model_name):
    has_more_prob_than_threshold = row[f'Pred_Prob_Threshold_{model_name}']
    target = row['Target']
    
    if has_more_prob_than_threshold and target == 1:
        return 'TP'
    elif has_more_prob_than_threshold and target == 0:
        return 'FP'
    elif not has_more_prob_than_threshold and target == 0:
        return 'TN'
    elif not has_more_prob_than_threshold and target == 1:
        return 'FN'
    
def add_suffix_to_cols(df, suffix, exclude_cols):
    return df.rename(
        columns={col: f"{col}{suffix}" for col in df.columns if col not in exclude_cols + ['Quarter', 'Set']}
    )


In [28]:

if len(set(['Rank','Returns'])-set(df_train1.columns))!=0:
    print("There are no rank and returns columns, merging now..!")
    df_train1 = df_train1.merge(df_master[['Ticker','Quarter','Returns','Rank']],on=['Ticker','Quarter'],how='left')
    df_test1 = df_test1.merge(df_master[['Ticker','Quarter','Returns','Rank']],on=['Ticker','Quarter'],how='left')

if np.isnan(pred_probs_train).all():
    print("All predictionsa re NaNs")

df_train_pred = df_train1[['Ticker','Quarter','Target','Returns','Rank']]
df_test_pred = df_test1[['Ticker','Quarter','Target','Returns','Rank']]

df_test_pred[f'Pred_Prob_{model_name}'] = pred_probs_test.reshape(-1)
df_train_pred[f'Pred_Prob_{model_name}'] = pred_probs_train.reshape(-1)

#### Top30 as fund - irrespective of the prob values
df_train_pred.loc[:,'Set'] = 'Train'
df_test_pred.loc[:,'Set'] = 'Test'
df_pred = pd.concat([df_train_pred,df_test_pred],axis=0)



df_pred.reset_index(drop=True,inplace=True)
df_pred.sort_values(by=['Quarter',f'Pred_Prob_{model_name}'],ascending=False,inplace=True)
df_pred[f'Top_{top_n}_Pred_Prob_{model_name}'] = df_pred.groupby('Quarter', group_keys=False).apply(flag_top50_nlargest, f'Pred_Prob_{model_name}',top_n)

df_pred[f'Top_50_Pred_Prob_{model_name}'] = df_pred.groupby('Quarter', group_keys=False).apply(flag_top50_nlargest, f'Pred_Prob_{model_name}',50)
# df_pred[(df_pred['Top_{top_n}_Pred_Prob_lstm_top50']==True)&(df_pred['Quarter']=='2005-12-31')]

# test_qtr_preds = df_pred[(df_pred['Quarter']=='2005-12-31')]['Pred_Prob_lstm_top50']
# test_qtr_preds[test_qtr_preds>0.9].shape
df_pred['Pred_Category_Top30'] = df_pred.apply(lambda row: label_prediction(row, model_name=model_name,top_n=30), axis=1)
df_pred['Pred_Category_Top50'] = df_pred.apply(lambda row: label_prediction(row, model_name=model_name,top_n=50), axis=1)
df_pred.reset_index(drop=True,inplace=True)

qtr_summary_df = df_pred.groupby('Quarter').agg(
                    Set = ('Set','first'),
                    Avg_Returns_Predicted_Top30=('Returns',lambda x: x[df_pred.loc[x.index, f'Top_30_Pred_Prob_{model_name}'] == True].mean()),
                    Avg_Returns_Predicted_Top50=('Returns',lambda x: x[df_pred.loc[x.index, f'Top_50_Pred_Prob_{model_name}'] == True].mean()),
                    Count_TP_Top30=('Pred_Category_Top30', lambda x: (x == 'TP').sum()),
                    Count_TN_Top30=('Pred_Category_Top30', lambda x: (x == 'TN').sum()),
                    Count_FP_Top30=('Pred_Category_Top30', lambda x: (x == 'FP').sum()),
                    Count_FN_Top30=('Pred_Category_Top30', lambda x: (x == 'FN').sum()),
                    Count_TP_Top50=('Pred_Category_Top50', lambda x: (x == 'TP').sum()),
                    Count_TN_Top50=('Pred_Category_Top50', lambda x: (x == 'TN').sum()),
                    Count_FP_Top50=('Pred_Category_Top50', lambda x: (x == 'FP').sum()),
                    Count_FN_Top50=('Pred_Category_Top50', lambda x: (x == 'FN').sum())
                    
                    ).reset_index()

# set_summary_df = qtr_summary_df.groupby('Set').agg(
#     Count_TP = ('Count_TP','sum'),
#     Count_TN = ('Count_TN','sum'),
#     Count_FP = ('Count_FP','sum'),
#     Count_FN = ('Count_FN','sum')
# )

df_pred1 = df_pred[['Ticker','Quarter','Target','Returns','Rank',f'Pred_Prob_{model_name}','Set']].copy()
df_pred1[f'Pred_Prob_Threshold_{model_name}'] = df_pred1[f'Pred_Prob_{model_name}'].between(0.5,1)



df_pred1['Pred_Category'] = df_pred1.apply(lambda row: label_prediction1(row, model_name=model_name), axis=1)
quarterly_returns_f2 = df_pred1.groupby('Quarter').agg(
                    Set = ('Set','first'),
                    Avg_Returns_Predicted_Thresh=('Returns', lambda x: x[df_pred1.loc[x.index, f'Pred_Prob_Threshold_{model_name}'] == True].mean()),
                    Count_TP_Thresh=('Pred_Category', lambda x: (x == 'TP').sum()),
                    Count_TN_Thresh=('Pred_Category', lambda x: (x == 'TN').sum()),
                    Count_FP_Thresh=('Pred_Category', lambda x: (x == 'FP').sum()),
                    Count_FN_Thresh=('Pred_Category', lambda x: (x == 'FN').sum())
                    ).reset_index()
# if quarterly_returns_f2['Avg_Returns_Predicted'].isna().any():
#     print("there are quarters where there were no 1s by the model")
# else:
#     print("there are no quarters where there were no 1s by the model")
qtr_summary_df2 = pd.DataFrame(quarterly_returns_f2)


#### Actual and Index returns mapping
##### Actual returns
act_returns = df_pred[df_pred['Target']==1].groupby(by='Quarter')['Returns'].mean()
df_act_returns = pd.DataFrame(act_returns)
df_act_returns.reset_index(inplace=True)
df_act_returns.rename(columns={'Returns':'Avg_Returns_Actual'},inplace=True)
qtr_summary_df = qtr_summary_df.merge(df_act_returns,on='Quarter',how='left')

##### Index returns - Qtrly
df_index = pd.read_excel(qtrly_returns_path,sheet_name='Qtrly_Stock_Returns')
df_index1 = df_index[df_index['Ticker'] == 'Nsemcap'].drop(columns=['Industry']).set_index(keys=['Ticker'])
df_index1 = df_index1.T
df_index1.reset_index(inplace=True)
df_index1.rename(columns={'index':'Quarter','Nsemcap':'Index_Returns'},inplace=True)

df_index1['Quarter'] = df_index1['Quarter'].apply(convert_qtr_to_date)
df_index1['Quarter'] = pd.to_datetime(df_index1['Quarter'],dayfirst=True)

qtr_summary_df = qtr_summary_df.merge(df_index1,how='left',on='Quarter')
qtr_summary_df['delta_returns_act_vs_pred_Top30'] = qtr_summary_df['Avg_Returns_Actual']-qtr_summary_df['Avg_Returns_Predicted_Top30']
qtr_summary_df['delta_returns_pred_vs_index_Top30'] = qtr_summary_df['Avg_Returns_Predicted_Top30']-qtr_summary_df['Index_Returns']
qtr_summary_df['delta_returns_act_vs_pred_Top50'] = qtr_summary_df['Avg_Returns_Actual']-qtr_summary_df['Avg_Returns_Predicted_Top50']
qtr_summary_df['delta_returns_pred_vs_index_Top50'] = qtr_summary_df['Avg_Returns_Predicted_Top50']-qtr_summary_df['Index_Returns']

qtr_summary_df2 = qtr_summary_df2.merge(df_act_returns,on='Quarter',how='left')
qtr_summary_df2 = qtr_summary_df2.merge(df_index1,on='Quarter',how='left')

qtr_summary_df2['delta_returns_act_vs_pred_Thresh'] = qtr_summary_df2['Avg_Returns_Actual']-qtr_summary_df2['Avg_Returns_Predicted_Thresh']
qtr_summary_df2['delta_returns_pred_vs_index_Thresh'] = qtr_summary_df2['Avg_Returns_Predicted_Thresh']-qtr_summary_df2['Index_Returns']

# set_summary_df2 = qtr_summary_df2.groupby('Set').agg(
#     Count_TP = ('Count_TP','sum'),
#     Count_TN = ('Count_TN','sum'),
#     Count_FP = ('Count_FP','sum'),
#     Count_FN = ('Count_FN','sum')
# )

exclude_cols = ['Avg_Returns_Actual', 'Index_Returns']
# suffix_map = {
#     'qtr_summary_df2': '_thresh',
#     'qtr_summary_df': '_top'
# }


# Reset index for merge
qtr_summary_df2_reset = qtr_summary_df2.reset_index(drop=True)
qtr_summary_df_reset = qtr_summary_df.reset_index(drop=True)

# Add suffixes
# qtr_summary_df2_suff = add_suffix_to_cols(qtr_summary_df2_reset, suffix_map['qtr_summary_df2'], exclude_cols)
# qtr_summary_df_suff = add_suffix_to_cols(qtr_summary_df_reset, suffix_map['qtr_summary_df'], exclude_cols)
# ...existing code...

# Merge as before
merged_qtr_summary = pd.merge(
    qtr_summary_df_reset, qtr_summary_df2_reset,
    on=['Quarter', 'Set', 'Avg_Returns_Actual', 'Index_Returns'],
    how='inner',
    suffixes=('', '')
)

# Define the desired column order
ordered_cols = [
    'Quarter','Set',
    'Avg_Returns_Predicted_Top30','Avg_Returns_Predicted_Top50',
    'Avg_Returns_Predicted_Thresh',
    'Avg_Returns_Actual', 'Index_Returns',
    'delta_returns_act_vs_pred_Top30','delta_returns_pred_vs_index_Top30',
    'delta_returns_act_vs_pred_Top50','delta_returns_pred_vs_index_Top50',
    'delta_returns_act_vs_pred_Thresh','delta_returns_pred_vs_index_Thresh',
    'Count_TP_Top30', 'Count_TP_Top50', 'Count_TP_Thresh',
    'Count_TN_Top30', 'Count_TN_Top50', 'Count_TN_Thresh',
    'Count_FP_Top30', 'Count_FP_Top50', 'Count_FP_Thresh',
    'Count_FN_Top30', 'Count_FN_Top50', 'Count_FN_Thresh'
]

# Add any other columns that might exist but are not in the list
ordered_cols += [col for col in merged_qtr_summary.columns if col not in ordered_cols]

# Reorder columns
merged_qtr_summary = merged_qtr_summary[ordered_cols]
# merged_qtr_summary.reset_index(inplace=True)
merged_qtr_summary.head()

There are no rank and returns columns, merging now..!


C:\Users\43737\AppData\Local\Temp\ipykernel_26176\4113010357.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test_pred[f'Pred_Prob_{model_name}'] = pred_probs_test.reshape(-1)
C:\Users\43737\AppData\Local\Temp\ipykernel_26176\4113010357.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train_pred[f'Pred_Prob_{model_name}'] = pred_probs_train.reshape(-1)
C:\Users\43737\AppData\Local\Temp\ipykernel_26176\4113010357.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice 

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Avg_Returns_Predicted_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,...,Count_TP_Thresh,Count_TN_Top30,Count_TN_Top50,Count_TN_Thresh,Count_FP_Top30,Count_FP_Top50,Count_FP_Thresh,Count_FN_Top30,Count_FN_Top50,Count_FN_Thresh
0,2005-03-31,Train,0.202669,0.324026,0.343297,0.462237,-0.019376,0.259568,0.222045,0.138212,...,65,285,281,267,6,10,24,170,154,129
1,2005-06-30,Train,0.304000,0.330705,0.314607,0.495418,0.045476,0.191418,0.258525,0.164713,...,65,283,277,261,9,15,31,172,158,128
2,2005-09-30,Train,0.129293,0.092910,0.077915,0.180268,0.244277,0.050974,-0.114984,0.087358,...,32,278,269,265,11,20,24,177,166,164
3,2005-12-31,Train,0.424620,0.346235,0.445298,0.407782,0.058583,-0.016837,0.366036,0.061547,...,17,286,276,288,4,14,2,169,159,178
4,2006-03-31,Train,-0.001229,-0.057411,-0.067324,0.018142,0.187746,0.019371,-0.188975,0.075552,...,42,292,286,285,3,9,10,168,154,153


In [29]:
merged_qtr_summary['Avg_Returns_Predicted_Thresh'].isna().all()

np.False_

In [533]:
merged_qtr_summary.columns

Index(['Quarter', 'Set', 'Avg_Returns_Predicted_Top30',
       'Avg_Returns_Predicted_Top50', 'Avg_Returns_Predicted_Thresh',
       'Avg_Returns_Actual', 'Index_Returns',
       'delta_returns_act_vs_pred_Top30', 'delta_returns_pred_vs_index_Top30',
       'delta_returns_act_vs_pred_Top50', 'delta_returns_pred_vs_index_Top50',
       'delta_returns_act_vs_pred_Thresh',
       'delta_returns_pred_vs_index_Thresh', 'Count_TP_Top30',
       'Count_TP_Top50', 'Count_TP_Thresh', 'Count_TN_Top30', 'Count_TN_Top50',
       'Count_TN_Thresh', 'Count_FP_Top30', 'Count_FP_Top50',
       'Count_FP_Thresh', 'Count_FN_Top30', 'Count_FN_Top50',
       'Count_FN_Thresh'],
      dtype='object')

In [30]:
df_pred[df_pred['Set']=='Test'][f'Pred_Prob_{model_name}'].unique()

array([0.7347319 , 0.63331719, 0.62042498, ..., 0.01804842, 0.01567157,
       0.01388437])

In [31]:
set_summary_df3 =  merged_qtr_summary.groupby('Set').agg(
    Count_TP_Thresh = ('Count_TP_Thresh','sum'),
    Count_TN_Thresh = ('Count_TN_Thresh','sum'),
    Count_FP_Thresh = ('Count_FP_Thresh','sum'),
    Count_FN_Thresh = ('Count_FN_Thresh','sum'),
    Count_TP_Top30 = ('Count_TP_Top30','sum'),
    Count_TN_Top30 = ('Count_TN_Top30','sum'),
    Count_FP_Top30 = ('Count_FP_Top30','sum'),
    Count_FN_Top30 = ('Count_FN_Top30','sum'),
    Count_TP_Top50 = ('Count_TP_Top50','sum'),
    Count_TN_Top50 = ('Count_TN_Top50','sum'),
    Count_FP_Top50 = ('Count_FP_Top50','sum'),
    Count_FN_Top50 = ('Count_FN_Top50','sum')
)
set_summary_df3

,Count_TP_Thresh,Count_TN_Thresh,Count_FP_Thresh,Count_FN_Thresh,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50
Set,,,,,,,,,,,,
Test,354,8162,310,1962,197,8309,163,2119,324,8196,276,1992
Train,1752,28534,1191,11508,1225,28910,815,12035,1986,28311,1414,11274


In [32]:

def calc_metrics(tp, tn, fp, fn):
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return accuracy, precision, recall, f1


In [33]:
for suffix in ['_Top30','_Top50', '_Thresh']:
    print(f"\nMetrics for {suffix}:")
    for set_name, row in set_summary_df3.iterrows():
        tp = row[f'Count_TP{suffix}']
        tn = row[f'Count_TN{suffix}']
        fp = row[f'Count_FP{suffix}']
        fn = row[f'Count_FN{suffix}']
        accuracy, precision, recall, f1 = calc_metrics(tp, tn, fp, fn)
        print(f"{set_name}: accuracy={accuracy:.4f}, precision={precision:.4f}, recall={recall:.4f}, f1={f1:.4f}")


Metrics for _Top30:
Test: accuracy=0.7885, precision=0.5472, recall=0.0851, f1=0.1472
Train: accuracy=0.7011, precision=0.6005, recall=0.0924, f1=0.1601

Metrics for _Top50:
Test: accuracy=0.7898, precision=0.5400, recall=0.1399, f1=0.2222
Train: accuracy=0.7048, precision=0.5841, recall=0.1498, f1=0.2384

Metrics for _Thresh:
Test: accuracy=0.7894, precision=0.5331, recall=0.1528, f1=0.2376
Train: accuracy=0.7046, precision=0.5953, recall=0.1321, f1=0.2163


In [34]:
if merged_qtr_summary['Avg_Returns_Predicted_Thresh'].mean()==merged_qtr_summary['Avg_Returns_Predicted_Top30'].mean():
    print("Both the stock picking strategies (Top 30 and Pred prob theshold) have same returns")
if merged_qtr_summary['Avg_Returns_Predicted_Thresh'].mean()==merged_qtr_summary['Avg_Returns_Predicted_Top50'].mean():
    print("Both the stock picking strategies (Top 50 and Pred prob theshold) have same returns")
if merged_qtr_summary['Avg_Returns_Predicted_Top30'].mean()==merged_qtr_summary['Avg_Returns_Predicted_Top50'].mean():
    print("Both the stock picking strategies (Top 30 and Top 50) have same returns")
else:
    print("All of the stock picking strategies (Top 30, Top 50 and Pred prob theshold) have different returns")

All of the stock picking strategies (Top 30, Top 50 and Pred prob theshold) have different returns


In [539]:
qtr_summary_df[qtr_summary_df['Quarter']=='2021-09-30']

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,delta_returns_pred_vs_index_Top50
66,2021-09-30,Train,0.408642,0.400875,6,769,24,23,9,752,41,20,1.519144,0.126554,1.110502,0.282088,1.118268,0.274322


In [540]:
qtr_summary_df2[qtr_summary_df2['Quarter']=='2021-09-30']

,Quarter,Set,Avg_Returns_Predicted_Thresh,Count_TP_Thresh,Count_TN_Thresh,Count_FP_Thresh,Count_FN_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Thresh,delta_returns_pred_vs_index_Thresh
66,2021-09-30,Train,-0.340932,0,792,1,29,1.519144,0.126554,1.860076,-0.467486


In [541]:
df_pred.columns

Index(['Ticker', 'Quarter', 'Target', 'Returns', 'Rank',
       'Pred_Prob_lr_top100_xform', 'Set', 'Top_30_Pred_Prob_lr_top100_xform',
       'Top_50_Pred_Prob_lr_top100_xform', 'Pred_Category_Top30',
       'Pred_Category_Top50'],
      dtype='object')

In [542]:
df_pred1.columns

Index(['Ticker', 'Quarter', 'Target', 'Returns', 'Rank',
       'Pred_Prob_lr_top100_xform', 'Set',
       'Pred_Prob_Threshold_lr_top100_xform', 'Pred_Category'],
      dtype='object')

In [543]:
merged_qtr_summary[merged_qtr_summary['Quarter']=='2021-09-30']

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Avg_Returns_Predicted_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,...,Count_TP_Thresh,Count_TN_Top30,Count_TN_Top50,Count_TN_Thresh,Count_FP_Top30,Count_FP_Top50,Count_FP_Thresh,Count_FN_Top30,Count_FN_Top50,Count_FN_Thresh
66,2021-09-30,Train,0.408642,0.400875,-0.340932,1.519144,0.126554,1.110502,0.282088,1.118268,...,0,769,752,792,24,41,1,23,20,29


In [544]:
merged_qtr_summary[merged_qtr_summary['Avg_Returns_Predicted_Top30'].isna()].shape

(2, 25)

In [545]:
df_pred[df_pred['Set']=='Test'][f'Pred_Prob_{model_name}'].unique()

array([0.72703602, 0.13351212, 0.13345738, ..., 0.01079372, 0.01077693,
       0.00891556])

In [546]:
df_pred[df_pred['Set']=='Test'][f'Pred_Prob_{model_name}'].describe()

count    10788.000000
mean         0.038835
std          0.025169
min          0.001414
25%          0.023678
50%          0.033558
75%          0.047641
max          0.751089
Name: Pred_Prob_lr_top100_xform, dtype: float64

In [547]:
merged_qtr_summary['Quarter'].is_monotonic_increasing

True

In [548]:
set_summary_df3

,Count_TP_Thresh,Count_TN_Thresh,Count_FP_Thresh,Count_FN_Thresh,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50
Set,,,,,,,,,,,,
Test,0,10438,4,346,42,10124,318,304,62,9904,538,284
Train,0,41050,6,1929,255,39271,1785,1674,373,38029,3027,1556


In [35]:
merged_qtr_summary['Avg_Returns_Predicted_Top30'] = merged_qtr_summary['Avg_Returns_Predicted_Top30'].ffill()
merged_qtr_summary['Avg_Returns_Predicted_Top50'] = merged_qtr_summary['Avg_Returns_Predicted_Top50'].ffill()
merged_qtr_summary['Avg_Returns_Predicted_Thresh'] = merged_qtr_summary['Avg_Returns_Predicted_Thresh'].ffill()

In [36]:
merged_qtr_summary['Avg_Returns_Predicted_Top30'].fillna(0,inplace=True)
merged_qtr_summary['Avg_Returns_Predicted_Top50'].fillna(0,inplace=True)
merged_qtr_summary['Avg_Returns_Predicted_Thresh'].fillna(0,inplace=True)

C:\Users\43737\AppData\Local\Temp\ipykernel_26176\4263771019.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_qtr_summary['Avg_Returns_Predicted_Top30'].fillna(0,inplace=True)
C:\Users\43737\AppData\Local\Temp\ipykernel_26176\4263771019.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always beh

In [551]:
merged_qtr_summary.head()

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Avg_Returns_Predicted_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,...,Count_TP_Thresh,Count_TN_Top30,Count_TN_Top50,Count_TN_Thresh,Count_FP_Top30,Count_FP_Top50,Count_FP_Thresh,Count_FN_Top30,Count_FN_Top50,Count_FN_Thresh
0,2005-03-31,Train,0.487041,0.436550,0.0,1.455923,-0.019376,0.968882,0.506417,1.019373,...,0,431,415,457,26,42,0,24,20,28
1,2005-06-30,Train,0.517589,0.444255,0.0,1.790164,0.045476,1.272575,0.472114,1.345909,...,0,434,416,458,24,42,0,21,19,27
2,2005-09-30,Train,0.088815,0.052912,0.0,0.496120,0.244277,0.407304,-0.155461,0.443207,...,0,431,412,456,25,44,0,24,23,29
3,2005-12-31,Train,0.299423,0.301841,0.0,1.147137,0.058583,0.847714,0.240840,0.845296,...,0,431,414,457,26,43,0,24,21,28
4,2006-03-31,Train,-0.033331,0.193029,0.0,0.772814,0.187746,0.806145,-0.221077,0.579785,...,0,436,422,461,25,39,0,24,18,29


In [552]:
df_pred.head()

,Ticker,Quarter,Target,Returns,Rank,Pred_Prob_lr_top100_xform,Set,Top_30_Pred_Prob_lr_top100_xform,Top_50_Pred_Prob_lr_top100_xform,Pred_Category_Top30,Pred_Category_Top50
0,uel,2024-12-31,0,NaN,NaN,0.727036,Test,True,True,FP,FP
1,ut,2024-12-31,0,-0.358306,745.0,0.133512,Test,True,True,FP,FP
2,rtpow,2024-12-31,0,-0.283309,635.0,0.133457,Test,True,True,FP,FP
3,sdb,2024-12-31,0,0.122062,36.0,0.119482,Test,True,True,FP,FP
4,dcxindia,2024-12-31,0,-0.341313,730.0,0.115898,Test,True,True,FP,FP


In [553]:
qtr_summary_df.head()

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,delta_returns_pred_vs_index_Top50
0,2005-03-31,Train,0.487041,0.436550,4,431,26,24,8,415,42,20,1.455923,-0.019376,0.968882,0.506417,1.019373,0.455926
1,2005-06-30,Train,0.517589,0.444255,6,434,24,21,8,416,42,19,1.790164,0.045476,1.272575,0.472114,1.345909,0.398779
2,2005-09-30,Train,0.088815,0.052912,5,431,25,24,6,412,44,23,0.496120,0.244277,0.407304,-0.155461,0.443207,-0.191364
3,2005-12-31,Train,0.299423,0.301841,4,431,26,24,7,414,43,21,1.147137,0.058583,0.847714,0.240840,0.845296,0.243258
4,2006-03-31,Train,-0.033331,0.193029,5,436,25,24,11,422,39,18,0.772814,0.187746,0.806145,-0.221077,0.579785,0.005283


In [37]:
qtr_summary_df['Set'].value_counts()

Set
Train    68
Test     12
Name: count, dtype: int64

In [39]:
with pd.ExcelWriter(r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\Fund1\results\lag\top100\3yrs_as_test_data_ts\Pred_Vs_Act_LR_xform_time_split_25_07_2025.xlsx", ) as writer:
    merged_qtr_summary.to_excel(writer, sheet_name='Qtr_Summary', index=False)
    qtr_summary_df.to_excel(writer, sheet_name='Qtr_Summary_Top', index=False)
    qtr_summary_df2.to_excel(writer, sheet_name='Qtr_Summary_Thresh', index=False)
    df_pred.to_excel(writer, sheet_name='Pred_Vs_Act_Top', index=False)
    df_pred1.to_excel(writer, sheet_name='Pred_Vs_Act_Thresh', index=False)
    set_summary_df3.to_excel(writer, sheet_name='Set_Summary')

### Top200 as tgt

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Flatten, Concatenate, Reshape, RepeatVector
from tensorflow.keras.models import Model
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
import tensorflow as tf
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,confusion_matrix,roc_auc_score
from tensorflow.keras.metrics import AUC, Precision
import os

In [2]:
train_data_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\Fund1\data\lag\train_data_after_multicollinearity_correlation_3yrs_test_size_time_based_split_2025_07_25.xlsx"
test_data_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\Fund1\data\lag\test_data_after_multicollinearity_correlation_3yrs_test_size_time_based_split_2025_07_25.xlsx"
master_data_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\data\Time_Series_Master_wo_Index_Last20Yrs1.xlsx"

df_train = pd.read_excel(train_data_path)
df_test = pd.read_excel(test_data_path)
df_master = pd.read_excel(master_data_path)
qtrly_returns_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\data\lag\Quarterly_Price_Returns_Labels.xlsx"

In [3]:
df_train.reset_index(drop=True,inplace=True)
df_test.reset_index(drop=True,inplace=True)

In [4]:
df_train.columns

Index(['Ticker', 'Quarter', 'TMR', 'Momentum', 'Beta', 'Alpha', 'EP', 'BP',
       'SP', 'Dividend_Flag', 'DE', 'EPS', 'Zscore', 'Target'],
      dtype='object')

##### Scaling

In [5]:
scaler = StandardScaler()

In [561]:
df_train.columns

Index(['Ticker', 'Quarter', 'TMR', 'Momentum', 'Beta', 'Alpha', 'EP', 'BP',
       'SP', 'Dividend_Flag', 'DE', 'EPS', 'Zscore', 'Target'],
      dtype='object')

In [6]:
vif_cols = list(df_train.columns)
for col in ['Ticker','Quarter','Set','Target']:
    if col in vif_cols:
        vif_cols.remove(col)

In [563]:
vif_cols

['TMR',
 'Momentum',
 'Beta',
 'Alpha',
 'EP',
 'BP',
 'SP',
 'Dividend_Flag',
 'DE',
 'EPS',
 'Zscore']

In [7]:
df_test.replace([np.inf, -np.inf], 0, inplace=True)

In [565]:
# df_test.replace([np.inf, -np.inf], 0, inplace=True)

In [8]:
train_data = scaler.fit_transform(df_train[vif_cols])
test_data = scaler.transform(df_test[vif_cols])

In [9]:
df_train1 = pd.DataFrame(train_data,columns=vif_cols)
df_test1 = pd.DataFrame(test_data,columns=vif_cols)

In [10]:
for col in ['Ticker','Target', 'Quarter']:
    df_train1[col] = df_train[col]
    df_test1[col] = df_test[col]

In [14]:

def convert_qtr_to_date(qtr):
    y = int(qtr.split("FY")[1])
    q = int(qtr.split("FY")[0][-1])
    quarter_end_dates = {
        "Q1": "30-06-",
        "Q2": "30-09-",
        "Q3": "31-12-",
        "Q4": "31-03-"
    }
    y = y if q==4 else y-1
    fiscal_year = 2000 + int(y)
    end_date = quarter_end_dates.get(f'Q{q}')
    if end_date:
        return end_date + str(fiscal_year)  # Concatenate the end date with the full year
    else:
        raise ValueError("Invalid quarter format")

In [15]:
df_master['Quarter'] = pd.to_datetime(df_master['Quarter'].apply(convert_qtr_to_date),dayfirst=True)

In [16]:
df_train1 = df_train1.merge(df_master[['Ticker','Quarter','Returns','Rank']],on=['Ticker','Quarter'])
df_test1 = df_test1.merge(df_master[['Ticker','Quarter','Returns','Rank']],on=['Ticker','Quarter'])

In [17]:
df_train1['Target'] = np.where(df_train1['Rank']<=200,1,0)
df_test1['Target'] = np.where(df_test1['Rank']<=200,1,0)

#### Model

In [18]:
features = vif_cols.copy()
features

['TMR',
 'Momentum',
 'Beta',
 'Alpha',
 'EP',
 'BP',
 'SP',
 'Dividend_Flag',
 'DE',
 'EPS',
 'Zscore']

In [19]:
# features.remove("Quarter")
X_train = df_train1[features].values
y_train = df_train1['Target'].values

X_test = df_test1[features].values
y_test = df_test1['Target'].values

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd
import numpy as np

def calculate_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'roc':roc_auc_score(y_true,y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
        'confusion_matrix': confusion_matrix(y_true, y_pred)
    }

def train_logistic_regression_top200(X_train, y_train, X_test, y_test, k=5):
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    fold_metrics = []

    for train_index, val_index in kf.split(X_train):
        X_train_fold, X_val = X_train[train_index], X_train[val_index]
        y_train_fold, y_val = y_train[train_index], y_train[val_index]

        model = LogisticRegression(max_iter=1000)
        model.fit(X_train_fold, y_train_fold)

        y_val_pred = model.predict(X_val)
        metrics = calculate_metrics(y_val, y_val_pred)
        fold_metrics.append(metrics)

    # Average scalar metrics
    avg_metrics = {
        key: np.mean([m[key] for m in fold_metrics]) for key in ['accuracy','roc', 'precision', 'recall', 'f1']
    }

    # Sum confusion matrices
    conf_matrix_sum = np.sum([m['confusion_matrix'] for m in fold_metrics], axis=0)


    # Final model on full training set
    final_model = LogisticRegression(max_iter=1000)
    final_model.fit(X_train, y_train)
    y_test_pred = final_model.predict(X_test)
    test_metrics = calculate_metrics(y_test, y_test_pred)

    # Combine into DataFrame
    metrics_df = pd.DataFrame({
        'set': ['cross_val_avg', 'test'],
        'accuracy': [avg_metrics['accuracy'], test_metrics['accuracy']],
        'roc':[avg_metrics['roc'],test_metrics['roc']],
        'precision': [avg_metrics['precision'], test_metrics['precision']],
        'recall': [avg_metrics['recall'], test_metrics['recall']],
        'f1': [avg_metrics['f1'], test_metrics['f1']],
        'confusion_matrix': [conf_matrix_sum, test_metrics['confusion_matrix']]
    })

    return metrics_df,model

# Reshape if needed
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)

In [20]:
metrics_df,lr_model_top200 = train_logistic_regression_top200(X_train, y_train, X_test, y_test)

In [21]:
metrics_df

,set,accuracy,roc,precision,recall,f1,confusion_matrix
0,cross_val_avg,0.700105,0.501029,0.469687,0.004349,0.008615,"[[30038, 69], [12822, 56]]"
1,test,0.783556,0.500715,0.320000,0.003439,0.006806,"[[8445, 17], [2318, 8]]"


In [ ]:
with open(r"D:\Python\Final Quant Fund\Quant Fund\Fund1\src\3yrs_as_test_data_ts\lr_top200.pkl","wb") as file:
    pkl.dump(lr_model_top200, file)

In [22]:
pred_probs_test = lr_model_top200.predict_proba(X_test)[:,1]
pred_probs_train = lr_model_top200.predict_proba(X_train)[:,1]

In [23]:
np.isnan(pred_probs_test).sum()

np.int64(0)

In [24]:
np.isnan(pred_probs_train).sum()

np.int64(0)

In [25]:

def calculate_metrics1(y_true, y_pred_label,y_pred_prob):
    accuracy = accuracy_score(y_true, y_pred_label)
    precision = precision_score(y_true, y_pred_label)
    recall = recall_score(y_true, y_pred_label)
    f1 = f1_score(y_true, y_pred_label)
    confusion = confusion_matrix(y_true,y_pred_label)
    roc_score = roc_auc_score(y_true,y_pred_prob)
    
    return {
        'accuracy': accuracy,
        'roc':roc_score,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix':confusion
    }



In [26]:
train_metrics1 = calculate_metrics1(y_train,(pred_probs_train>=0.5).astype(int),pred_probs_train)
test_metrics1 = calculate_metrics1(y_test,(pred_probs_test>=0.5).astype(int),pred_probs_test)

In [27]:
train_metrics_df1 = pd.DataFrame([train_metrics1])
train_metrics_df1['Set'] = "Train"
test_metrics_df1 = pd.DataFrame([test_metrics1])
test_metrics_df1['Set'] = "Test"
model_metrics1 = pd.concat([train_metrics_df1,test_metrics_df1],axis=0)
model_metrics1.reset_index(drop=True,inplace=True)
model_metrics1.head()

,accuracy,roc,precision,recall,f1,confusion_matrix,Set
0,0.700151,0.550254,0.451327,0.003960,0.007852,"[[30045, 62], [12827, 51]]",Train
1,0.783834,0.563888,0.375000,0.003869,0.007660,"[[8447, 15], [2317, 9]]",Test


In [28]:
model_name = 'lr_top200'
top_n = 30

In [29]:

def convert_qtr_to_date(qtr):
    y = int(qtr.split("FY")[1])
    q = int(qtr.split("FY")[0][-1])
    quarter_end_dates = {
        "Q1": "30-06-",
        "Q2": "30-09-",
        "Q3": "31-12-",
        "Q4": "31-03-"
    }
    y = y if q==4 else y-1
    fiscal_year = 2000 + int(y)
    end_date = quarter_end_dates.get(f'Q{q}')
    if end_date:
        return end_date + str(fiscal_year)  # Concatenate the end date with the full year
    else:
        raise ValueError("Invalid quarter format")

In [30]:
df_master = pd.read_excel(master_data_path)

In [31]:
df_master['Quarter'] = pd.to_datetime(df_master['Quarter'].apply(convert_qtr_to_date),dayfirst=True)

In [588]:
# df_pred1.columns

In [32]:
def label_prediction(row, model_name,top_n):
    in_top_50 = row[f'Top_{top_n}_Pred_Prob_{model_name}']
    target = row['Target']
    
    if in_top_50 and target == 1:
        return 'TP'
    elif in_top_50 and target == 0:
        return 'FP'
    elif not in_top_50 and target == 0:
        return 'TN'
    elif not in_top_50 and target == 1:
        return 'FN'

def flag_top50_nlargest(group, pred_prob_col,top_n):
    # If all pred_probs are zero, return all False
    if group[pred_prob_col].sum() == 0:
        return pd.Series([False] * len(group), index=group.index)
    # Use nlargest to get exactly 50
    top50_idx = group[pred_prob_col].nlargest(top_n).index
    flag = pd.Series(False, index=group.index)
    flag.loc[top50_idx] = True
    return flag

def label_prediction1(row, model_name):
    has_more_prob_than_threshold = row[f'Pred_Prob_Threshold_{model_name}']
    target = row['Target']
    
    if has_more_prob_than_threshold and target == 1:
        return 'TP'
    elif has_more_prob_than_threshold and target == 0:
        return 'FP'
    elif not has_more_prob_than_threshold and target == 0:
        return 'TN'
    elif not has_more_prob_than_threshold and target == 1:
        return 'FN'
    
def add_suffix_to_cols(df, suffix, exclude_cols):
    return df.rename(
        columns={col: f"{col}{suffix}" for col in df.columns if col not in exclude_cols + ['Quarter', 'Set']}
    )


In [33]:
def label_prediction(row, model_name,top_n):
    in_top_50 = row[f'Top_{top_n}_Pred_Prob_{model_name}']
    target = row['Target']
    
    if in_top_50 and target == 1:
        return 'TP'
    elif in_top_50 and target == 0:
        return 'FP'
    elif not in_top_50 and target == 0:
        return 'TN'
    elif not in_top_50 and target == 1:
        return 'FN'

def flag_top50_nlargest(group, pred_prob_col,top_n):
    # If all pred_probs are zero, return all False
    if group[pred_prob_col].sum() == 0:
        return pd.Series([False] * len(group), index=group.index)
    # Use nlargest to get exactly 50
    top50_idx = group[pred_prob_col].nlargest(top_n).index
    flag = pd.Series(False, index=group.index)
    flag.loc[top50_idx] = True
    return flag

def label_prediction1(row, model_name):
    has_more_prob_than_threshold = row[f'Pred_Prob_Threshold_{model_name}']
    target = row['Target']
    
    if has_more_prob_than_threshold and target == 1:
        return 'TP'
    elif has_more_prob_than_threshold and target == 0:
        return 'FP'
    elif not has_more_prob_than_threshold and target == 0:
        return 'TN'
    elif not has_more_prob_than_threshold and target == 1:
        return 'FN'
    
def add_suffix_to_cols(df, suffix, exclude_cols):
    return df.rename(
        columns={col: f"{col}{suffix}" for col in df.columns if col not in exclude_cols + ['Quarter', 'Set']}
    )


if len(set(['Rank','Returns'])-set(df_train1.columns))!=0:
    print("There are no rank and returns columns, merging now..!")
    df_train1 = df_train1.merge(df_master[['Ticker','Quarter','Returns','Rank']],on=['Ticker','Quarter'],how='left')
    df_test1 = df_test1.merge(df_master[['Ticker','Quarter','Returns','Rank']],on=['Ticker','Quarter'],how='left')

if np.isnan(pred_probs_train).all():
    print("All predictionsa re NaNs")

df_train_pred = df_train1[['Ticker','Quarter','Target','Returns','Rank']]
df_test_pred = df_test1[['Ticker','Quarter','Target','Returns','Rank']]

df_test_pred[f'Pred_Prob_{model_name}'] = pred_probs_test.reshape(-1)
df_train_pred[f'Pred_Prob_{model_name}'] = pred_probs_train.reshape(-1)

#### Top30 as fund - irrespective of the prob values
df_train_pred.loc[:,'Set'] = 'Train'
df_test_pred.loc[:,'Set'] = 'Test'
df_pred = pd.concat([df_train_pred,df_test_pred],axis=0)



df_pred.reset_index(drop=True,inplace=True)
df_pred.sort_values(by=['Quarter',f'Pred_Prob_{model_name}'],ascending=False,inplace=True)
df_pred[f'Top_{top_n}_Pred_Prob_{model_name}'] = df_pred.groupby('Quarter', group_keys=False).apply(flag_top50_nlargest, f'Pred_Prob_{model_name}',top_n)

df_pred[f'Top_50_Pred_Prob_{model_name}'] = df_pred.groupby('Quarter', group_keys=False).apply(flag_top50_nlargest, f'Pred_Prob_{model_name}',50)
# df_pred[(df_pred['Top_{top_n}_Pred_Prob_lstm_top50']==True)&(df_pred['Quarter']=='2005-12-31')]

# test_qtr_preds = df_pred[(df_pred['Quarter']=='2005-12-31')]['Pred_Prob_lstm_top50']
# test_qtr_preds[test_qtr_preds>0.9].shape
df_pred['Pred_Category_Top30'] = df_pred.apply(lambda row: label_prediction(row, model_name=model_name,top_n=30), axis=1)
df_pred['Pred_Category_Top50'] = df_pred.apply(lambda row: label_prediction(row, model_name=model_name,top_n=50), axis=1)
df_pred.reset_index(drop=True,inplace=True)

qtr_summary_df = df_pred.groupby('Quarter').agg(
                    Set = ('Set','first'),
                    Avg_Returns_Predicted_Top30=('Returns',lambda x: x[df_pred.loc[x.index, f'Top_30_Pred_Prob_{model_name}'] == True].mean()),
                    Avg_Returns_Predicted_Top50=('Returns',lambda x: x[df_pred.loc[x.index, f'Top_50_Pred_Prob_{model_name}'] == True].mean()),
                    Count_TP_Top30=('Pred_Category_Top30', lambda x: (x == 'TP').sum()),
                    Count_TN_Top30=('Pred_Category_Top30', lambda x: (x == 'TN').sum()),
                    Count_FP_Top30=('Pred_Category_Top30', lambda x: (x == 'FP').sum()),
                    Count_FN_Top30=('Pred_Category_Top30', lambda x: (x == 'FN').sum()),
                    Count_TP_Top50=('Pred_Category_Top50', lambda x: (x == 'TP').sum()),
                    Count_TN_Top50=('Pred_Category_Top50', lambda x: (x == 'TN').sum()),
                    Count_FP_Top50=('Pred_Category_Top50', lambda x: (x == 'FP').sum()),
                    Count_FN_Top50=('Pred_Category_Top50', lambda x: (x == 'FN').sum())
                    
                    ).reset_index()

# set_summary_df = qtr_summary_df.groupby('Set').agg(
#     Count_TP = ('Count_TP','sum'),
#     Count_TN = ('Count_TN','sum'),
#     Count_FP = ('Count_FP','sum'),
#     Count_FN = ('Count_FN','sum')
# )

df_pred1 = df_pred[['Ticker','Quarter','Target','Returns','Rank',f'Pred_Prob_{model_name}','Set']].copy()
df_pred1[f'Pred_Prob_Threshold_{model_name}'] = df_pred1[f'Pred_Prob_{model_name}'].between(0.5,1)



df_pred1['Pred_Category'] = df_pred1.apply(lambda row: label_prediction1(row, model_name=model_name), axis=1)
quarterly_returns_f2 = df_pred1.groupby('Quarter').agg(
                    Set = ('Set','first'),
                    Avg_Returns_Predicted_Thresh=('Returns', lambda x: x[df_pred1.loc[x.index, f'Pred_Prob_Threshold_{model_name}'] == True].mean()),
                    Count_TP_Thresh=('Pred_Category', lambda x: (x == 'TP').sum()),
                    Count_TN_Thresh=('Pred_Category', lambda x: (x == 'TN').sum()),
                    Count_FP_Thresh=('Pred_Category', lambda x: (x == 'FP').sum()),
                    Count_FN_Thresh=('Pred_Category', lambda x: (x == 'FN').sum())
                    ).reset_index()

qtr_summary_df2 = pd.DataFrame(quarterly_returns_f2)


#### Actual and Index returns mapping
##### Actual returns
act_returns = df_pred[df_pred['Target']==1].groupby(by='Quarter')['Returns'].mean()
df_act_returns = pd.DataFrame(act_returns)
df_act_returns.reset_index(inplace=True)
df_act_returns.rename(columns={'Returns':'Avg_Returns_Actual'},inplace=True)
qtr_summary_df = qtr_summary_df.merge(df_act_returns,on='Quarter',how='left')

##### Index returns - Qtrly
df_index = pd.read_excel(qtrly_returns_path,sheet_name='Qtrly_Stock_Returns')
df_index1 = df_index[df_index['Ticker'] == 'Nsemcap'].drop(columns=['Industry']).set_index(keys=['Ticker'])
df_index1 = df_index1.T
df_index1.reset_index(inplace=True)
df_index1.rename(columns={'index':'Quarter','Nsemcap':'Index_Returns'},inplace=True)

df_index1['Quarter'] = df_index1['Quarter'].apply(convert_qtr_to_date)
df_index1['Quarter'] = pd.to_datetime(df_index1['Quarter'],dayfirst=True)

qtr_summary_df = qtr_summary_df.merge(df_index1,how='left',on='Quarter')
qtr_summary_df['delta_returns_act_vs_pred_Top30'] = qtr_summary_df['Avg_Returns_Actual']-qtr_summary_df['Avg_Returns_Predicted_Top30']
qtr_summary_df['delta_returns_pred_vs_index_Top30'] = qtr_summary_df['Avg_Returns_Predicted_Top30']-qtr_summary_df['Index_Returns']
qtr_summary_df['delta_returns_act_vs_pred_Top50'] = qtr_summary_df['Avg_Returns_Actual']-qtr_summary_df['Avg_Returns_Predicted_Top50']
qtr_summary_df['delta_returns_pred_vs_index_Top50'] = qtr_summary_df['Avg_Returns_Predicted_Top50']-qtr_summary_df['Index_Returns']

qtr_summary_df2 = qtr_summary_df2.merge(df_act_returns,on='Quarter',how='left')
qtr_summary_df2 = qtr_summary_df2.merge(df_index1,on='Quarter',how='left')

qtr_summary_df2['delta_returns_act_vs_pred_Thresh'] = qtr_summary_df2['Avg_Returns_Actual']-qtr_summary_df2['Avg_Returns_Predicted_Thresh']
qtr_summary_df2['delta_returns_pred_vs_index_Thresh'] = qtr_summary_df2['Avg_Returns_Predicted_Thresh']-qtr_summary_df2['Index_Returns']


exclude_cols = ['Avg_Returns_Actual', 'Index_Returns']

# Reset index for merge
qtr_summary_df2_reset = qtr_summary_df2.reset_index(drop=True)
qtr_summary_df_reset = qtr_summary_df.reset_index(drop=True)

# Merge as before
merged_qtr_summary = pd.merge(
    qtr_summary_df_reset, qtr_summary_df2_reset,
    on=['Quarter', 'Set', 'Avg_Returns_Actual', 'Index_Returns'],
    how='inner',
    suffixes=('', '')
)

# Define the desired column order
ordered_cols = [
    'Quarter','Set',
    'Avg_Returns_Predicted_Top30','Avg_Returns_Predicted_Top50',
    'Avg_Returns_Predicted_Thresh',
    'Avg_Returns_Actual', 'Index_Returns',
    'delta_returns_act_vs_pred_Top30','delta_returns_pred_vs_index_Top30',
    'delta_returns_act_vs_pred_Top50','delta_returns_pred_vs_index_Top50',
    'delta_returns_act_vs_pred_Thresh','delta_returns_pred_vs_index_Thresh',
    'Count_TP_Top30', 'Count_TP_Top50', 'Count_TP_Thresh',
    'Count_TN_Top30', 'Count_TN_Top50', 'Count_TN_Thresh',
    'Count_FP_Top30', 'Count_FP_Top50', 'Count_FP_Thresh',
    'Count_FN_Top30', 'Count_FN_Top50', 'Count_FN_Thresh'
]

# Add any other columns that might exist but are not in the list
ordered_cols += [col for col in merged_qtr_summary.columns if col not in ordered_cols]

# Reorder columns
merged_qtr_summary = merged_qtr_summary[ordered_cols]
# merged_qtr_summary.reset_index(inplace=True)
merged_qtr_summary.head()

C:\Users\43737\AppData\Local\Temp\ipykernel_25352\2793545741.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test_pred[f'Pred_Prob_{model_name}'] = pred_probs_test.reshape(-1)
C:\Users\43737\AppData\Local\Temp\ipykernel_25352\2793545741.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train_pred[f'Pred_Prob_{model_name}'] = pred_probs_train.reshape(-1)
C:\Users\43737\AppData\Local\Temp\ipykernel_25352\2793545741.py:58: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice 

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Avg_Returns_Predicted_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,...,Count_TP_Thresh,Count_TN_Top30,Count_TN_Top50,Count_TN_Thresh,Count_FP_Top30,Count_FP_Top50,Count_FP_Thresh,Count_FN_Top30,Count_FN_Top50,Count_FN_Thresh
0,2005-03-31,Train,0.369210,0.426247,0.492816,0.532415,-0.019376,0.163205,0.388586,0.106168,...,3,279,271,294,15,23,0,176,164,188
1,2005-06-30,Train,0.343900,0.547388,0.432813,0.645971,0.045476,0.302071,0.298424,0.098583,...,4,276,271,287,15,20,4,179,164,190
2,2005-09-30,Train,0.117348,0.117327,0.364651,0.231957,0.244277,0.114609,-0.126929,0.114630,...,4,273,264,286,14,23,1,182,171,194
3,2005-12-31,Train,0.283221,0.269397,0.902536,0.471668,0.058583,0.188447,0.224637,0.202271,...,1,273,264,288,15,24,0,182,171,196
4,2006-03-31,Train,0.269096,0.132320,0.086849,0.085933,0.187746,-0.183163,0.081350,-0.046386,...,3,279,267,292,14,26,1,181,173,194


In [34]:
merged_qtr_summary['Avg_Returns_Predicted_Thresh'].isna().all()

np.False_

In [35]:
merged_qtr_summary.columns

Index(['Quarter', 'Set', 'Avg_Returns_Predicted_Top30',
       'Avg_Returns_Predicted_Top50', 'Avg_Returns_Predicted_Thresh',
       'Avg_Returns_Actual', 'Index_Returns',
       'delta_returns_act_vs_pred_Top30', 'delta_returns_pred_vs_index_Top30',
       'delta_returns_act_vs_pred_Top50', 'delta_returns_pred_vs_index_Top50',
       'delta_returns_act_vs_pred_Thresh',
       'delta_returns_pred_vs_index_Thresh', 'Count_TP_Top30',
       'Count_TP_Top50', 'Count_TP_Thresh', 'Count_TN_Top30', 'Count_TN_Top50',
       'Count_TN_Thresh', 'Count_FP_Top30', 'Count_FP_Top50',
       'Count_FP_Thresh', 'Count_FN_Top30', 'Count_FN_Top50',
       'Count_FN_Thresh'],
      dtype='object')

In [36]:
df_pred[df_pred['Set']=='Test'][f'Pred_Prob_{model_name}'].unique()

array([0.99999998, 0.44581378, 0.3929035 , ..., 0.20020551, 0.18565925,
       0.10029739])

In [37]:
set_summary_df3 =  merged_qtr_summary.groupby('Set').agg(
    Count_TP_Thresh = ('Count_TP_Thresh','sum'),
    Count_TN_Thresh = ('Count_TN_Thresh','sum'),
    Count_FP_Thresh = ('Count_FP_Thresh','sum'),
    Count_FN_Thresh = ('Count_FN_Thresh','sum'),
    Count_TP_Top30 = ('Count_TP_Top30','sum'),
    Count_TN_Top30 = ('Count_TN_Top30','sum'),
    Count_FP_Top30 = ('Count_FP_Top30','sum'),
    Count_FN_Top30 = ('Count_FN_Top30','sum'),
    Count_TP_Top50 = ('Count_TP_Top50','sum'),
    Count_TN_Top50 = ('Count_TN_Top50','sum'),
    Count_FP_Top50 = ('Count_FP_Top50','sum'),
    Count_FN_Top50 = ('Count_FN_Top50','sum')
)
set_summary_df3

,Count_TP_Thresh,Count_TN_Thresh,Count_FP_Thresh,Count_FN_Thresh,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50
Set,,,,,,,,,,,,
Test,9,8447,15,2317,124,8226,236,2202,194,8056,406,2132
Train,51,30045,62,12827,785,28852,1255,12093,1286,27993,2114,11592


In [38]:

def calc_metrics(tp, tn, fp, fn):
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return accuracy, precision, recall, f1


In [39]:
for suffix in ['_Top30','_Top50', '_Thresh']:
    print(f"\nMetrics for {suffix}:")
    for set_name, row in set_summary_df3.iterrows():
        tp = row[f'Count_TP{suffix}']
        tn = row[f'Count_TN{suffix}']
        fp = row[f'Count_FP{suffix}']
        fn = row[f'Count_FN{suffix}']
        accuracy, precision, recall, f1 = calc_metrics(tp, tn, fp, fn)
        print(f"{set_name}: accuracy={accuracy:.4f}, precision={precision:.4f}, recall={recall:.4f}, f1={f1:.4f}")


Metrics for _Top30:
Test: accuracy=0.7740, precision=0.3444, recall=0.0533, f1=0.0923
Train: accuracy=0.6895, precision=0.3848, recall=0.0610, f1=0.1052

Metrics for _Top50:
Test: accuracy=0.7647, precision=0.3233, recall=0.0834, f1=0.1326
Train: accuracy=0.6811, precision=0.3782, recall=0.0999, f1=0.1580

Metrics for _Thresh:
Test: accuracy=0.7838, precision=0.3750, recall=0.0039, f1=0.0077
Train: accuracy=0.7002, precision=0.4513, recall=0.0040, f1=0.0079


In [40]:
if merged_qtr_summary['Avg_Returns_Predicted_Thresh'].mean()==merged_qtr_summary['Avg_Returns_Predicted_Top30'].mean():
    print("Both the stock picking strategies (Top 30 and Pred prob theshold) have same returns")
if merged_qtr_summary['Avg_Returns_Predicted_Thresh'].mean()==merged_qtr_summary['Avg_Returns_Predicted_Top50'].mean():
    print("Both the stock picking strategies (Top 50 and Pred prob theshold) have same returns")
if merged_qtr_summary['Avg_Returns_Predicted_Top30'].mean()==merged_qtr_summary['Avg_Returns_Predicted_Top50'].mean():
    print("Both the stock picking strategies (Top 30 and Top 50) have same returns")
else:
    print("All of the stock picking strategies (Top 30, Top 50 and Pred prob theshold) have different returns")

All of the stock picking strategies (Top 30, Top 50 and Pred prob theshold) have different returns


In [598]:
qtr_summary_df[qtr_summary_df['Quarter']=='2021-09-30']

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,delta_returns_pred_vs_index_Top50
66,2021-09-30,Train,0.46959,0.406957,18,617,12,175,25,604,25,168,0.484905,0.126554,0.015315,0.343036,0.077948,0.280403


In [599]:
qtr_summary_df2[qtr_summary_df2['Quarter']=='2021-09-30']

,Quarter,Set,Avg_Returns_Predicted_Thresh,Count_TP_Thresh,Count_TN_Thresh,Count_FP_Thresh,Count_FN_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Thresh,delta_returns_pred_vs_index_Thresh
66,2021-09-30,Train,0.781025,4,628,1,189,0.484905,0.126554,-0.29612,0.654471


In [600]:
df_pred.columns

Index(['Ticker', 'Quarter', 'Target', 'Returns', 'Rank', 'Pred_Prob_lr_top200',
       'Set', 'Top_30_Pred_Prob_lr_top200', 'Top_50_Pred_Prob_lr_top200',
       'Pred_Category_Top30', 'Pred_Category_Top50'],
      dtype='object')

In [601]:
df_pred1.columns

Index(['Ticker', 'Quarter', 'Target', 'Returns', 'Rank', 'Pred_Prob_lr_top200',
       'Set', 'Pred_Prob_Threshold_lr_top200', 'Pred_Category'],
      dtype='object')

In [602]:
merged_qtr_summary[merged_qtr_summary['Quarter']=='2021-09-30']

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Avg_Returns_Predicted_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,...,Count_TP_Thresh,Count_TN_Top30,Count_TN_Top50,Count_TN_Thresh,Count_FP_Top30,Count_FP_Top50,Count_FP_Thresh,Count_FN_Top30,Count_FN_Top50,Count_FN_Thresh
66,2021-09-30,Train,0.46959,0.406957,0.781025,0.484905,0.126554,0.015315,0.343036,0.077948,...,4,617,604,628,12,25,1,175,168,189


In [41]:
merged_qtr_summary[merged_qtr_summary['Avg_Returns_Predicted_Top30'].isna()].shape

(2, 25)

In [42]:
df_pred[df_pred['Set']=='Test'][f'Pred_Prob_{model_name}'].unique()

array([0.99999998, 0.44581378, 0.3929035 , ..., 0.20020551, 0.18565925,
       0.10029739])

In [43]:
df_pred[df_pred['Set']=='Test'][f'Pred_Prob_{model_name}'].describe()

count    10788.000000
mean         0.296558
std          0.033730
min          0.087024
25%          0.285306
50%          0.295458
75%          0.305917
max          1.000000
Name: Pred_Prob_lr_top200, dtype: float64

In [44]:
merged_qtr_summary['Quarter'].is_monotonic_increasing

True

In [45]:
set_summary_df3

,Count_TP_Thresh,Count_TN_Thresh,Count_FP_Thresh,Count_FN_Thresh,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50
Set,,,,,,,,,,,,
Test,9,8447,15,2317,124,8226,236,2202,194,8056,406,2132
Train,51,30045,62,12827,785,28852,1255,12093,1286,27993,2114,11592


In [46]:
merged_qtr_summary['Avg_Returns_Predicted_Top30'] = merged_qtr_summary['Avg_Returns_Predicted_Top30'].ffill()
merged_qtr_summary['Avg_Returns_Predicted_Top50'] = merged_qtr_summary['Avg_Returns_Predicted_Top50'].ffill()
merged_qtr_summary['Avg_Returns_Predicted_Thresh'] = merged_qtr_summary['Avg_Returns_Predicted_Thresh'].ffill()

In [47]:
merged_qtr_summary['Avg_Returns_Predicted_Top30'].fillna(0,inplace=True)
merged_qtr_summary['Avg_Returns_Predicted_Top50'].fillna(0,inplace=True)
merged_qtr_summary['Avg_Returns_Predicted_Thresh'].fillna(0,inplace=True)

C:\Users\43737\AppData\Local\Temp\ipykernel_25352\4263771019.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_qtr_summary['Avg_Returns_Predicted_Top30'].fillna(0,inplace=True)
C:\Users\43737\AppData\Local\Temp\ipykernel_25352\4263771019.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always beh

In [610]:
merged_qtr_summary.head()

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Avg_Returns_Predicted_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,...,Count_TP_Thresh,Count_TN_Top30,Count_TN_Top50,Count_TN_Thresh,Count_FP_Top30,Count_FP_Top50,Count_FP_Thresh,Count_FN_Top30,Count_FN_Top50,Count_FN_Thresh
0,2005-03-31,Train,0.369210,0.426247,0.492816,0.532415,-0.019376,0.163205,0.388586,0.106168,...,3,279,271,294,15,23,0,176,164,188
1,2005-06-30,Train,0.343900,0.547388,0.432813,0.645971,0.045476,0.302071,0.298424,0.098583,...,4,276,271,287,15,20,4,179,164,190
2,2005-09-30,Train,0.117348,0.117327,0.364651,0.231957,0.244277,0.114609,-0.126929,0.114630,...,4,273,264,286,14,23,1,182,171,194
3,2005-12-31,Train,0.283221,0.269397,0.902536,0.471668,0.058583,0.188447,0.224637,0.202271,...,1,273,264,288,15,24,0,182,171,196
4,2006-03-31,Train,0.269096,0.132320,0.086849,0.085933,0.187746,-0.183163,0.081350,-0.046386,...,3,279,267,292,14,26,1,181,173,194


In [611]:
df_pred.head()

,Ticker,Quarter,Target,Returns,Rank,Pred_Prob_lr_top200,Set,Top_30_Pred_Prob_lr_top200,Top_50_Pred_Prob_lr_top200,Pred_Category_Top30,Pred_Category_Top50
0,uel,2024-12-31,0,NaN,NaN,1.000000,Test,True,True,FP,FP
1,diacabs,2024-12-31,0,-0.418070,777.0,0.445814,Test,True,True,FP,FP
2,wesl,2024-12-31,0,-0.279325,626.0,0.392903,Test,True,True,FP,FP
3,skpi,2024-12-31,0,-0.081491,228.0,0.376469,Test,True,True,FP,FP
4,apr,2024-12-31,0,NaN,NaN,0.369224,Test,True,True,FP,FP


In [612]:
qtr_summary_df.head()

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,delta_returns_pred_vs_index_Top50
0,2005-03-31,Train,0.369210,0.426247,15,279,15,176,27,271,23,164,0.532415,-0.019376,0.163205,0.388586,0.106168,0.445623
1,2005-06-30,Train,0.343900,0.547388,15,276,15,179,30,271,20,164,0.645971,0.045476,0.302071,0.298424,0.098583,0.501912
2,2005-09-30,Train,0.117348,0.117327,16,273,14,182,27,264,23,171,0.231957,0.244277,0.114609,-0.126929,0.114630,-0.126950
3,2005-12-31,Train,0.283221,0.269397,15,273,15,182,26,264,24,171,0.471668,0.058583,0.188447,0.224637,0.202271,0.210814
4,2006-03-31,Train,0.269096,0.132320,16,279,14,181,24,267,26,173,0.085933,0.187746,-0.183163,0.081350,-0.046386,-0.055426


In [48]:
qtr_summary_df['Set'].value_counts()

Set
Train    68
Test     12
Name: count, dtype: int64

In [49]:
with pd.ExcelWriter(r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\Fund1\results\lag\top200\3yrs_as_test_data_ts\Pred_Vs_Act_LR_time_split_25_07_2025.xlsx", ) as writer:
    merged_qtr_summary.to_excel(writer, sheet_name='Qtr_Summary', index=False)
    qtr_summary_df.to_excel(writer, sheet_name='Qtr_Summary_Top', index=False)
    qtr_summary_df2.to_excel(writer, sheet_name='Qtr_Summary_Thresh', index=False)
    df_pred.to_excel(writer, sheet_name='Pred_Vs_Act_Top', index=False)
    df_pred1.to_excel(writer, sheet_name='Pred_Vs_Act_Thresh', index=False)
    set_summary_df3.to_excel(writer, sheet_name='Set_Summary')

### Top200 Xform

In [50]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Flatten, Concatenate, Reshape, RepeatVector
from tensorflow.keras.models import Model
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
import tensorflow as tf
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,confusion_matrix,roc_auc_score
from tensorflow.keras.metrics import AUC, Precision
import os
from scipy.stats import skew

In [51]:

def convert_qtr_to_date(qtr):
    y = int(qtr.split("FY")[1])
    q = int(qtr.split("FY")[0][-1])
    quarter_end_dates = {
        "Q1": "30-06-",
        "Q2": "30-09-",
        "Q3": "31-12-",
        "Q4": "31-03-"
    }
    y = y if q==4 else y-1
    fiscal_year = 2000 + int(y)
    end_date = quarter_end_dates.get(f'Q{q}')
    if end_date:
        return end_date + str(fiscal_year)  # Concatenate the end date with the full year
    else:
        raise ValueError("Invalid quarter format")

In [52]:
train_data_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\Fund1\data\lag\train_data_after_multicollinearity_correlation_3yrs_test_size_time_based_split_2025_07_25.xlsx"
test_data_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\Fund1\data\lag\test_data_after_multicollinearity_correlation_3yrs_test_size_time_based_split_2025_07_25.xlsx"
master_data_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\data\Time_Series_Master_wo_Index_Last20Yrs1.xlsx"

df_train = pd.read_excel(train_data_path)
df_test = pd.read_excel(test_data_path)
df_master = pd.read_excel(master_data_path)
qtrly_returns_path = r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\data\lag\Quarterly_Price_Returns_Labels.xlsx"

In [53]:
df_train.reset_index(drop=True,inplace=True)
df_test.reset_index(drop=True,inplace=True)

In [619]:
df_train.columns

Index(['Ticker', 'Quarter', 'TMR', 'Momentum', 'Beta', 'Alpha', 'EP', 'BP',
       'SP', 'Dividend_Flag', 'DE', 'EPS', 'Zscore', 'Target'],
      dtype='object')

##### Scaling

In [54]:
scaler = StandardScaler()

In [621]:
df_train.columns

Index(['Ticker', 'Quarter', 'TMR', 'Momentum', 'Beta', 'Alpha', 'EP', 'BP',
       'SP', 'Dividend_Flag', 'DE', 'EPS', 'Zscore', 'Target'],
      dtype='object')

In [55]:
vif_cols = list(df_train.columns)
for col in ['Ticker','Quarter','Set','Target']:
    if col in vif_cols:
        vif_cols.remove(col)

In [623]:
vif_cols

['TMR',
 'Momentum',
 'Beta',
 'Alpha',
 'EP',
 'BP',
 'SP',
 'Dividend_Flag',
 'DE',
 'EPS',
 'Zscore']

In [56]:
df_test.replace([np.inf, -np.inf], 0, inplace=True)

In [625]:
# df_test.replace([np.inf, -np.inf], 0, inplace=True)

In [57]:
train_data = scaler.fit_transform(df_train[vif_cols])
test_data = scaler.transform(df_test[vif_cols])

In [58]:
df_train1 = pd.DataFrame(train_data,columns=vif_cols)
df_test1 = pd.DataFrame(test_data,columns=vif_cols)

In [59]:
for col in ['Ticker','Target', 'Quarter']:
    df_train1[col] = df_train[col]
    df_test1[col] = df_test[col]

#### tgt definition

In [60]:
df_master['Quarter'] = pd.to_datetime(df_master['Quarter'].apply(convert_qtr_to_date),dayfirst=True)
df_train1 = df_train1.merge(df_master[['Ticker','Quarter','Returns','Rank']],on=['Ticker','Quarter'])
df_test1 = df_test1.merge(df_master[['Ticker','Quarter','Returns','Rank']],on=['Ticker','Quarter'])
df_train1['Target'] = np.where(df_train1['Rank']<=200,1,0)
df_test1['Target'] = np.where(df_test1['Rank']<=200,1,0)

In [61]:
# ...existing code...
from sklearn.preprocessing import PowerTransformer
from scipy.stats import boxcox
import numpy as np

best_transforms = {}
for col in vif_cols:
    x = df_train[col].dropna()
    results = {}
    if (x <= 0).any():
        # Only Yeo-Johnson possible
        pt = PowerTransformer(method='yeo-johnson')
        x_yj = pt.fit_transform(x.values.reshape(-1, 1)).flatten()
        results['yeo-johnson'] = abs(skew(x_yj))
    else:
        x_log = np.log1p(x)
        results['log1p'] = abs(skew(x_log))
        try:
            x_bc, _ = boxcox(x)
            results['boxcox'] = abs(skew(x_bc))
        except Exception:
            results['boxcox'] = np.nan
    # Always include original for comparison
    results['original'] = abs(skew(x))
    # Pick the transform with minimum absolute skewness
    best = min(results, key=lambda k: results[k] if not np.isnan(results[k]) else np.inf)
    best_transforms[col] = best
    print(f"{col}: Best transform = {best}, skewness = {results[best]:.3f}")
# ...existing code...
#### Power xforms
# ...existing code...
from sklearn.preprocessing import PowerTransformer
from scipy.stats import boxcox
import numpy as np

# 1. Decide best transform for each feature based on your previous results
# best_transform_map = {
#     'TMR': 'yeo-johnson',
#     'Momentum': 'yeo-johnson',
#     'Beta': 'yeo-johnson',
#     'Alpha': 'yeo-johnson',
#     'EP': 'boxcox',
#     'BP': 'yeo-johnson',
#     'SP': 'boxcox',
#     'EPS': 'yeo-johnson',
#     'Zscore': 'yeo-johnson',
#     'FCF': 'yeo-johnson',
#     'ICR': 'yeo-johnson',
#     'Year': 'log1p',
#     'Quarter_number': 'boxcox',  # or 'log1p', both are close
#     # 'Target': 'none',  # Target is not a feature
# }

def transform_feature(col, x, method):
    if method == 'yeo-johnson':
        pt = PowerTransformer(method='yeo-johnson')
        return pt.fit_transform(x.values.reshape(-1, 1)).flatten()
    elif method == 'boxcox':
        # Box-Cox requires strictly positive values
        # Add a small constant if min(x) == 0
        x_pos = x.copy()
        if (x_pos <= 0).any():
            x_pos = x_pos - x_pos.min() + 1e-6
        return boxcox(x_pos)[0]
    elif method == 'log1p':
        x_pos = x.copy()
        if (x_pos < 0).any():
            x_pos = x_pos - x_pos.min() + 1e-6
        return np.log1p(x_pos)
    else:
        return x.values

# 2. Apply transforms to train and test
df_train_trans = df_train.copy()
df_test_trans = df_test.copy()

for col in vif_cols:
    method = best_transforms.get(col, 'none')
    if method != 'none':
        df_train_trans[col] = transform_feature(col, df_train[col], method)
        df_test_trans[col] = transform_feature(col, df_test[col], method)

# 3. Scaling
scaler = StandardScaler()
train_data = scaler.fit_transform(df_train_trans[vif_cols])
test_data = scaler.transform(df_test_trans[vif_cols])

df_train1 = pd.DataFrame(train_data, columns=vif_cols)
df_test1 = pd.DataFrame(test_data, columns=vif_cols)

for col in ['Ticker', 'Target', 'Quarter']:
    df_train1[col] = df_train[col]
    df_test1[col] = df_test[col]

# 4. Prepare X_train, y_train, X_test, y_test
features = vif_cols.copy()
X_train = df_train1[features].values
y_train = df_train1['Target'].values
X_test = df_test1[features].values
y_test = df_test1['Target'].values

# If your model expects 3D input (samples, timesteps, features):
# X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
# X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
# ...existing code...

TMR: Best transform = yeo-johnson, skewness = 1.374
Momentum: Best transform = yeo-johnson, skewness = 0.373
Beta: Best transform = yeo-johnson, skewness = 4.617
Alpha: Best transform = yeo-johnson, skewness = 2.939
EP: Best transform = boxcox, skewness = 0.110
BP: Best transform = yeo-johnson, skewness = 0.206
SP: Best transform = boxcox, skewness = 0.044
Dividend_Flag: Best transform = original, skewness = 1.256
DE: Best transform = yeo-johnson, skewness = 0.049
EPS: Best transform = yeo-johnson, skewness = 1.697
Zscore: Best transform = yeo-johnson, skewness = 0.086
X_train shape: (42985, 11)
y_train shape: (42985,)
X_test shape: (10788, 11)
y_test shape: (10788,)


In [62]:
features = vif_cols.copy()
features

['TMR',
 'Momentum',
 'Beta',
 'Alpha',
 'EP',
 'BP',
 'SP',
 'Dividend_Flag',
 'DE',
 'EPS',
 'Zscore']

#### Model

In [63]:
# features.remove("Quarter")
X_train = df_train1[features].values
y_train = df_train1['Target'].values

X_test = df_test1[features].values
y_test = df_test1['Target'].values

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd
import numpy as np

def calculate_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'roc':roc_auc_score(y_true,y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
        'confusion_matrix': confusion_matrix(y_true, y_pred)
    }

def train_logistic_regression_top200_xform(X_train, y_train, X_test, y_test, k=5):
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    fold_metrics = []

    for train_index, val_index in kf.split(X_train):
        X_train_fold, X_val = X_train[train_index], X_train[val_index]
        y_train_fold, y_val = y_train[train_index], y_train[val_index]

        model = LogisticRegression(max_iter=1000)
        model.fit(X_train_fold, y_train_fold)

        y_val_pred = model.predict(X_val)
        metrics = calculate_metrics(y_val, y_val_pred)
        fold_metrics.append(metrics)

    # Average scalar metrics
    avg_metrics = {
        key: np.mean([m[key] for m in fold_metrics]) for key in ['accuracy','roc', 'precision', 'recall', 'f1']
    }

    # Sum confusion matrices
    conf_matrix_sum = np.sum([m['confusion_matrix'] for m in fold_metrics], axis=0)


    # Final model on full training set
    final_model = LogisticRegression(max_iter=1000)
    final_model.fit(X_train, y_train)
    y_test_pred = final_model.predict(X_test)
    test_metrics = calculate_metrics(y_test, y_test_pred)

    # Combine into DataFrame
    metrics_df = pd.DataFrame({
        'set': ['cross_val_avg', 'test'],
        'accuracy': [avg_metrics['accuracy'], test_metrics['accuracy']],
        'roc':[avg_metrics['roc'],test_metrics['roc']],
        'precision': [avg_metrics['precision'], test_metrics['precision']],
        'recall': [avg_metrics['recall'], test_metrics['recall']],
        'f1': [avg_metrics['f1'], test_metrics['f1']],
        'confusion_matrix': [conf_matrix_sum, test_metrics['confusion_matrix']]
    })

    return metrics_df,model

# Reshape if needed
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)

metrics_df,lr_model_top200_xform = train_logistic_regression_top200_xform(X_train, y_train, X_test, y_test)

metrics_df

,set,accuracy,roc,precision,recall,f1,confusion_matrix
0,cross_val_avg,0.704525,0.545714,0.596134,0.131084,0.214829,"[[28546, 1179], [11522, 1738]]"
1,test,0.789303,0.557286,0.532824,0.150691,0.234938,"[[8166, 306], [1967, 349]]"


In [634]:
import pickle as pkl

In [ ]:
with open(r"D:\Python\Final Quant Fund\Quant Fund\Fund1\src\3yrs_as_test_data_ts\lr_top200_xform.pkl","wb") as file:
    pkl.dump(lr_model_top200_xform, file)

In [64]:
metrics_df

,set,accuracy,roc,precision,recall,f1,confusion_matrix
0,cross_val_avg,0.704525,0.545714,0.596134,0.131084,0.214829,"[[28546, 1179], [11522, 1738]]"
1,test,0.789303,0.557286,0.532824,0.150691,0.234938,"[[8166, 306], [1967, 349]]"


In [65]:
pred_probs_test = lr_model_top200_xform.predict_proba(X_test)[:,1]
pred_probs_train = lr_model_top200_xform.predict_proba(X_train)[:,1]

In [638]:
np.isnan(pred_probs_test).sum()

np.int64(0)

In [639]:
np.isnan(pred_probs_train).sum()

np.int64(0)

In [66]:

def calculate_metrics1(y_true, y_pred_label,y_pred_prob):
    accuracy = accuracy_score(y_true, y_pred_label)
    precision = precision_score(y_true, y_pred_label)
    recall = recall_score(y_true, y_pred_label)
    f1 = f1_score(y_true, y_pred_label)
    confusion = confusion_matrix(y_true,y_pred_label)
    roc_score = roc_auc_score(y_true,y_pred_prob)
    
    return {
        'accuracy': accuracy,
        'roc':roc_score,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix':confusion
    }



In [67]:
train_metrics1 = calculate_metrics1(y_train,(pred_probs_train>=0.5).astype(int),pred_probs_train)
test_metrics1 = calculate_metrics1(y_test,(pred_probs_test>=0.5).astype(int),pred_probs_test)

In [68]:
train_metrics_df1 = pd.DataFrame([train_metrics1])
train_metrics_df1['Set'] = "Train"
test_metrics_df1 = pd.DataFrame([test_metrics1])
test_metrics_df1['Set'] = "Test"
model_metrics1 = pd.concat([train_metrics_df1,test_metrics_df1],axis=0)
model_metrics1.reset_index(drop=True,inplace=True)
model_metrics1.head()

,accuracy,roc,precision,recall,f1,confusion_matrix,Set
0,0.704571,0.660936,0.595311,0.132127,0.216256,"[[28534, 1191], [11508, 1752]]",Train
1,0.789396,0.694597,0.533133,0.152850,0.237584,"[[8162, 310], [1962, 354]]",Test


In [69]:
model_name = 'lr_top200_xform'
top_n = 30

In [70]:

def convert_qtr_to_date(qtr):
    y = int(qtr.split("FY")[1])
    q = int(qtr.split("FY")[0][-1])
    quarter_end_dates = {
        "Q1": "30-06-",
        "Q2": "30-09-",
        "Q3": "31-12-",
        "Q4": "31-03-"
    }
    y = y if q==4 else y-1
    fiscal_year = 2000 + int(y)
    end_date = quarter_end_dates.get(f'Q{q}')
    if end_date:
        return end_date + str(fiscal_year)  # Concatenate the end date with the full year
    else:
        raise ValueError("Invalid quarter format")

In [72]:
df_master = pd.read_excel(master_data_path)

In [73]:
df_master['Quarter'] = pd.to_datetime(df_master['Quarter'].apply(convert_qtr_to_date),dayfirst=True)

In [74]:
def label_prediction(row, model_name,top_n):
    in_top_50 = row[f'Top_{top_n}_Pred_Prob_{model_name}']
    target = row['Target']
    
    if in_top_50 and target == 1:
        return 'TP'
    elif in_top_50 and target == 0:
        return 'FP'
    elif not in_top_50 and target == 0:
        return 'TN'
    elif not in_top_50 and target == 1:
        return 'FN'

def flag_top50_nlargest(group, pred_prob_col,top_n):
    # If all pred_probs are zero, return all False
    if group[pred_prob_col].sum() == 0:
        return pd.Series([False] * len(group), index=group.index)
    # Use nlargest to get exactly 50
    top50_idx = group[pred_prob_col].nlargest(top_n).index
    flag = pd.Series(False, index=group.index)
    flag.loc[top50_idx] = True
    return flag

def label_prediction1(row, model_name):
    has_more_prob_than_threshold = row[f'Pred_Prob_Threshold_{model_name}']
    target = row['Target']
    
    if has_more_prob_than_threshold and target == 1:
        return 'TP'
    elif has_more_prob_than_threshold and target == 0:
        return 'FP'
    elif not has_more_prob_than_threshold and target == 0:
        return 'TN'
    elif not has_more_prob_than_threshold and target == 1:
        return 'FN'
    
def add_suffix_to_cols(df, suffix, exclude_cols):
    return df.rename(
        columns={col: f"{col}{suffix}" for col in df.columns if col not in exclude_cols + ['Quarter', 'Set']}
    )


In [75]:

if len(set(['Rank','Returns'])-set(df_train1.columns))!=0:
    print("There are no rank and returns columns, merging now..!")
    df_train1 = df_train1.merge(df_master[['Ticker','Quarter','Returns','Rank']],on=['Ticker','Quarter'],how='left')
    df_test1 = df_test1.merge(df_master[['Ticker','Quarter','Returns','Rank']],on=['Ticker','Quarter'],how='left')

if np.isnan(pred_probs_train).all():
    print("All predictionsa re NaNs")

df_train_pred = df_train1[['Ticker','Quarter','Target','Returns','Rank']]
df_test_pred = df_test1[['Ticker','Quarter','Target','Returns','Rank']]

df_test_pred[f'Pred_Prob_{model_name}'] = pred_probs_test.reshape(-1)
df_train_pred[f'Pred_Prob_{model_name}'] = pred_probs_train.reshape(-1)

#### Top30 as fund - irrespective of the prob values
df_train_pred.loc[:,'Set'] = 'Train'
df_test_pred.loc[:,'Set'] = 'Test'
df_pred = pd.concat([df_train_pred,df_test_pred],axis=0)



df_pred.reset_index(drop=True,inplace=True)
df_pred.sort_values(by=['Quarter',f'Pred_Prob_{model_name}'],ascending=False,inplace=True)
df_pred[f'Top_{top_n}_Pred_Prob_{model_name}'] = df_pred.groupby('Quarter', group_keys=False).apply(flag_top50_nlargest, f'Pred_Prob_{model_name}',top_n)

df_pred[f'Top_50_Pred_Prob_{model_name}'] = df_pred.groupby('Quarter', group_keys=False).apply(flag_top50_nlargest, f'Pred_Prob_{model_name}',50)
# df_pred[(df_pred['Top_{top_n}_Pred_Prob_lstm_top50']==True)&(df_pred['Quarter']=='2005-12-31')]

# test_qtr_preds = df_pred[(df_pred['Quarter']=='2005-12-31')]['Pred_Prob_lstm_top50']
# test_qtr_preds[test_qtr_preds>0.9].shape
df_pred['Pred_Category_Top30'] = df_pred.apply(lambda row: label_prediction(row, model_name=model_name,top_n=30), axis=1)
df_pred['Pred_Category_Top50'] = df_pred.apply(lambda row: label_prediction(row, model_name=model_name,top_n=50), axis=1)
df_pred.reset_index(drop=True,inplace=True)

qtr_summary_df = df_pred.groupby('Quarter').agg(
                    Set = ('Set','first'),
                    Avg_Returns_Predicted_Top30=('Returns',lambda x: x[df_pred.loc[x.index, f'Top_30_Pred_Prob_{model_name}'] == True].mean()),
                    Avg_Returns_Predicted_Top50=('Returns',lambda x: x[df_pred.loc[x.index, f'Top_50_Pred_Prob_{model_name}'] == True].mean()),
                    Count_TP_Top30=('Pred_Category_Top30', lambda x: (x == 'TP').sum()),
                    Count_TN_Top30=('Pred_Category_Top30', lambda x: (x == 'TN').sum()),
                    Count_FP_Top30=('Pred_Category_Top30', lambda x: (x == 'FP').sum()),
                    Count_FN_Top30=('Pred_Category_Top30', lambda x: (x == 'FN').sum()),
                    Count_TP_Top50=('Pred_Category_Top50', lambda x: (x == 'TP').sum()),
                    Count_TN_Top50=('Pred_Category_Top50', lambda x: (x == 'TN').sum()),
                    Count_FP_Top50=('Pred_Category_Top50', lambda x: (x == 'FP').sum()),
                    Count_FN_Top50=('Pred_Category_Top50', lambda x: (x == 'FN').sum())
                    
                    ).reset_index()

# set_summary_df = qtr_summary_df.groupby('Set').agg(
#     Count_TP = ('Count_TP','sum'),
#     Count_TN = ('Count_TN','sum'),
#     Count_FP = ('Count_FP','sum'),
#     Count_FN = ('Count_FN','sum')
# )

df_pred1 = df_pred[['Ticker','Quarter','Target','Returns','Rank',f'Pred_Prob_{model_name}','Set']].copy()
df_pred1[f'Pred_Prob_Threshold_{model_name}'] = df_pred1[f'Pred_Prob_{model_name}'].between(0.5,1)



df_pred1['Pred_Category'] = df_pred1.apply(lambda row: label_prediction1(row, model_name=model_name), axis=1)
quarterly_returns_f2 = df_pred1.groupby('Quarter').agg(
                    Set = ('Set','first'),
                    Avg_Returns_Predicted_Thresh=('Returns', lambda x: x[df_pred1.loc[x.index, f'Pred_Prob_Threshold_{model_name}'] == True].mean()),
                    Count_TP_Thresh=('Pred_Category', lambda x: (x == 'TP').sum()),
                    Count_TN_Thresh=('Pred_Category', lambda x: (x == 'TN').sum()),
                    Count_FP_Thresh=('Pred_Category', lambda x: (x == 'FP').sum()),
                    Count_FN_Thresh=('Pred_Category', lambda x: (x == 'FN').sum())
                    ).reset_index()
# if quarterly_returns_f2['Avg_Returns_Predicted'].isna().any():
#     print("there are quarters where there were no 1s by the model")
# else:
#     print("there are no quarters where there were no 1s by the model")
qtr_summary_df2 = pd.DataFrame(quarterly_returns_f2)


#### Actual and Index returns mapping
##### Actual returns
act_returns = df_pred[df_pred['Target']==1].groupby(by='Quarter')['Returns'].mean()
df_act_returns = pd.DataFrame(act_returns)
df_act_returns.reset_index(inplace=True)
df_act_returns.rename(columns={'Returns':'Avg_Returns_Actual'},inplace=True)
qtr_summary_df = qtr_summary_df.merge(df_act_returns,on='Quarter',how='left')

##### Index returns - Qtrly
df_index = pd.read_excel(qtrly_returns_path,sheet_name='Qtrly_Stock_Returns')
df_index1 = df_index[df_index['Ticker'] == 'Nsemcap'].drop(columns=['Industry']).set_index(keys=['Ticker'])
df_index1 = df_index1.T
df_index1.reset_index(inplace=True)
df_index1.rename(columns={'index':'Quarter','Nsemcap':'Index_Returns'},inplace=True)

df_index1['Quarter'] = df_index1['Quarter'].apply(convert_qtr_to_date)
df_index1['Quarter'] = pd.to_datetime(df_index1['Quarter'],dayfirst=True)

qtr_summary_df = qtr_summary_df.merge(df_index1,how='left',on='Quarter')
qtr_summary_df['delta_returns_act_vs_pred_Top30'] = qtr_summary_df['Avg_Returns_Actual']-qtr_summary_df['Avg_Returns_Predicted_Top30']
qtr_summary_df['delta_returns_pred_vs_index_Top30'] = qtr_summary_df['Avg_Returns_Predicted_Top30']-qtr_summary_df['Index_Returns']
qtr_summary_df['delta_returns_act_vs_pred_Top50'] = qtr_summary_df['Avg_Returns_Actual']-qtr_summary_df['Avg_Returns_Predicted_Top50']
qtr_summary_df['delta_returns_pred_vs_index_Top50'] = qtr_summary_df['Avg_Returns_Predicted_Top50']-qtr_summary_df['Index_Returns']

qtr_summary_df2 = qtr_summary_df2.merge(df_act_returns,on='Quarter',how='left')
qtr_summary_df2 = qtr_summary_df2.merge(df_index1,on='Quarter',how='left')

qtr_summary_df2['delta_returns_act_vs_pred_Thresh'] = qtr_summary_df2['Avg_Returns_Actual']-qtr_summary_df2['Avg_Returns_Predicted_Thresh']
qtr_summary_df2['delta_returns_pred_vs_index_Thresh'] = qtr_summary_df2['Avg_Returns_Predicted_Thresh']-qtr_summary_df2['Index_Returns']

# set_summary_df2 = qtr_summary_df2.groupby('Set').agg(
#     Count_TP = ('Count_TP','sum'),
#     Count_TN = ('Count_TN','sum'),
#     Count_FP = ('Count_FP','sum'),
#     Count_FN = ('Count_FN','sum')
# )

exclude_cols = ['Avg_Returns_Actual', 'Index_Returns']
# suffix_map = {
#     'qtr_summary_df2': '_thresh',
#     'qtr_summary_df': '_top'
# }


# Reset index for merge
qtr_summary_df2_reset = qtr_summary_df2.reset_index(drop=True)
qtr_summary_df_reset = qtr_summary_df.reset_index(drop=True)

# Add suffixes
# qtr_summary_df2_suff = add_suffix_to_cols(qtr_summary_df2_reset, suffix_map['qtr_summary_df2'], exclude_cols)
# qtr_summary_df_suff = add_suffix_to_cols(qtr_summary_df_reset, suffix_map['qtr_summary_df'], exclude_cols)
# ...existing code...

# Merge as before
merged_qtr_summary = pd.merge(
    qtr_summary_df_reset, qtr_summary_df2_reset,
    on=['Quarter', 'Set', 'Avg_Returns_Actual', 'Index_Returns'],
    how='inner',
    suffixes=('', '')
)

# Define the desired column order
ordered_cols = [
    'Quarter','Set',
    'Avg_Returns_Predicted_Top30','Avg_Returns_Predicted_Top50',
    'Avg_Returns_Predicted_Thresh',
    'Avg_Returns_Actual', 'Index_Returns',
    'delta_returns_act_vs_pred_Top30','delta_returns_pred_vs_index_Top30',
    'delta_returns_act_vs_pred_Top50','delta_returns_pred_vs_index_Top50',
    'delta_returns_act_vs_pred_Thresh','delta_returns_pred_vs_index_Thresh',
    'Count_TP_Top30', 'Count_TP_Top50', 'Count_TP_Thresh',
    'Count_TN_Top30', 'Count_TN_Top50', 'Count_TN_Thresh',
    'Count_FP_Top30', 'Count_FP_Top50', 'Count_FP_Thresh',
    'Count_FN_Top30', 'Count_FN_Top50', 'Count_FN_Thresh'
]

# Add any other columns that might exist but are not in the list
ordered_cols += [col for col in merged_qtr_summary.columns if col not in ordered_cols]

# Reorder columns
merged_qtr_summary = merged_qtr_summary[ordered_cols]
# merged_qtr_summary.reset_index(inplace=True)
merged_qtr_summary.head()

There are no rank and returns columns, merging now..!


C:\Users\43737\AppData\Local\Temp\ipykernel_25352\4113010357.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test_pred[f'Pred_Prob_{model_name}'] = pred_probs_test.reshape(-1)
C:\Users\43737\AppData\Local\Temp\ipykernel_25352\4113010357.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train_pred[f'Pred_Prob_{model_name}'] = pred_probs_train.reshape(-1)
C:\Users\43737\AppData\Local\Temp\ipykernel_25352\4113010357.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice 

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Avg_Returns_Predicted_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,...,Count_TP_Thresh,Count_TN_Top30,Count_TN_Top50,Count_TN_Thresh,Count_FP_Top30,Count_FP_Top50,Count_FP_Thresh,Count_FN_Top30,Count_FN_Top50,Count_FN_Thresh
0,2005-03-31,Train,0.202669,0.324026,0.343297,0.462237,-0.019376,0.259568,0.222045,0.138212,...,65,285,281,267,6,10,24,170,154,129
1,2005-06-30,Train,0.304000,0.330705,0.314607,0.495418,0.045476,0.191418,0.258525,0.164713,...,65,283,277,261,9,15,31,172,158,128
2,2005-09-30,Train,0.129293,0.092910,0.077915,0.180268,0.244277,0.050974,-0.114984,0.087358,...,32,278,269,265,11,20,24,177,166,164
3,2005-12-31,Train,0.424620,0.346235,0.445298,0.407782,0.058583,-0.016837,0.366036,0.061547,...,17,286,276,288,4,14,2,169,159,178
4,2006-03-31,Train,-0.001229,-0.057411,-0.067324,0.018142,0.187746,0.019371,-0.188975,0.075552,...,42,292,286,285,3,9,10,168,154,153


In [76]:
merged_qtr_summary['Avg_Returns_Predicted_Thresh'].isna().all()

np.False_

In [650]:
merged_qtr_summary.columns

Index(['Quarter', 'Set', 'Avg_Returns_Predicted_Top30',
       'Avg_Returns_Predicted_Top50', 'Avg_Returns_Predicted_Thresh',
       'Avg_Returns_Actual', 'Index_Returns',
       'delta_returns_act_vs_pred_Top30', 'delta_returns_pred_vs_index_Top30',
       'delta_returns_act_vs_pred_Top50', 'delta_returns_pred_vs_index_Top50',
       'delta_returns_act_vs_pred_Thresh',
       'delta_returns_pred_vs_index_Thresh', 'Count_TP_Top30',
       'Count_TP_Top50', 'Count_TP_Thresh', 'Count_TN_Top30', 'Count_TN_Top50',
       'Count_TN_Thresh', 'Count_FP_Top30', 'Count_FP_Top50',
       'Count_FP_Thresh', 'Count_FN_Top30', 'Count_FN_Top50',
       'Count_FN_Thresh'],
      dtype='object')

In [77]:
df_pred[df_pred['Set']=='Test'][f'Pred_Prob_{model_name}'].unique()

array([0.7347319 , 0.63331719, 0.62042498, ..., 0.01804842, 0.01567157,
       0.01388437])

In [78]:
set_summary_df3 =  merged_qtr_summary.groupby('Set').agg(
    Count_TP_Thresh = ('Count_TP_Thresh','sum'),
    Count_TN_Thresh = ('Count_TN_Thresh','sum'),
    Count_FP_Thresh = ('Count_FP_Thresh','sum'),
    Count_FN_Thresh = ('Count_FN_Thresh','sum'),
    Count_TP_Top30 = ('Count_TP_Top30','sum'),
    Count_TN_Top30 = ('Count_TN_Top30','sum'),
    Count_FP_Top30 = ('Count_FP_Top30','sum'),
    Count_FN_Top30 = ('Count_FN_Top30','sum'),
    Count_TP_Top50 = ('Count_TP_Top50','sum'),
    Count_TN_Top50 = ('Count_TN_Top50','sum'),
    Count_FP_Top50 = ('Count_FP_Top50','sum'),
    Count_FN_Top50 = ('Count_FN_Top50','sum')
)
set_summary_df3

,Count_TP_Thresh,Count_TN_Thresh,Count_FP_Thresh,Count_FN_Thresh,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50
Set,,,,,,,,,,,,
Test,354,8162,310,1962,197,8309,163,2119,324,8196,276,1992
Train,1752,28534,1191,11508,1225,28910,815,12035,1986,28311,1414,11274


In [79]:

def calc_metrics(tp, tn, fp, fn):
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return accuracy, precision, recall, f1


In [80]:
for suffix in ['_Top30','_Top50', '_Thresh']:
    print(f"\nMetrics for {suffix}:")
    for set_name, row in set_summary_df3.iterrows():
        tp = row[f'Count_TP{suffix}']
        tn = row[f'Count_TN{suffix}']
        fp = row[f'Count_FP{suffix}']
        fn = row[f'Count_FN{suffix}']
        accuracy, precision, recall, f1 = calc_metrics(tp, tn, fp, fn)
        print(f"{set_name}: accuracy={accuracy:.4f}, precision={precision:.4f}, recall={recall:.4f}, f1={f1:.4f}")


Metrics for _Top30:
Test: accuracy=0.7885, precision=0.5472, recall=0.0851, f1=0.1472
Train: accuracy=0.7011, precision=0.6005, recall=0.0924, f1=0.1601

Metrics for _Top50:
Test: accuracy=0.7898, precision=0.5400, recall=0.1399, f1=0.2222
Train: accuracy=0.7048, precision=0.5841, recall=0.1498, f1=0.2384

Metrics for _Thresh:
Test: accuracy=0.7894, precision=0.5331, recall=0.1528, f1=0.2376
Train: accuracy=0.7046, precision=0.5953, recall=0.1321, f1=0.2163


In [81]:
if merged_qtr_summary['Avg_Returns_Predicted_Thresh'].mean()==merged_qtr_summary['Avg_Returns_Predicted_Top30'].mean():
    print("Both the stock picking strategies (Top 30 and Pred prob theshold) have same returns")
if merged_qtr_summary['Avg_Returns_Predicted_Thresh'].mean()==merged_qtr_summary['Avg_Returns_Predicted_Top50'].mean():
    print("Both the stock picking strategies (Top 50 and Pred prob theshold) have same returns")
if merged_qtr_summary['Avg_Returns_Predicted_Top30'].mean()==merged_qtr_summary['Avg_Returns_Predicted_Top50'].mean():
    print("Both the stock picking strategies (Top 30 and Top 50) have same returns")
else:
    print("All of the stock picking strategies (Top 30, Top 50 and Pred prob theshold) have different returns")

All of the stock picking strategies (Top 30, Top 50 and Pred prob theshold) have different returns


In [656]:
qtr_summary_df[qtr_summary_df['Quarter']=='2021-09-30']

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,delta_returns_pred_vs_index_Top50
66,2021-09-30,Train,0.408642,0.400875,6,769,24,23,9,752,41,20,1.519144,0.126554,1.110502,0.282088,1.118268,0.274322


In [657]:
qtr_summary_df2[qtr_summary_df2['Quarter']=='2021-09-30']

,Quarter,Set,Avg_Returns_Predicted_Thresh,Count_TP_Thresh,Count_TN_Thresh,Count_FP_Thresh,Count_FN_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Thresh,delta_returns_pred_vs_index_Thresh
66,2021-09-30,Train,-0.340932,0,792,1,29,1.519144,0.126554,1.860076,-0.467486


In [658]:
df_pred.columns

Index(['Ticker', 'Quarter', 'Target', 'Returns', 'Rank',
       'Pred_Prob_lr_top200_xform', 'Set', 'Top_30_Pred_Prob_lr_top200_xform',
       'Top_50_Pred_Prob_lr_top200_xform', 'Pred_Category_Top30',
       'Pred_Category_Top50'],
      dtype='object')

In [659]:
df_pred1.columns

Index(['Ticker', 'Quarter', 'Target', 'Returns', 'Rank',
       'Pred_Prob_lr_top200_xform', 'Set',
       'Pred_Prob_Threshold_lr_top200_xform', 'Pred_Category'],
      dtype='object')

In [660]:
merged_qtr_summary[merged_qtr_summary['Quarter']=='2021-09-30']

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Avg_Returns_Predicted_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,...,Count_TP_Thresh,Count_TN_Top30,Count_TN_Top50,Count_TN_Thresh,Count_FP_Top30,Count_FP_Top50,Count_FP_Thresh,Count_FN_Top30,Count_FN_Top50,Count_FN_Thresh
66,2021-09-30,Train,0.408642,0.400875,-0.340932,1.519144,0.126554,1.110502,0.282088,1.118268,...,0,769,752,792,24,41,1,23,20,29


In [82]:
merged_qtr_summary[merged_qtr_summary['Avg_Returns_Predicted_Top30'].isna()].shape

(2, 25)

In [83]:
df_pred[df_pred['Set']=='Test'][f'Pred_Prob_{model_name}'].unique()

array([0.7347319 , 0.63331719, 0.62042498, ..., 0.01804842, 0.01567157,
       0.01388437])

In [84]:
df_pred[df_pred['Set']=='Test'][f'Pred_Prob_{model_name}'].describe()

count    10788.000000
mean         0.289112
std          0.118714
min          0.000281
25%          0.204663
50%          0.268304
75%          0.353909
max          0.917692
Name: Pred_Prob_lr_top200_xform, dtype: float64

In [85]:
merged_qtr_summary['Quarter'].is_monotonic_increasing

True

In [665]:
set_summary_df3

,Count_TP_Thresh,Count_TN_Thresh,Count_FP_Thresh,Count_FN_Thresh,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50
Set,,,,,,,,,,,,
Test,0,10438,4,346,42,10124,318,304,62,9904,538,284
Train,0,41050,6,1929,255,39271,1785,1674,373,38029,3027,1556


In [86]:
merged_qtr_summary['Avg_Returns_Predicted_Top30'] = merged_qtr_summary['Avg_Returns_Predicted_Top30'].ffill()
merged_qtr_summary['Avg_Returns_Predicted_Top50'] = merged_qtr_summary['Avg_Returns_Predicted_Top50'].ffill()
merged_qtr_summary['Avg_Returns_Predicted_Thresh'] = merged_qtr_summary['Avg_Returns_Predicted_Thresh'].ffill()

In [87]:
merged_qtr_summary['Avg_Returns_Predicted_Top30'].fillna(0,inplace=True)
merged_qtr_summary['Avg_Returns_Predicted_Top50'].fillna(0,inplace=True)
merged_qtr_summary['Avg_Returns_Predicted_Thresh'].fillna(0,inplace=True)

C:\Users\43737\AppData\Local\Temp\ipykernel_25352\4263771019.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_qtr_summary['Avg_Returns_Predicted_Top30'].fillna(0,inplace=True)
C:\Users\43737\AppData\Local\Temp\ipykernel_25352\4263771019.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always beh

In [88]:
merged_qtr_summary.head()

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Avg_Returns_Predicted_Thresh,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,...,Count_TP_Thresh,Count_TN_Top30,Count_TN_Top50,Count_TN_Thresh,Count_FP_Top30,Count_FP_Top50,Count_FP_Thresh,Count_FN_Top30,Count_FN_Top50,Count_FN_Thresh
0,2005-03-31,Train,0.202669,0.324026,0.343297,0.462237,-0.019376,0.259568,0.222045,0.138212,...,65,285,281,267,6,10,24,170,154,129
1,2005-06-30,Train,0.304000,0.330705,0.314607,0.495418,0.045476,0.191418,0.258525,0.164713,...,65,283,277,261,9,15,31,172,158,128
2,2005-09-30,Train,0.129293,0.092910,0.077915,0.180268,0.244277,0.050974,-0.114984,0.087358,...,32,278,269,265,11,20,24,177,166,164
3,2005-12-31,Train,0.424620,0.346235,0.445298,0.407782,0.058583,-0.016837,0.366036,0.061547,...,17,286,276,288,4,14,2,169,159,178
4,2006-03-31,Train,-0.001229,-0.057411,-0.067324,0.018142,0.187746,0.019371,-0.188975,0.075552,...,42,292,286,285,3,9,10,168,154,153


In [669]:
df_pred.head()

,Ticker,Quarter,Target,Returns,Rank,Pred_Prob_lr_top200_xform,Set,Top_30_Pred_Prob_lr_top200_xform,Top_50_Pred_Prob_lr_top200_xform,Pred_Category_Top30,Pred_Category_Top50
0,uel,2024-12-31,0,NaN,NaN,0.727036,Test,True,True,FP,FP
1,ut,2024-12-31,0,-0.358306,745.0,0.133512,Test,True,True,FP,FP
2,rtpow,2024-12-31,0,-0.283309,635.0,0.133457,Test,True,True,FP,FP
3,sdb,2024-12-31,0,0.122062,36.0,0.119482,Test,True,True,FP,FP
4,dcxindia,2024-12-31,0,-0.341313,730.0,0.115898,Test,True,True,FP,FP


In [89]:
qtr_summary_df.head()

,Quarter,Set,Avg_Returns_Predicted_Top30,Avg_Returns_Predicted_Top50,Count_TP_Top30,Count_TN_Top30,Count_FP_Top30,Count_FN_Top30,Count_TP_Top50,Count_TN_Top50,Count_FP_Top50,Count_FN_Top50,Avg_Returns_Actual,Index_Returns,delta_returns_act_vs_pred_Top30,delta_returns_pred_vs_index_Top30,delta_returns_act_vs_pred_Top50,delta_returns_pred_vs_index_Top50
0,2005-03-31,Train,0.202669,0.324026,24,285,6,170,40,281,10,154,0.462237,-0.019376,0.259568,0.222045,0.138212,0.343401
1,2005-06-30,Train,0.304000,0.330705,21,283,9,172,35,277,15,158,0.495418,0.045476,0.191418,0.258525,0.164713,0.285229
2,2005-09-30,Train,0.129293,0.092910,19,278,11,177,30,269,20,166,0.180268,0.244277,0.050974,-0.114984,0.087358,-0.151367
3,2005-12-31,Train,0.424620,0.346235,26,286,4,169,36,276,14,159,0.407782,0.058583,-0.016837,0.366036,0.061547,0.287652
4,2006-03-31,Train,-0.001229,-0.057411,27,292,3,168,41,286,9,154,0.018142,0.187746,0.019371,-0.188975,0.075552,-0.245156


In [90]:
qtr_summary_df['Set'].value_counts()

Set
Train    68
Test     12
Name: count, dtype: int64

In [92]:
with pd.ExcelWriter(r"D:\Python\Final Quant Fund\tf_219_env\Quant Fund\Fund1\results\lag\top200\3yrs_as_test_data_ts\Pred_Vs_Act_LR_xform_time_split_25_07_2025.xlsx", ) as writer:
    merged_qtr_summary.to_excel(writer, sheet_name='Qtr_Summary', index=False)
    qtr_summary_df.to_excel(writer, sheet_name='Qtr_Summary_Top', index=False)
    qtr_summary_df2.to_excel(writer, sheet_name='Qtr_Summary_Thresh', index=False)
    df_pred.to_excel(writer, sheet_name='Pred_Vs_Act_Top', index=False)
    df_pred1.to_excel(writer, sheet_name='Pred_Vs_Act_Thresh', index=False)
    set_summary_df3.to_excel(writer, sheet_name='Set_Summary')